# eye_cnn.ipynb — CNN 눈 뜸/감음 판정 모델

> 서버가 앱에서 받은 '졸음 의심' 사진을 한 번 더 확인할 때 쓰는 눈 감음 판정 CNN. 학습 · 추가 학습(미세조정) · 시험 기록

## 한눈에 보기

**목적:** 폰(ML Kit)이 '눈 감김'으로 본 사진을 서버가 다시 확인 → 눈 사진만 잘라 감음 확률 계산 (EAR 과 함께 씀)

**쓴 데이터**

| 데이터 | 내용 | 쓴 곳 |
|---|---|---|
| MRL Eye Dataset (Kaggle) | 외국인 37명 · 적외선 · 눈만 잘린 흑백 84,898장 | 1 · 2번: 처음 학습 |
| AI-Hub 173 「졸음운전 예방을 위한 운전자 상태 정보 영상」 | 한국인 실제 운전자 (버스 · 승용 · 택시 · 트럭) · 적외선 | 3번: 추가 학습(미세조정) · 시험 |
| 팀원 사진 · 영상 프레임 | 시연 폰 컬러 사진 428장 | 4번: 서버 값 확인 (KPI ①, 학습에 안 씀) |
| AI-Hub 548 「안구 움직임 영상 데이터」 | 한국인 웹캠 **컬러(RGB)** 얼굴 사진 · 뜸 / 반쯤 / 감음 라벨 | 6번: 컬러 사진 시험 · 7번: 추가 학습 (반쯤 감음은 뺌) |

**결과 요약 (감음 판정 균형 정확도 = 감은 눈 맞힌 비율과 뜬 눈 맞힌 비율의 평균)**

| 모델 | MRL test | AI-Hub 173 (한국인 운전자) | 팀 사진 (KPI ①) | AI-Hub 548 컬러 (Validation 151명) |
|---|---|---|---|---|
| MRL만 (`eye_cnn.pt`) | 95.9% | 82.5% (뜬 눈 4개 중 1개를 감음으로 오판) | 완전히 감은 눈 100% | — |
| 추가 학습(미세조정) (`eye_cnn_aihub.pt`, 10/28 채택 → 적외선 카메라용으로 보관) | 96.2% | **93.8%** (트럭 96.5%) | 완전히 감은 눈 100% · 진짜 졸음 취소 0건 | 83.1% (뜬 눈 오판 0.1% · 감은 눈 66.3%) |
| **548 컬러 추가 학습 (`eye_cnn_548.pt`, 11/03 채택)** | 80.8% | 83.0% (적외선을 잊음) | 완전히 감은 눈 100% · 뜬 눈 오판 0 · 반쯤 감음 78/109 | **99.4%** (뜬 눈 오판 0.8% · 감은 눈 99.7%) |

- **10/28 채택:** 미세조정 모델 + 감음 기준 0.4, 서버는 CNN · EAR 둘 다 '확실히 뜸'일 때만 경고 취소
- **11/03 교체:** 앱은 폰 컬러 카메라만 씀 → 컬러로 추가 학습한 `eye_cnn_548.pt` (기준 0.4 그대로). 카메라에 맞는 모델을 고름 — 적외선 카메라면 `eye_cnn_aihub.pt` (7-6)
- **남은 약점 (발표 한계):** 선글라스 — 눈이 안 보여도 판정함 (판별기 없음, 7-6). 548 모델은 적외선 사진에서 나빠짐

**구간 목차**

| 구간 | 내용 | 실행 |
|---|---|---|
| 1 | MRL 데이터 확인 · 사람 단위 분할 | Colab |
| 2 | MRL 학습 → `eye_cnn.pt` | Colab (T4 GPU) |
| 3 | AI-Hub 173 추가 학습(미세조정) → `eye_cnn_aihub.pt` · Grad-CAM (3-8) | 받기 · 자르기는 PC, 학습은 Colab |
| 4 | 팀 사진 CNN 확률 = 서버 값 확인 | Colab 또는 PC |
| 5 | W&B 기록 (프로젝트 `eye-cnn`) | Colab |
| 6 | AI-Hub 548 컬러 사진 시험 | PC (`aihub548.py`) |
| 7 | AI-Hub 548 컬러 추가 학습(미세조정) → `eye_cnn_548.pt` | 자르기 · Validation 시험은 PC, 학습은 Colab |
| 8 | 상황별 증강 (색온도 · 밤 · 역광 · 그림자 · 안경 반사 · 흔들림 · 저해상도 · JPEG) + 모델 3개 앙상블 | 컬러 자르기 · 시험은 PC, 학습은 Colab |
| 9 | 의심 확인 (라벨 섞기 · Grad-CAM) · 좌우 반전 평균 · 적외선 섞기 · 틀린 사진 모아 보기 | Colab (8번 다음), PC 시험 |
| 10 | PC 시험 모아 보기 — 기준값 곡선 · 보고 형식 · 조건별 · CEW 원인 (중간 점검 피드백) | PC 결과 CSV (Colab · PC 어디서나) |
| 11 | (보류 — 12번 실험 뒤) 정규화 비교 (적외선 / 컬러 · 외국인 / 한국인) + 사람 단위 k-fold → CEW 는 그 뒤에 | Colab (GPU) — 1-1 ~ 2-4 · 3-1 · 3-2 · 7-1 다음 |
| 12 | 세로 누르기 (vscale) 설정 5개 × MRL → 173 → 548 다시 학습 → 548 학습용 val 로 고르기 · 151명 한 번 (팀장 규칙 11/04) | Colab (GPU) + PC (aihub548.py eval --set trainval · pick · final) |

**규칙**
- 전처리는 서버와 같은 파일 `sleep_detector_api/services/eye_preprocess.py` (`crop_eye` · `prep_eye`) 그대로 씀
- 🔴 `kaggle.json` · 데이터 사진 · 모델(`.pt`)은 구글 드라이브에만 (저장소 · 단톡 X). AI-Hub 는 재배포 금지
- 🔴 GitHub 에 올리기 전: Colab → 수정 → **모든 출력 지우기** (사진이 출력에 남지 않게)

## 1. MRL 데이터 확인 (학습 전)

확인 결과는 1-13 표에 정리함

| # | 확인할 것 | 셀 |
|---|---|---|
| 1 | 장수 84,898 · 이름 규칙 · 중복 | 1-3 |
| 2 | 폴더 라벨(awake/sleepy) = 파일 이름 눈 상태(1/0) — **전부** | 1-4 |
| 3 | Kaggle 기존 train/val/test 에 같은 사람이 섞였는지 — **전부** | 1-5 |
| 4 | 피험자별 장수 · 한 클래스만 있는 사람 · 안경 · 센서 | 1-6 |
| 5 | 이미지 크기 · 흑백 여부 · 가로세로 비 | 1-7 |
| 6 | 눈으로 보기 — 뜸/감음 · 센서 · 안경 · 반사 · 가장 작은/큰 사진 | 1-8 |
| 7 | **눈이 얼마나 꽉 차게 잘렸나** → 서버에서 자를 여유(margin) 정하기 | 1-9 |
| 8 | 64×64 흑백으로 줄였을 때 밝기 평균 · 표준편차 → 정규화 값 후보 | 1-10 |
| 9 | 사람 단위 분할 + 감음/뜸 비율 균형 | 1-11 |

- 결과 그림 · 분할표는 `OUT_DIR`(드라이브)에 저장됨

### 1-1. 드라이브 연결 · Kaggle 키 · 한글 글꼴

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
KAGGLE_JSON = '/content/drive/MyDrive/sleep_detector/kaggle.json'   # ← 본인 kaggle.json 위치
OUT_DIR = '/content/drive/MyDrive/sleep_detector/mrl_check'    # ← 결과 그림 · 분할표 저장 폴더
os.makedirs(OUT_DIR, exist_ok=True)

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy(KAGGLE_JSON, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('키 준비 완료 (내용은 출력하지 않음)')

# 그림 한글 글꼴 (Colab 기본 글꼴에는 한글이 없음 → 나눔고딕 설치)
import subprocess, matplotlib as mpl, matplotlib.font_manager as fm
NANUM = '/usr/share/fonts/truetype/nanum'
if not fm.findSystemFonts([NANUM]):
    subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
for f in fm.findSystemFonts([NANUM]):
    fm.fontManager.addfont(f)
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False   # 한글 글꼴에서 '-' 가 깨지지 않게

### 1-2. 내려받기 (Colab 로컬 디스크, 약 330MB — 드라이브에 풀지 않음)

In [ ]:
%%bash
if [ ! -d /content/mrl ]; then
  pip -q install kaggle
  kaggle datasets download -d akashshingha850/mrl-eye-dataset -p /content/mrl --unzip -q
fi
echo "폴더:"; find /content/mrl -maxdepth 3 -type d | head -20
echo "PNG 수:"; find /content/mrl -name '*.png' | wc -l

### 1-3. 파일 이름 읽기 — 장수 · 이름 규칙 · 중복

In [ ]:
import glob, re, collections
from pathlib import Path
import pandas as pd

# s0001_00001_0_0_0_0_0_01 = 피험자_번호_성별_안경_눈상태_반사_조명_센서
NAME = re.compile(r'^(s\d{4})_(\d+)_([01])_([01])_([01])_([012])_([01])_(\d{2})$')
paths = sorted(glob.glob('/content/mrl/**/*.png', recursive=True))
rows, bad = [], []
for p in paths:
    stem = os.path.basename(p)[:-4]
    m = NAME.match(stem)
    if not m:
        bad.append(p)
        continue
    parts = set(Path(p).parts)
    rows.append(dict(path=p, name=stem, subject=m[1], img_no=int(m[2]), gender=int(m[3]), glasses=int(m[4]),
                     eye=int(m[5]), reflection=int(m[6]), light=int(m[7]), sensor=m[8],
                     kaggle_split=next((s for s in ('train', 'val', 'test') if s in parts), '?'),
                     folder=next((s for s in ('awake', 'sleepy') if s in parts), '?')))
df = pd.DataFrame(rows)

print('PNG 전체      :', len(paths))
print('이름 규칙 통과 :', len(df), '/ 불통과:', len(bad), bad[:5])
print('84,898 장인가  :', len(df) == 84898)
print('같은 이름 중복 :', int(df['name'].duplicated().sum()))
print('피험자 수      :', df['subject'].nunique())

### 1-4. 폴더 라벨 = 파일 이름 눈 상태? (전부)

- 기대값: `sleepy` = 0(감음), `awake` = 1(뜸), 불일치 0장

In [ ]:
df['folder_eye'] = df['folder'].map({'sleepy': 0, 'awake': 1})
mismatch = df[df['folder_eye'] != df['eye']]
print('폴더 라벨 ≠ 파일 이름 눈 상태:', len(mismatch), '장')
display(pd.crosstab(df['folder'], df['eye'], margins=True))
display(mismatch.head(10)[['name', 'kaggle_split', 'folder', 'eye']])

### 1-5. Kaggle 기존 분할에 같은 사람이 섞였나? (전부)

- 섞였으면 기존 train/val/test 는 안 쓰고 1-11 의 사람 단위 분할 사용

In [ ]:
subj_split = df.groupby('subject')['kaggle_split'].agg(lambda s: '+'.join(sorted(set(s))))
print(subj_split.value_counts().to_string())
print('\n여러 split 에 걸친 피험자:', int((subj_split.str.contains('\\+')).sum()), '/', len(subj_split))
display(pd.crosstab(df['subject'], df['kaggle_split']).head(10))

### 1-6. 피험자별 장수 · 클래스 · 안경 · 센서

In [ ]:
per = (df.pivot_table(index='subject', columns='eye', values='name', aggfunc='count', fill_value=0)
         .rename(columns={0: 'closed', 1: 'open'}))
per['total'] = per['closed'] + per['open']
per['closed_ratio'] = (per['closed'] / per['total']).round(2)
per['glasses_ratio'] = df.groupby('subject')['glasses'].mean().round(2)
per['gender'] = df.groupby('subject')['gender'].agg(lambda s: s.mode()[0])
per['sensors'] = df.groupby('subject')['sensor'].agg(lambda s: ','.join(sorted(set(s))))
display(per.sort_values('total'))

print('한 클래스만 있는 피험자:', list(per[(per['closed'] == 0) | (per['open'] == 0)].index))
print('장수 최소 / 최대:', per['total'].idxmin(), per['total'].min(), '/', per['total'].idxmax(), per['total'].max())
for col in ['gender', 'glasses', 'eye', 'reflection', 'light', 'sensor']:
    print(f'{col:10s}', df[col].value_counts().sort_index().to_dict())

### 1-7. 이미지 크기 · 흑백 여부 (전부, 파일 머리 부분만 읽어 빠름)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sizes, modes = [], collections.Counter()
for p in df['path']:
    with Image.open(p) as im:
        sizes.append(im.size)
        modes[im.mode] += 1
df['w'] = [s[0] for s in sizes]
df['h'] = [s[1] for s in sizes]
df['aspect'] = (df['w'] / df['h']).round(2)

print('이미지 모드 (L = 흑백):', dict(modes))
display(df[['w', 'h', 'aspect']].describe().round(1))
print('센서별 중간값'); display(df.groupby('sensor')[['w', 'h', 'aspect']].median())
print('뜸/감음별 중간값'); display(df.groupby('eye')[['w', 'h', 'aspect']].median())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(df['w'], bins=60); ax[0].set_title('가로 (px)')
ax[1].hist(df['aspect'], bins=60); ax[1].set_title('가로 / 세로 비')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/size_hist.png', dpi=100); plt.show()

### 1-8. 눈으로 보기

- 그림 제목: `피험자 뜸/감음 (안경)` / `가로x세로 센서번호`
- 라벨이 틀려 보이는 사진 · 반쯤 감은 사진이 있는지 확인

In [ ]:
def grid(sub, title, fname, n=24, cols=8, seed=0):
    sub = sub.sample(min(n, len(sub)), random_state=seed)
    nrows = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(nrows, cols, figsize=(cols * 1.6, nrows * 1.9))
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        ax.imshow(Image.open(r.path), cmap='gray', vmin=0, vmax=255)
        ax.set_title(f"{r.subject} {'뜸' if r.eye else '감음'}{' 안경' if r.glasses else ''}\n{r.w}x{r.h} 센서{r.sensor}", fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=100); plt.show()

grid(df[df.eye == 1], '뜸 (eye=1)', 'grid_open.png')
grid(df[df.eye == 0], '감음 (eye=0)', 'grid_closed.png')
for s in sorted(df['sensor'].unique()):
    grid(df[df.sensor == s], f'센서 {s}', f'grid_sensor{s}.png', n=16)
grid(df[df.glasses == 1], '안경', 'grid_glasses.png', n=16)
grid(df[df.reflection == 2], '반사 큼', 'grid_reflection.png', n=16)
grid(df[df.light == 0], '조명 나쁨', 'grid_badlight.png', n=16)
grid(df.nsmallest(16, 'w'), '가장 작은 사진', 'grid_smallest.png', n=16)
grid(df.nlargest(16, 'w'), '가장 큰 사진', 'grid_largest.png', n=16)

### 1-9. 눈이 얼마나 꽉 차게 잘렸나 (서버 자르기 여유 정하기)

- 빨간 선 = 사진 가로 · 세로의 10% 간격, 뜬 눈 16장
- 이 값으로 서버의 **MediaPipe 눈 점 → 자르기 범위** 정함 (학습과 서버가 같은 모양이어야 함)

In [ ]:
sub = df[df.eye == 1].sample(16, random_state=1)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, (_, r) in zip(axes.flat, sub.iterrows()):
    ax.imshow(Image.open(r.path).convert('L').resize((130, 130)), cmap='gray', vmin=0, vmax=255)
    for t in range(0, 131, 13):
        ax.axvline(t, color='r', lw=0.8, alpha=0.6); ax.axhline(t, color='r', lw=0.8, alpha=0.6)
    ax.set_xlim(0, 129); ax.set_ylim(129, 0)  # 선 때문에 그림 밖에 흰 여백이 생기지 않게
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{r.subject} {r.w}x{r.h}', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/crop_tightness.png', dpi=100); plt.show()

**1-9 결과 (1-8 · 1-9 그림을 눈으로 본 값 — 1-10b 평균 사진으로 다시 확인)**

- 사진은 **전부 정사각형** (가로세로 비 1.0, 84,898장 모두)
- 눈 가로 폭(눈꼬리~눈꼬리): 사진 가로의 약 **55~70%** (센서 02 의 s0031 은 약 35~40% 로 눈이 작게 찍힘)
- 눈 위치: 가로는 가운데, 세로는 가운데보다 **약간 아래** (눈 중심이 위에서 약 50~55%)
- 눈썹: **들어 있는 사진 많음** (위쪽 20~30% 안), 사진마다 다름 → 자른 범위가 사진마다 조금씩 다름
- 라벨이 틀려 보이는 사진: 이번 표본에서는 못 찾음. 반쯤 감은 눈은 '뜸'(e1) 쪽에 일부 있음 (예: s0014 · s0019 의 가늘게 뜬 눈)

### 1-10. 64×64 흑백으로 줄였을 때 밝기 (정규화 값 후보)

- 피험자 · 눈 상태마다 최대 100장씩 표본
- 여기서는 측정용으로 PIL BILINEAR 로 줄임. 학습(2번) · 서버는 `prep_eye` (cv2 `INTER_AREA`) 사용

In [ ]:
import numpy as np
SIZE = 64  # 서버 config.CNN_INPUT_SIZE

samp = df.sample(frac=1, random_state=0).groupby(['subject', 'eye']).head(100)  # 피험자 · 눈 상태마다 최대 100장
arr = np.stack([np.asarray(Image.open(p).convert('L').resize((SIZE, SIZE), Image.BILINEAR), dtype=np.float32) / 255
                for p in samp['path']])
print(f'표본 {len(arr)}장 / 평균 {arr.mean():.4f} / 표준편차 {arr.std():.4f}  ← 정규화 값 후보')
samp = samp.assign(mean=arr.mean(axis=(1, 2)))
print('센서별 밝기'); display(samp.groupby('sensor')['mean'].describe().round(3))
print('뜸/감음별 밝기'); display(samp.groupby('eye')['mean'].describe().round(3))

### 1-10b. 평균 눈 사진 — 눈이 사진의 어디에, 얼마나 크게 있나 (1-9 를 숫자로 확인)

- 1-10 의 64×64 표본을 뜸 / 감음으로 나눠 평균. 어두운 띠 = 눈 자리
- 자르기 범위는 **뜬 눈 평균**으로 잼. 사진 가장자리는 원래 어두워서(비네팅) 가장자리 15%는 빼고 찾음
- 눈 폭 = 눈 높이의 가로 띠에서 '그 열 평균보다 어두운 정도'가 가장 어두운 곳의 절반 이상인 구간

In [ ]:
def inner_argmin(p, lo=0.15, hi=0.85):
    a, b = int(len(p) * lo), int(len(p) * hi)
    return a + int(np.argmin(p[a:b]))

m_open = arr[(samp['eye'] == 1).to_numpy()].mean(axis=0)
m_closed = arr[(samp['eye'] == 0).to_numpy()].mean(axis=0)

# 눈 중심 = 가장자리 15% 를 뺀 안쪽에서 가장 어두운 점 (5×5 평균으로 부드럽게) — 가로줄 평균은 눈썹 띠에 끌려감
pad = np.pad(m_open, 2, mode='edge')
smooth = sum(pad[dy:dy + SIZE, dx:dx + SIZE] for dy in range(5) for dx in range(5)) / 25
a, b = int(SIZE * 0.15), int(SIZE * 0.85)
cy, cx = np.unravel_index(np.argmin(smooth[a:b, a:b]), (b - a, b - a))
cy, cx = int(cy) + a, int(cx) + a
dark = m_open[max(cy - 3, 0):cy + 4].mean(axis=0) - m_open.mean(axis=0)  # 눈 띠가 그 열 평균보다 얼마나 어두운가
def span(frac):
    level = dark[cx] * frac  # 가장 어두운 깊이의 frac 만큼 어두운 곳까지
    left = right = cx
    while left > 0 and dark[left - 1] < level:
        left -= 1
    while right < SIZE - 1 and dark[right + 1] < level:
        right += 1
    return left, right

left, right = span(0.5)
l20, r20 = span(0.2)
eye_w50, eye_w20 = (right - left + 1) / SIZE, (r20 - l20 + 1) / SIZE

print(f'뜬 눈 중심: 위에서 {cy / SIZE:.0%}, 왼쪽에서 {cx / SIZE:.0%}')
print(f'눈 폭 — 절반 깊이: 사진 가로의 {eye_w50:.0%} / 20% 깊이: {eye_w20:.0%}')
print('  (평균 사진은 흐려서 눈꼬리가 옅다 → MediaPipe 의 눈꼬리~눈꼬리 폭에는 20% 깊이 값이 더 가깝다. 1-9 눈 확인값 55~70% 와 비교)')
print(f'  → 자르기 한 변 S ≈ 눈 폭 W ÷ {eye_w20:.2f} = {1 / eye_w20:.2f}W  (20% 깊이 기준)')
print(f'자르기 중심 = 눈 중심보다 {cy / SIZE - 0.5:+.2f}S 아래에 눈이 오도록 (음수면 위)')
brow = inner_argmin(m_closed.mean(axis=1), 0.10, 0.45)
print(f'감은 눈 평균의 눈썹 띠: 위에서 {brow / SIZE:.0%}')

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, m, name in ((ax[0], m_open, '뜬 눈 평균'), (ax[1], m_closed, '감은 눈 평균')):
    a.imshow(m, cmap='gray')
    for t in np.linspace(0, SIZE - 1, 11):
        a.axvline(t, color='r', lw=0.8, alpha=0.6); a.axhline(t, color='r', lw=0.8, alpha=0.6)
    a.set_xlim(0, SIZE - 1); a.set_ylim(SIZE - 1, 0); a.set_title(name)
ax[0].axvspan(left, right, ymin=0.45, ymax=0.55, color='y', alpha=0.4)
ax[2].plot(np.arange(SIZE) / SIZE, m_open.mean(axis=1), label='뜬 눈 가로줄 밝기 (위→아래)')
ax[2].plot(np.arange(SIZE) / SIZE, dark, '--', label='뜬 눈 띠 − 열 평균 (왼쪽→오른쪽)')
ax[2].plot(np.arange(SIZE) / SIZE, m_closed.mean(axis=1), label='감은 눈 가로줄 밝기 (위→아래)')
ax[2].axhline(dark[cx] * 0.5, color='gray', lw=0.6); ax[2].axhline(dark[cx] * 0.2, color='gray', lw=0.6, ls=':'); ax[2].set_xlabel('위치 (0 = 위 · 왼쪽, 1 = 아래 · 오른쪽)'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/mean_eye.png', dpi=100); plt.show()

### 1-11. 사람 단위 분할 (70 / 15 / 15) + 감음/뜸 비율 균형

1. 장수가 많은 사람부터 목표 비율에 가장 모자란 쪽에 배정
2. 피험자 두 명을 맞바꿔 **장수 비율과 감음 비율이 목표에 더 가까워지면** 바꿈 — 더 나아지지 않을 때까지 (같은 데이터면 항상 같은 결과)

In [ ]:
import itertools, json, datetime

TARGET = {'train': 0.70, 'val': 0.15, 'test': 0.15}
total = int(per['total'].sum())
overall_closed = per['closed'].sum() / total

split, cnt = {}, collections.Counter()
for s in per.sort_values('total', ascending=False).index:
    k = min(TARGET, key=lambda k: cnt[k] / total - TARGET[k])
    split[s] = k
    cnt[k] += per.loc[s, 'total']

def stats(split):
    out = {}
    for k in TARGET:
        ss = sorted(s for s in split if split[s] == k)
        n, c = int(per.loc[ss, 'total'].sum()), int(per.loc[ss, 'closed'].sum())
        out[k] = dict(subjects=ss, n_subjects=len(ss), images=n, ratio=round(n / total, 3),
                      closed=c, open=n - c, closed_ratio=round(c / n, 3))
    return out

def cost(split):
    st = stats(split)
    return sum(2 * abs(st[k]['ratio'] - TARGET[k]) + abs(st[k]['closed_ratio'] - overall_closed) for k in TARGET)

def show(title, st):
    print(title)
    for k, v in st.items():
        print(f"  {k:5s} {v['n_subjects']:2d}명 {v['images']:6d}장 ({v['ratio']:.1%})  감음 {v['closed_ratio']:.1%}")

show('처음 배정', stats(split))
while True:
    best_c, best_pair = cost(split), None
    for a, b in itertools.combinations(sorted(split), 2):
        if split[a] == split[b]:
            continue
        split[a], split[b] = split[b], split[a]
        c = cost(split)
        split[a], split[b] = split[b], split[a]
        if c < best_c - 1e-9:
            best_c, best_pair = c, (a, b)
    if best_pair is None:
        break
    a, b = best_pair
    split[a], split[b] = split[b], split[a]

final = stats(split)
show(f'\n균형 조정 후 (전체 감음 비율 {overall_closed:.1%})', final)
for k, v in final.items():
    print(f"{k}: {', '.join(v['subjects'])}")
one_class = list(per[(per['closed'] == 0) | (per['open'] == 0)].index)
print('\n한 클래스만 있는 피험자의 배정:', {s: split[s] for s in one_class})

json.dump({'made': str(datetime.date.today()), 'method': 'greedy + swap balance', 'split': split, 'summary': final},
          open(f'{OUT_DIR}/mrl_person_split.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
df['split'] = df['subject'].map(split)
print('\n저장:', f'{OUT_DIR}/mrl_person_split.json')

### 1-12. 결과 요약

In [ ]:
summary = {
    '장수': len(df), '84898과 같음': len(df) == 84898, '이름 불통과': len(bad), '이름 중복': int(df['name'].duplicated().sum()),
    '피험자': df['subject'].nunique(),
    '폴더 라벨 불일치': len(mismatch),
    'Kaggle 분할에서 여러 split 에 걸친 피험자': int(subj_split.str.contains('\\+').sum()),
    '한 클래스만 있는 피험자': one_class,
    '이미지 모드': dict(modes),
    '크기 중간값 (w, h)': (int(df['w'].median()), int(df['h'].median())),
    '크기 범위 w': (int(df['w'].min()), int(df['w'].max())),
    '64x64 밝기 평균 / 표준편차': (round(float(arr.mean()), 4), round(float(arr.std()), 4)),
    '분할': {k: f"{v['n_subjects']}명 {v['images']}장 감음 {v['closed_ratio']:.1%}" for k, v in final.items()},
}
for k, v in summary.items():
    print(f'- {k}: {v}')

### 1-13. 확인 결과 정리

| 확인 | 결과 | 판정 |
|---|---|---|
| 장수 · 이름 규칙 · 중복 | 84,898장, 이름 규칙 불통과 0, 중복 0, 피험자 37명 | ✅ |
| 폴더 라벨 = 파일 이름 눈 상태 | 불일치 **0장** (sleepy = 0 감음 41,946 / awake = 1 뜸 42,952) | ✅ |
| Kaggle 기존 분할 | **37명 전원**이 train · val · test 에 다 들어 있음 → 이미지 단위 무작위 분할임 | ❌ 안 씀 → 1-11 분할 사용 |
| 이미지 | 전부 흑백(L), 전부 정사각형, 52~299px (중간값 87) | ✅ |
| 센서 | 01 이 83% (70,142장). 센서마다 크기(01: 84 · 02: 126 · 03: 73px) · 밝기 다름 | ⚠️ |
| **피험자별 감음 비율** | 0.02 ~ 1.00 으로 **극단적** (s0007 98% 감음, s0035 3% 감음) | ⚠️ 사람 단위 분할 + 비율 균형 꼭 필요 |
| 한 클래스만 있는 피험자 | s0004 · s0008 (감음만) → 둘 다 test | ⚠️ test 감음 재현율이 두 사람에 영향받음 |
| 뜸 / 감음 사진 크기 | 감음 중간값 91px, 뜸 82px — 크기가 라벨과 관련 | ⚠️ 64×64 로 줄이면 크기는 사라지지만 눈이 차지하는 비율 차이는 남을 수 있음 |
| 64×64 밝기 | 평균 0.3426, 표준편차 0.1024 | 참고 (정규화는 사진마다) |
| 분할 (1-11) | train 16명 59,560 / val 10명 12,724 / test 11명 12,614, **세 구간 모두 감음 49.4%** | ✅ 이 분할로 학습 |

**전처리 (`eye_preprocess.py` — 노트북과 서버가 같은 함수)**

| 단계 | 서버 (폰 얼굴 사진) | 학습 (MRL 눈 사진) |
|---|---|---|
| 자르기 `crop_eye` | MediaPipe 눈꼬리 점(33 · 133 / 362 · 263) 사이 폭 W → **한 변 S = 1.5W 정사각형**, 중심은 **눈 중심보다 0.03S 위** | 없음 (이미 눈만 잘림) |
| `prep_eye` | 흑백 → 64×64 (`cv2.INTER_AREA`) → **사진마다 평균 0 · 표준편차 1** | 같음 |

- 자르기 범위 근거 (1-10b): 뜬 눈 평균의 눈 중심이 위에서 53% · 왼쪽에서 50%, 눈 폭 66% → S ≈ 1.52W. 1-9 눈 확인값 55~70% (≈ 1.4~1.8W) 와 맞음. 감은 눈 평균의 눈썹 띠는 위에서 23%
- 사진마다 정규화하는 이유: MRL 은 적외선 흑백, 폰은 컬러라 전체 밝기 차이가 큼
- MRL 은 자른 범위가 사진마다 달라서 학습 증강에 크기 · 위치 흔들기 넣음 (2-4)

## 2. 학습 (MRL → `eye_cnn.pt`)

- **실행 순서:** 1-1 · 1-2 · 1-3 (드라이브 · 내려받기 · `df`) → **2-1 부터**. 1-4 ~ 1-12 는 다시 안 돌려도 됨 (분할은 1-11 이 저장한 `mrl_person_split.json` 을 읽음)
- **런타임:** 런타임 → 런타임 유형 변경 → **T4 GPU**
- **전처리는 `eye_preprocess.prep_eye` 하나만** — 노트북에서 `/255` · 정규화를 따로 안 함 (서버 `face_service.py` 와 같은 함수)
- **증강은 train 에만, 노트북에만** (원본 눈 사진 → 증강 → `prep_eye`). val · test 는 증강 없이 `prep_eye` 만
- **모델:** ResNet18 ImageNet 전이학습 (딥러닝 14차시). 흑백 1채널을 3채널로 복사, 출력 1개 = **감음 확률**
- Colab 이 끊기면: 1-1 · 1-2 · 1-3 → 2-1 ~ 2-4 다시 실행 → 2-5 가 드라이브의 `last.pt` 에서 **이어서** 학습

### 2-1. 준비 — `eye_preprocess.py` 올리기 + 학습 설정

- 왼쪽 **파일 탭 → 업로드** 로 저장소의 `sleep_detector_api/services/eye_preprocess.py` 를 `/content` 에 올림 (저장소가 비공개라 git clone 대신 업로드)
- 파일을 고쳤으면 다시 올리고 이 셀 다시 실행

In [ ]:
import os, sys, time, json, datetime, importlib
import numpy as np, pandas as pd, cv2, torch

assert os.path.exists('/content/eye_preprocess.py'), '왼쪽 파일 탭에서 eye_preprocess.py 를 /content 에 올려 주세요'
sys.path.insert(0, '/content')
import eye_preprocess
importlib.reload(eye_preprocess)
from eye_preprocess import prep_eye

SIZE = 64            # 서버 config.CNN_INPUT_SIZE 와 같게
EPOCHS = 12          # 최대 (조기 종료 있음)
PATIENCE = 3         # val 손실이 이만큼 연속으로 안 좋아지면 멈춤
BATCH = 256
LR = 3e-4
PRETRAINED = True    # ImageNet 가중치 (14차시 전이학습)
NUM_WORKERS = 2      # Colab CPU 2개
SEED = 0             # 시드 비교(5번 구간)만 1 · 2 로 바꿈. 다른 설정은 그대로
RUN_SFX = '' if SEED == 0 else f'_s{SEED}'   # 시드 0 이 아니면 last · best 파일을 따로 (eye_cnn.pt 학습 파일을 덮어쓰지 않게)

TRAIN_DIR = '/content/drive/MyDrive/sleep_detector/eye_cnn'   # 체크포인트 · eye_cnn.pt · 그림 (GitHub 에 안 올림)
os.makedirs(TRAIN_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '← GPU 아님: 런타임 유형을 T4 GPU 로')
print('torch', torch.__version__, '/ prep_eye 출력:', prep_eye(np.zeros((90, 90), np.uint8) + 128, SIZE).shape)

### 2-1b. W&B 로그인 (학습 기록 · 프로젝트 `eye-cnn`)

- **처음 한 번:** wandb.ai 무료 개인 계정 (Pro 체험 · 팀 만들기 · 카드 입력 화면이 나오면 멈춤) → wandb.ai/authorize 의 키 복사 → Colab 왼쪽 🔑 **보안 비밀** 에 이름 `WANDB_API_KEY` 로 넣고 **노트북 액세스** 켬
- 🔴 키는 노트북 · GitHub · 단톡에 안 씀. 🔴 올리는 것은 **숫자 · 표 · 그래프 · 설정값만** — `wandb.Image` 금지 (MRL · AI-Hub · 팀원 사진 전부), 사진마다 한 줄인 표 금지
- `USE_WANDB = True` 로 바꾸면 2-5 · 3-4 학습 곡선이 W&B 에 쌓이고, 2-6 · 3-5 시험 결과가 같은 run 의 summary 에 붙음
- run 이름 = `b_<모델>_lr<학습률>_s<시드>` (mrl · ft · aihub-only). id 를 이름과 같게 고정 → Colab 이 끊겨도 같은 run 에 이어서 기록
- 처음 실행 때 나오는 링크 → 프로젝트 → ⚙️ Settings → 공개 범위가 **Private** 인지 확인

In [ ]:
USE_WANDB = False    # ← True 로 바꾸면 W&B 에 기록 (보안 비밀 WANDB_API_KEY 먼저)

if USE_WANDB:
    %pip -q install wandb
    import wandb
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    # os.environ['WANDB_MODE'] = 'offline'   # 인터넷 · 로그인이 안 될 때 → 나중에 !wandb sync wandb/offline-run-*
    wandb.login()


def wb_id(key, lr):
    return f"b_{key}_lr{f'{lr:.0e}'.replace('e-0', 'e-')}_s{SEED}"     # 예) b_ft_lr3e-5_s0


def wb_start(key, lr, epochs, patience, **cfg):
    """학습 run 열기 (USE_WANDB 가 False 면 None). 하나만 바꾸고 시드는 고정, 설정은 config 에 전부"""
    if not USE_WANDB:
        return None
    return wandb.init(project='eye-cnn', name=wb_id(key, lr), id=wb_id(key, lr), group='train', resume='allow',
                      config=dict(model='resnet18', input=SIZE, batch=BATCH, seed=SEED, lr=lr, epochs=epochs, patience=patience,
                                  prep='prep_eye', **cfg),
                      settings=wandb.Settings(console='off'))     # print 가 W&B 로 같이 올라가지 않게


def wb_summary(key, lr, d):
    """끝난 학습 run 에 시험 결과 붙이기 (같은 id 로 다시 열어서)"""
    if not USE_WANDB:
        return
    wandb.init(project='eye-cnn', id=wb_id(key, lr), resume='allow', settings=wandb.Settings(console='off'))
    wandb.summary.update({k: float(v) for k, v in d.items()})
    wandb.finish()


print('W&B 기록:', 'O' if USE_WANDB else 'X (USE_WANDB = False)')

### 2-2. 사람 단위 분할 읽기 + 라벨

- 1-11 의 분할 (train 16명 / val 10명 / test 11명, 세 구간 감음 49.4%) 그대로 씀
- 라벨 `y` = **감음 1 / 뜸 0** (MRL 파일 이름의 눈 상태는 반대로 1 = 뜸)

In [ ]:
sp = json.load(open(f'{OUT_DIR}/mrl_person_split.json', encoding='utf-8'))
df['split'] = df['subject'].map(sp['split'])
assert df['split'].notna().all(), '분할 json 에 없는 피험자가 있음'
df['y'] = (1 - df['eye']).astype(np.float32)

print('분할 파일:', sp['made'], sp['method'])
display(df.groupby('split').agg(사람=('subject', 'nunique'), 장수=('y', 'size'), 감음비율=('y', 'mean')).round(3))

### 2-3. 사진 읽어 메모리에 두기 (약 1~2분)

- **원본 크기 그대로** 읽어 둠 (52~299px, 약 0.7GB) — train 은 매번 원본에서 증강 → `prep_eye`. 서버가 `crop_eye` 결과를 `prep_eye` 에 넣는 것과 같은 순서
- val · test 는 `prep_eye` 결과를 한 번만 만들어 둠

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def read_eye(p):
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    assert img is not None, p
    return img

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    IMGS = list(ex.map(read_eye, df['path'], chunksize=256))
Y = df['y'].to_numpy()
IDX = {k: np.flatnonzero(df['split'].to_numpy() == k) for k in ('train', 'val', 'test')}
PREP = {k: np.stack([prep_eye(IMGS[j], SIZE) for j in IDX[k]]) for k in ('val', 'test')}

print(f'{len(IMGS)}장 읽음 {time.time() - t0:.0f}초 / 원본 {sum(i.nbytes for i in IMGS) / 1e9:.2f}GB')
print({k: len(v) for k, v in IDX.items()}, '/ val 입력 평균 · 표준편차:', PREP['val'].mean().round(3), PREP['val'].std().round(3))

### 2-4. 증강 (train 만) + 데이터셋 + 모델

- **크기 ±15% · 위치 ±5% · 회전 ±10°** — 서버 `crop_eye` 범위가 사진마다 조금씩 다르고, MRL 도 사진마다 자른 범위가 다름 (1-9)
- **좌우 반전** — 왼눈 · 오른눈
- **밝기 · 대비 대신 감마:** `prep_eye` 가 사진마다 평균 0 · 표준편차 1 로 맞추므로 '전체를 더 밝게 · 대비를 몇 배로' 같은 직선 변화는 **정규화하면 사라져 효과 없음**. 감마(곡선)는 남음
- **흐림 · 잡음** — 폰 사진의 흔들림 · 저화질
- 아래 그림: 위 줄 = 원본(`prep_eye` 만), 아래 줄 = 증강 → `prep_eye`

In [ ]:
import torchvision
import matplotlib.pyplot as plt

def augment(img, rng):
    """원본 눈 사진(uint8) → 증강한 눈 사진(uint8). 정규화는 하지 않음 (prep_eye 가 함)"""
    if img.ndim == 3:   # MRL 은 전부 흑백이지만 혹시 몰라서
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[2] == 3 else cv2.COLOR_BGRA2GRAY)
    h, w = img.shape
    if rng.random() < 0.5:
        img = np.ascontiguousarray(img[:, ::-1])
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), rng.uniform(0.85, 1.15))
    M[:, 2] += rng.uniform(-0.05, 0.05, 2) * (w, h)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    img = 255.0 * (img / 255.0) ** rng.uniform(0.6, 1.6)
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5]))
        img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.3:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)


class EyeDS(torch.utils.data.Dataset):
    def __init__(self, split):
        self.split, self.idx = split, IDX[split]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        if self.split == 'train':
            x = prep_eye(augment(IMGS[self.idx[i]], np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))), SIZE)
        else:
            x = PREP[self.split][i]
        return torch.from_numpy(x)[None], torch.tensor(Y[self.idx[i]])


class EyeCNN(torch.nn.Module):
    """(N, 1, 64, 64) prep_eye 결과 → (N,) 감음 logit. sigmoid 하면 감음 확률"""
    def __init__(self, pretrained=True):
        super().__init__()
        self.net = torchvision.models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.net.fc = torch.nn.Linear(self.net.fc.in_features, 1)
    def forward(self, x):
        return self.net(x.expand(-1, 3, -1, -1)).squeeze(1)


loaders = {k: torch.utils.data.DataLoader(EyeDS(k), batch_size=BATCH, shuffle=(k == 'train'),
                                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                          persistent_workers=NUM_WORKERS > 0)
           for k in ('train', 'val', 'test')}

# 증강 눈으로 확인: 위 = 원본(prep_eye 만), 아래 = 증강 → prep_eye  (그림 한글 글꼴은 1-1)
rng = np.random.default_rng(1)
pick = rng.choice(IDX['train'], 8, replace=False)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for c, j in enumerate(pick):
    axes[0, c].imshow(prep_eye(IMGS[j], SIZE), cmap='gray'); axes[0, c].set_title('감음' if Y[j] else '뜸', fontsize=8)
    axes[1, c].imshow(prep_eye(augment(IMGS[j], rng), SIZE), cmap='gray')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/augment_check.png', dpi=80); plt.show()

### 2-5. 학습 (끊기면 이어서)

- 매 에포크 끝에 드라이브에 `last.pt` (이어 하기용) · `best_state.pt` (val 손실이 가장 낮은 가중치) 저장
- val 손실이 `PATIENCE` 번 연속 안 좋아지면 멈춤 (조기 종료)
- **처음부터 다시** 하려면 드라이브의 `last.pt` 를 지우고 실행

In [ ]:
model = EyeCNN(PRETRAINED).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))
loss_fn = torch.nn.BCEWithLogitsLoss()

LAST, BEST = f'{TRAIN_DIR}/last{RUN_SFX}.pt', f'{TRAIN_DIR}/best_state{RUN_SFX}.pt'
start, hist, best, bad = 0, [], float('inf'), 0
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, hist, best, bad = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
    print(f'이어서 학습: {start + 1}번째 에포크부터 (지금까지 최저 val 손실 {best:.4f})')
run = wb_start('mrl', LR, EPOCHS, PATIENCE, data='MRL', init='ImageNet', split='사람 단위')


@torch.no_grad()
def predict(split):
    """감음 확률 (numpy)"""
    model.eval()
    out = []
    for x, _ in loaders[split]:
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(model(x.to(device, non_blocking=True)).float()).cpu())
    return torch.cat(out).numpy()


def bce(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


for ep in range(start, EPOCHS):
    if bad >= PATIENCE:
        print('조기 종료'); break
    t0 = time.time(); model.train()
    tl, n = 0.0, 0
    for x, y in loaders['train']:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device, enabled=(device == 'cuda')):
            loss = loss_fn(model(x).float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * len(y); n += len(y)
    sched.step()

    p, yv = predict('val'), Y[IDX['val']]
    vl = bce(p, yv)
    pred = p >= 0.5
    rec_c, rec_o = pred[yv == 1].mean(), (~pred[yv == 0]).mean()
    row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(float((pred == yv).mean()), 4),
               val_closed_recall=round(float(rec_c), 4), val_open_recall=round(float(rec_o), 4),
               sec=round(time.time() - t0))
    hist.append(row); print(row)
    if run:
        wandb.log(row, step=row['epoch'])     # x축 = 에포크
    if vl < best:
        best, bad = vl, 0
        torch.save(model.net.state_dict(), BEST)
    else:
        bad += 1
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    epoch=ep, hist=hist, best=best, bad=bad), LAST)

if run:
    wandb.summary.update(dict(stopped_epoch=len(hist), best_val_loss=round(best, 4))); wandb.finish()
hist_df = pd.DataFrame(hist); display(hist_df)
hist_df.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', figsize=(6, 3)); plt.savefig(f'{TRAIN_DIR}/loss.png', dpi=80); plt.show()

### 2-6. test 평가 (가장 좋았던 가중치)

- 기준 0.5 (학습 당시 서버 기준, 10/28 부터 0.4)에서 정확도 · 감음 재현율(졸음을 놓치지 않는가) · 뜸 재현율(괜히 경고하지 않는가)
- **애매한 확률 0.4~0.6 비율** — 이 구간은 서버에서 '경고' 쪽으로 다룸
- 사람별 · 센서별 · 안경별 정확도 — s0004 · s0008 (감음만 있는 사람) 은 test 에 있음
- ⚠️ 여기 숫자는 **MRL(적외선)** 기준. 실제 성능은 AI-Hub 173 (3번) · 팀 사진 (KPI ①) · AI-Hub 548 (6번)으로 따로 잼

In [ ]:
model.net.load_state_dict(torch.load(BEST, map_location=device, weights_only=True))
p_test, y_test = predict('test'), Y[IDX['test']]
t = df.iloc[IDX['test']].copy()
t['p'], t['pred'] = p_test, (p_test >= 0.5).astype(int)
t['ok'] = (t['pred'] == t['y']).astype(int)

cm = pd.crosstab(t['y'].map({1: '정답 감음', 0: '정답 뜸'}), t['pred'].map({1: '예측 감음', 0: '예측 뜸'}))
test_metrics = dict(
    acc=round(float(t['ok'].mean()), 4),
    closed_recall=round(float(t.loc[t.y == 1, 'ok'].mean()), 4),
    open_recall=round(float(t.loc[t.y == 0, 'ok'].mean()), 4),
    closed_precision=round(float(t.loc[t.pred == 1, 'y'].mean()), 4),
    loss=round(bce(p_test, y_test), 4),
    ambiguous_0_4_to_0_6=round(float(((t.p >= 0.4) & (t.p <= 0.6)).mean()), 4),
)
print(test_metrics); display(cm)
wb_summary('mrl', LR, {f'mrl_test_{k}': v for k, v in test_metrics.items()})

print('\n기준값별 (서버 CNN_CLOSED_PROB 후보)')
display(pd.DataFrame([dict(기준=th, 정확도=((p_test >= th) == y_test).mean(), 감음재현율=(p_test[y_test == 1] >= th).mean(),
                           뜸재현율=(p_test[y_test == 0] < th).mean()) for th in (0.3, 0.4, 0.5, 0.6, 0.7)]).round(4))

by_subject = t.groupby('subject').agg(장수=('ok', 'size'), 감음비율=('y', 'mean'), 정확도=('ok', 'mean')).round(3)
display(by_subject.sort_values('정확도'))
for col in ('sensor', 'glasses', 'reflection'):
    display(t.groupby(col).agg(장수=('ok', 'size'), 정확도=('ok', 'mean')).round(3))

# 틀린 사진 보기 (확신하고 틀린 것부터)
wrong = t[t.ok == 0].assign(conf=lambda d: (d.p - 0.5).abs()).sort_values('conf', ascending=False).head(16)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax in axes.flat:
    ax.axis('off')
for ax, (_, r) in zip(axes.flat, wrong.iterrows()):
    ax.imshow(IMGS[r.name], cmap='gray')
    ax.set_title(f"{r.subject} {'감음' if r.y else '뜸'} p={r.p:.2f}", fontsize=7)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/test_wrong.png', dpi=80); plt.show()

### 2-7. CPU 속도 (서버는 GPU 없음)

- 사진 1장: `prep_eye` + CNN, CPU 스레드 1개 / 전체

In [ ]:
cpu_model = EyeCNN(pretrained=False)
cpu_model.net.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=True))
cpu_model.eval()
sample = IMGS[IDX['test'][0]]

def one(img):
    with torch.no_grad():
        return torch.sigmoid(cpu_model(torch.from_numpy(prep_eye(img, SIZE))[None, None])).item()

latency = {}
for th in (1, torch.get_num_threads()):
    torch.set_num_threads(th)
    for _ in range(5):
        one(sample)
    t0 = time.perf_counter()
    for _ in range(50):
        one(sample)
    latency[f'{th}스레드_ms'] = round((time.perf_counter() - t0) / 50 * 1000, 2)
print('사진 1장 (눈 하나):', latency, '— 서버는 눈 2개라 약 2배')

### 2-8. `eye_cnn.pt` 저장 (드라이브 → 서버 `ml_models/`, GitHub 에 안 올림)

- `state_dict` 는 **torchvision `resnet18` 그대로의 이름** (서버에서 클래스를 따로 안 만들어도 됨, 2-9)
- 50MB 이하 (ResNet18 은 약 45MB)

In [ ]:
assert SEED == 0, '시드 비교 학습은 eye_cnn.pt 로 저장하지 않음 — 2-8 건너뛰기'
PT = f'{TRAIN_DIR}/eye_cnn.pt'
torch.save(dict(
    state_dict=torch.load(BEST, map_location='cpu', weights_only=True),
    arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
    output='logit 1개, sigmoid = 감음 확률',
    prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
    crop='eye_preprocess.crop_eye (S = 1.5W, 중심은 눈 중심보다 0.03S 위)',
    data='MRL Eye Dataset, 사람 단위 분할 (mrl_person_split.json)',
    epochs=len(hist), best_val_loss=round(best, 4), test=test_metrics, cpu_latency=latency,
    made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),   # str 로: weights_only 로 읽을 수 있게
), PT)
mb = os.path.getsize(PT) / 1e6
print(f'저장: {PT}  {mb:.1f}MB', '✅' if mb < 50 else '❌ 50MB 넘음')

# 저장한 파일을 다시 읽어서 같은 값이 나오는지 (서버에서 읽는 방법 그대로)
ck = torch.load(PT, map_location='cpu', weights_only=True)
m2 = torchvision.models.resnet18(weights=None); m2.fc = torch.nn.Linear(512, 1)
m2.load_state_dict(ck['state_dict']); m2.eval()
with torch.no_grad():
    x = torch.from_numpy(PREP['test'][:64])[:, None]
    same = torch.allclose(torch.sigmoid(m2(x.expand(-1, 3, -1, -1)).squeeze(1)), torch.sigmoid(cpu_model(x)), atol=1e-5)
print('다시 읽어도 같은 결과:', same)

### 2-9. 서버에서 쓰는 법 (참고용 — 실제 서버 코드는 A 의 `face_service.py`)

```python
import torch, torchvision
import config
from services.eye_preprocess import crop_eye, prep_eye

def load_eye_cnn():
    ck = torch.load(config.CNN_MODEL_PATH, map_location="cpu", weights_only=True)
    m = torchvision.models.resnet18(weights=None)
    m.fc = torch.nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(ck["state_dict"])
    return m.eval()

@torch.no_grad()
def closed_prob(model, image, eye_points):
    """얼굴 사진(BGR) + 한 눈의 MediaPipe 눈 점 → 감음 확률 (0~1)"""
    x = torch.from_numpy(prep_eye(crop_eye(image, eye_points), config.CNN_INPUT_SIZE))[None, None]
    return torch.sigmoid(model(x.expand(-1, 3, -1, -1))).item()
```

- 두 눈의 확률 평균 ≥ `CNN_CLOSED_PROB` → 감음 (처음 0.5, **10/28 채택 때 0.4**)
- 경고 취소는 CNN · EAR 둘 다 '확실히 뜸'일 때만 (CNN 0.3 미만 + EAR 0.26 이상)
- torch · torchvision (CPU 판)은 `sleep_detector_api/tools/requirements-cnn.txt` 로 설치 (A · B · 팀장 PC 만, 없으면 서버가 EAR 만 씀)

### 2-10. 학습 결과 (Colab T4, 에포크당 약 37초)

| 항목 | 값 |
|---|---|
| 에포크 (조기 종료) | 4 에포크에서 멈춤 — **1 에포크 가중치가 최고** (그 뒤 train 손실만 줄고 val 손실은 늘어남) |
| 최저 val 손실 | 0.0986 (val 정확도 96.9%) |
| test 정확도 | **95.9%** (12,614장, 11명) |
| test 감음 재현율 / 뜸 재현율 | **97.5%** / 94.4% (감음 정밀도 94.5%) — 졸음을 놓치는 쪽보다 괜히 감음이라 하는 쪽이 조금 많음 |
| 기준값별 | 0.3: 감음 98.5% · 뜸 91.3% / **0.5: 97.5% · 94.4%** / 0.7: 96.4% · 97.0% |
| 애매한 확률(0.4~0.6) 비율 | 1.9% |
| 가장 낮은 사람 · 센서 | s0022 77.5%, s0028 90.2% (저화질) / 센서 01 95.8% · 02 98.6%, 안경 95.1% · 안경 없음 96.2% |
| 틀린 사진 (2-6 그림) | 안경 반사로 눈이 안 보이는 '뜸' 사진을 감음으로 · 가늘게 뜬 듯한 '감음' 사진을 뜸으로 (MRL 라벨 자체가 애매한 사진) |
| CPU 속도 (눈 1개, 1스레드) | Colab 13.3ms (개발 PC 약 5ms) → 두 눈 약 10~30ms |
| `eye_cnn.pt` 크기 | 44.8MB (50MB 안) · `weights_only=True` 로 다시 읽기 확인 |

- ⚠️ 모두 MRL(적외선) test 기준
- 3번 구간 비교의 'MRL만' = **이 모델 (MRL test 95.9%)**. 참고: 10.27 다른 GPU 런타임에서 2번 구간을 다시 학습했을 때는 MRL test 96.9% 였지만, AI-Hub 에서는 이 모델과 **반대 방향으로** 틀림 (3-7 · 5-3)

## 3. AI-Hub 173 추가 학습(미세조정) (한국인 · 실제 차량)

- **왜:** MRL 은 외국인 · 적외선 · 실험실 데이터라 MRL test 95.9% 가 우리 앱 성능을 뜻하지 않음 (강사님 피드백) → 한국인 운전자 데이터로 다시 재고 미세조정함
- **데이터:** AI-Hub 「졸음운전 예방을 위한 운전자 상태 정보 영상」 (dataSetSn=173) — **실제도로환경** (버스 · 승용 · 택시 · 트럭) + **통제환경**
- **자르기는 서버와 똑같이:** 얼굴 사진 → MediaPipe 눈 점 → `crop_eye` → `prep_eye`. JSON 눈 상자로 자르지 않음 (상자는 MediaPipe 가 같은 눈을 찾았는지 확인용). **라벨만 JSON 에서** (`Leye` · `Reye` 의 `Opened`)
- MediaPipe 가 얼굴을 못 찾은 사진은 버림 (서버에서도 CNN 까지 가지 않음)
- **운전자 단위 분할**, test 는 **실제 도로 운전자만**
- 비교 모델 3개: **MRL만** (`eye_cnn.pt`, 2번) / **미세조정** (MRL → AI-Hub, 학습률 1/10) / **AI-Hub만** (ImageNet 에서 시작) × MRL test · AI-Hub test
- 🔴 **재배포 금지:** AI-Hub 원본 · 자른 눈 사진(`.npz`)은 **본인 드라이브에만**. GitHub · 단톡 · 공유 드라이브 X. 숫자 · 학습된 모델은 공유 가능
- 🔴 GitHub 에 올리기 전 **모든 출력 지우기** (3-1 눈 사진 그림)

**라벨 JSON 확인 결과 (10.27, Validation 라벨 2개 — PC 에서 숫자만 확인)**

| 확인 | 실제도로환경 | 통제환경 |
|---|---|---|
| JSON 수 · 운전자 | 23,514 · 63명 (버스 23 · 승용 23 · 택시 8 · 트럭 9) | 12,563 · 27명 |
| 폴더 · 파일 이름 | `1.버스/R_237_40_M/R_237_40_M_20_M0_G0_C0_02.json` (R_번호_나이_성별_장면_마스크_안경_담배_프레임) | `045_G1/045_G1_01_<조명>_<행동>_<상태>_<날짜>_<시각>_<프레임>.json` |
| 운전자 번호 | 폴더 번호 = `UserInfo.ID` (불일치 0) | 폴더 앞 번호 = `UserInfo.ID` |
| 사진 | 720×1280, `Channel` 1 | 800×1280, `Channel` 1 |
| `Leye` 가 사진 왼쪽인가 | 99.98% → **`Leye` = 사진 왼쪽 눈 = 서버 `LEFT_EYE`(33 …)** | 99.85% |
| 감음 (두 눈 다) | 약 8% (운전자별 1~42%) | 약 28% |
| 한쪽만 감음 | 4.2% → 눈 하나씩 라벨을 따로 씀 | 5.2% |

- 감음이 적음 (실제 도로) → train 에서 감음 · 뜸을 **반반 뽑음** (MRL 학습과 같은 비율). 평가는 감음 재현율 · 뜸 재현율을 따로 봄
- `Channel` 1 → 원본도 흑백 (적외선 카메라). 3-1 그림으로 확인

**받기 · 자르기는 PC 에서** (`notebooks/aihub_crop.py`): AI-Hub 는 해외 IP 다운로드를 막아 Colab 에서 못 받음 (HTTP 502)
1. PC: `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub_crop.py download` → `crop` (서버와 같은 mediapipe 1.1.0 · `crop_eye`)
2. PC 의 `D:\aihub173\eye_npz\` 폴더 안 파일을 드라이브 `sleep_detector/aihub_eye/` 에 올림 (본인 드라이브만)
3. Colab: 1-1 · 1-2 · 1-3 → 2-1 ~ 2-4 (MRL test · `augment` · `EyeCNN`) → **3-1 부터**

### 3-1. 자른 눈 사진 읽기 (PC 의 `aihub_crop.py` 결과)

- 드라이브 `sleep_detector/aihub_eye/` 의 `crops_*.npz` + `summary.json`
- 눈 하나 = 한 줄: 96×96 흑백 눈 사진 + 라벨 (`y` 감음 1 / 뜸 0) · 운전자 번호 · `env` (road 실제도로 / ctrl 통제) · 차종 · 안경 · 마스크 · 성별 · 나이(10살 단위)
- PC 에서 이미 버린 것: 얼굴 못 찾음 · 안 보이는 눈 · MediaPipe 눈 중심이 JSON 눈 상자(50% 넓힘) 밖
- 그림은 눈만 (얼굴은 안 보여줌). GitHub 올리기 전 출력 지우기

In [ ]:
import collections, glob
CACHE_DIR = '/content/drive/MyDrive/sleep_detector/aihub_eye'   # 본인 드라이브에만
SUMMARY = json.load(open(f'{CACHE_DIR}/summary.json', encoding='utf-8'))
missing = [n for n in SUMMARY['chunks'] if not os.path.exists(f'{CACHE_DIR}/{n}')]
assert not missing, f'드라이브에 없는 파일 {len(missing)}개: {missing[:3]}'

parts = [np.load(f'{CACHE_DIR}/{n}') for n in SUMMARY['chunks']]
CROPS = np.concatenate([z['X'] for z in parts])
meta = pd.DataFrame({k: np.concatenate([z[k] for z in parts]) for k in ('fname', 'side', 'y', 'driver', 'env', 'cat', 'src', 'glasses', 'mask', 'gender', 'age')})
meta = meta.rename(columns={'fname': 'file'})
meta['gender'] = meta['gender'].map({1: '남', 2: '여'})   # JSON UserInfo.Gender (1 남 · 2 여)
AY = meta['y'].to_numpy(np.float32)
MAX_PER_DRIVER = SUMMARY['max_per_driver']

c = SUMMARY['counts']
print(f"PC 에서 만든 날: {SUMMARY['made']} / 사진 {c['files']}장 → 얼굴 없음 {c.get('noface', 0)} ({c.get('noface', 0) / c['files']:.1%})"
      f" · 안 보이는 눈 {c.get('eye_not_visible', 0)} · 상자 밖 {c.get('outside_box', 0)} → 쓰는 눈 {len(meta)}")
print(f'메모리 {CROPS.nbytes / 1e9:.2f}GB')
display(meta.groupby(['src', 'env', 'cat']).agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))
both = sorted(set(meta[meta.env == 'road'].driver) & set(meta[meta.env == 'ctrl'].driver))
print('실제도로 · 통제 양쪽에 있는 운전자 번호 (같은 사람으로 봄):', both)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row, yv in enumerate((0, 1)):
    for ax, j in zip(axes[row], np.random.default_rng(0).choice(np.flatnonzero(AY == yv), 8, replace=False)):
        ax.imshow(CROPS[j], cmap='gray'); ax.set_title(f"{'감음' if yv else '뜸'} {'왼쪽' if meta.side[j] == 'L' else '오른쪽'} {'실제도로' if meta.env[j] == 'road' else '통제'}", fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

### 3-2. 운전자 단위 분할 — test 는 실제 도로 운전자만

- 목표 장수 비율 train 70 / val 15 / test 15 (MRL 1-11 과 같은 방식 + 감음 비율 맞추기)
- test 후보 = 실제도로에만 있는 운전자. 통제환경 운전자는 train · val 에만
- 분할표는 드라이브 `aihub_person_split.json` (같은 데이터면 항상 같은 결과)

In [ ]:
per_a = meta.groupby('driver').agg(n=('y', 'size'), closed=('y', 'sum'), envs=('env', lambda s: '+'.join(sorted(set(s)))))
total_a, closed_all = per_a.n.sum(), per_a.closed.sum() / per_a.n.sum()
TGT = {'train': 0.70, 'val': 0.15, 'test': 0.15}
asplit, acnt = {}, collections.Counter()


def score(k, d):   # 이 운전자를 k 에 넣었을 때: 장수가 목표보다 얼마나 넘치나 + 감음 비율이 전체와 얼마나 다른가 (작을수록 좋음)
    n, c = acnt[k] + per_a.n[d], acnt[k + '_c'] + per_a.closed[d]
    return n / total_a - TGT[k] + 0.3 * abs(c / n - closed_all)


for d in per_a.sort_values('n', ascending=False).index:
    allowed = ['train', 'val', 'test'] if per_a.envs[d] == 'road' else ['train', 'val']
    k = min(allowed, key=lambda k: score(k, d))
    asplit[int(d)] = k; acnt[k] += per_a.n[d]; acnt[k + '_c'] += per_a.closed[d]

meta['split'] = meta.driver.map(asplit)
AIDX = {k: np.flatnonzero(meta.split.to_numpy() == k) for k in TGT}
display(meta.groupby('split').agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean'),
                                  실제도로비율=('env', lambda s: (s == 'road').mean())).round(3))
display(meta[meta.split == 'test'].groupby('cat').agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))
json.dump({'made': str(datetime.date.today()), 'max_per_driver': MAX_PER_DRIVER, 'npz_made': SUMMARY['made'], 'split': asplit},
          open(f'{CACHE_DIR}/aihub_person_split.json', 'w', encoding='utf-8'), indent=1)
APREP = {k: np.stack([prep_eye(CROPS[j], SIZE) for j in AIDX[k]]) for k in ('val', 'test')}

### 3-3. 학습 함수 — 미세조정 (MRL → AI-Hub) · AI-Hub만

- train: `augment`(2-4 와 같음) → `prep_eye`, 감음 · 뜸을 **반반 뽑기** (`WeightedRandomSampler`)
- **미세조정:** `eye_cnn.pt` 가중치에서 시작, 학습률 `LR / 10` (3e-5), 최대 5 에포크, val 손실 2번 연속 안 좋아지면 멈춤
- **AI-Hub만:** ImageNet 에서 시작, 2번 구간과 같은 설정 (`LR` · `EPOCHS` · `PATIENCE`)
- 끊기면 다시 실행 → 드라이브의 `aihub_<이름>_last.pt` 에서 이어서

In [ ]:
class AiDS(torch.utils.data.Dataset):
    def __init__(self, split):
        self.split, self.idx = split, AIDX[split]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = prep_eye(augment(CROPS[j], np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))), SIZE) if self.split == 'train' else APREP[self.split][i]
        return torch.from_numpy(x)[None], torch.tensor(AY[j])


ytr = AY[AIDX['train']]
w = np.where(ytr == 1, 0.5 / ytr.mean(), 0.5 / (1 - ytr.mean()))
aloader = {'train': torch.utils.data.DataLoader(AiDS('train'), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                                sampler=torch.utils.data.WeightedRandomSampler(w, len(w), replacement=True))}
for k in ('val', 'test'):
    aloader[k] = torch.utils.data.DataLoader(AiDS(k), batch_size=BATCH, num_workers=NUM_WORKERS)


@torch.no_grad()
def predict_arr(m, X, bs=512):
    """prep_eye 결과 (N, 64, 64) → 감음 확률"""
    m.eval(); out = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(X[i:i + bs])[:, None].to(device)
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(m(x).float()).cpu())
    return torch.cat(out).numpy()


def metrics(p, y, th=0.5):
    pr = p >= th
    rc, ro = float(pr[y == 1].mean()), float((~pr[y == 0]).mean())
    return dict(acc=round(float((pr == y).mean()), 4), balanced_acc=round((rc + ro) / 2, 4), closed_recall=round(rc, 4),
                open_recall=round(ro, 4), closed_precision=round(float(y[pr].mean()) if pr.any() else float('nan'), 4), n=int(len(y)))


def fit(name, init_state, lr, epochs, patience):
    m = EyeCNN(pretrained=init_state is None).to(device)
    if init_state is not None:
        m.net.load_state_dict(init_state)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    last, best_p = f'{TRAIN_DIR}/aihub_{name}{RUN_SFX}_last.pt', f'{TRAIN_DIR}/aihub_{name}{RUN_SFX}_best.pt'
    start, h, best_l, bad_n = 0, [], float('inf'), 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, h, best_l, bad_n = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
        print(f'{name}: {start + 1}번째 에포크부터 이어서')
    run = wb_start(WB_KEY[name], lr, epochs, patience, split='운전자 단위',
                   data='MRL→AI-Hub' if init_state is not None else 'AI-Hub', init='MRL만' if init_state is not None else 'ImageNet')
    yv = AY[AIDX['val']]
    for ep in range(start, epochs):
        if bad_n >= patience:
            print('조기 종료'); break
        t0 = time.time(); m.train(); tl = n = 0
        for x, y in aloader['train']:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        p = predict_arr(m, APREP['val']); vl = bce(p, yv); pred = p >= 0.5
        row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(float((pred == yv).mean()), 4),
                   val_closed_recall=round(float(pred[yv == 1].mean()), 4), val_open_recall=round(float((~pred[yv == 0]).mean()), 4),
                   sec=round(time.time() - t0))
        h.append(row); print(name, row)
        if run:
            wandb.log(row, step=row['epoch'])
        if vl < best_l:
            best_l, bad_n = vl, 0; torch.save(m.net.state_dict(), best_p)
        else:
            bad_n += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=h, best=best_l, bad=bad_n), last)
    if run:
        wandb.summary.update(dict(stopped_epoch=len(h), best_val_loss=round(best_l, 4))); wandb.finish()
    return torch.load(best_p, map_location='cpu', weights_only=True), h


WB_KEY = {'finetune': 'ft', 'scratch': 'aihub-only'}     # W&B run 이름
loss_fn = torch.nn.BCEWithLogitsLoss()
STATE = {'MRL만': torch.load(f'{TRAIN_DIR}/eye_cnn.pt', map_location='cpu', weights_only=True)['state_dict']}
print('train 눈', len(AIDX['train']), '/ 감음', f'{ytr.mean():.1%}', '→ 반반 뽑기')

### 3-4. 학습 실행

In [ ]:
STATE['미세조정'], HIST_FT = fit('finetune', STATE['MRL만'], LR / 10, 5, 2)
STATE['AI-Hub만'], HIST_AI = fit('scratch', None, LR, EPOCHS, PATIENCE)

### 3-5. 세 모델 비교 (MRL test · AI-Hub test)

- 기준 0.5 (당시 서버 기준). AI-Hub test 는 감음이 적어 **균형 정확도** (감음 재현율과 뜸 재현율의 평균) 같이 봄
- AI-Hub test 는 차종별 (트럭 = 우리 대상) · 안경 · 마스크 · 성별 · 나이별로도 봄
- 틀린 눈 사진 16장 (미세조정 모델, 확신하고 틀린 것부터) — 가는 눈 · 반쯤 감은 눈이 어떻게 틀리는지
- test 운전자는 학습 · val 에 안 쓴 사람만 (3-2 운전자 단위 분할)

In [ ]:
ta = meta.iloc[AIDX['test']].copy()
rows_cmp, P_AI = [], {}
for k in ('MRL만', '미세조정', 'AI-Hub만'):
    m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(STATE[k])
    p_mrl, P_AI[k] = predict_arr(m, PREP['test']), predict_arr(m, APREP['test'])
    rows_cmp.append({'모델': k, **{f'MRL_{a}': b for a, b in metrics(p_mrl, Y[IDX['test']]).items()},
                     **{f'AIHub_{a}': b for a, b in metrics(P_AI[k], AY[AIDX['test']]).items()}})
cmp_df = pd.DataFrame(rows_cmp).set_index('모델')
for k, key, lr in (('MRL만', 'mrl', LR), ('미세조정', 'ft', LR / 10), ('AI-Hub만', 'aihub-only', LR)):
    if SEED == 0:                            # MRL만 = eye_cnn.pt (시드 0)
        wb_summary(key, lr, cmp_df.loc[k].to_dict())
display(cmp_df[['MRL_acc', 'MRL_closed_recall', 'MRL_open_recall', 'AIHub_acc', 'AIHub_balanced_acc', 'AIHub_closed_recall', 'AIHub_open_recall', 'AIHub_closed_precision']])

for col in ('cat', 'glasses', 'mask', 'gender', 'age'):
    display(pd.DataFrame({k: {g: metrics(P_AI[k][(ta[col] == g).to_numpy()], AY[AIDX['test']][(ta[col] == g).to_numpy()])['balanced_acc']
                              for g in sorted(ta[col].unique())} for k in P_AI}).rename_axis(col))

# 틀린 눈 사진 (미세조정, 확신하고 틀린 것부터) — 눈만, 얼굴 X. GitHub 올리기 전 출력 지우기
yt = AY[AIDX['test']]
wrong = np.flatnonzero((P_AI['미세조정'] >= 0.5) != (yt == 1))
wrong = wrong[np.argsort(-np.abs(P_AI['미세조정'][wrong] - 0.5))][:16]
print(f"틀린 눈 {int(((P_AI['미세조정'] >= 0.5) != (yt == 1)).sum())} / {len(yt)}")
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, wrong):
    r = ta.iloc[i]
    ax.imshow(CROPS[AIDX['test'][i]], cmap='gray')
    ax.set_title(f"{'감음' if r.y else '뜸'} p={P_AI['미세조정'][i]:.2f}\n{'안경 ' if r.glasses else ''}{'마스크' if r['mask'] else ''}", fontsize=7)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/aihub_test_wrong.png', dpi=80); plt.show()

print('기준값별 (미세조정)')
display(pd.DataFrame([{'기준': th, **metrics(P_AI['미세조정'], AY[AIDX['test']], th)} for th in (0.3, 0.4, 0.5, 0.6, 0.7)]).set_index('기준'))
print('기준값별 (MRL만) — 같은 AI-Hub test')
display(pd.DataFrame([{'기준': th, **metrics(P_AI['MRL만'], AY[AIDX['test']], th)} for th in (0.3, 0.5, 0.7)]).set_index('기준'))

### 3-6. 저장 — `eye_cnn_aihub.pt` (미세조정) · 비교 숫자

- 형식은 `eye_cnn.pt` 와 같음 → 서버 코드 그대로, `config.CNN_MODEL_PATH` 파일만 바꾸면 됨 (**10/28 채택**, 서버 반영됨)
- AI-Hub만 모델은 `eye_cnn_aihub_only.pt`. 비교 숫자 `aihub_compare.json` 은 공유 가능 (사진 없음)

In [ ]:
for k, fname in (('미세조정', 'eye_cnn_aihub.pt'), ('AI-Hub만', 'eye_cnn_aihub_only.pt')):
    torch.save(dict(
        state_dict=STATE[k], arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
        output='logit 1개, sigmoid = 감음 확률',
        prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
        crop='MediaPipe 눈 점 → eye_preprocess.crop_eye (서버와 같음)',
        data=('MRL → ' if k == '미세조정' else '') + 'AI-Hub 173 졸음운전 예방을 위한 운전자 상태 정보 영상 (bbox 실제도로 + 통제), 운전자 단위 분할',
        test={'MRL': {a[4:]: float(v) for a, v in cmp_df.loc[k].items() if a.startswith('MRL_')},   # float: weights_only=True 로 읽히게
              'AIHub': {a[6:]: float(v) for a, v in cmp_df.loc[k].items() if a.startswith('AIHub_')}},
        made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),
    ), f'{TRAIN_DIR}/{fname}')
    torch.load(f'{TRAIN_DIR}/{fname}', map_location='cpu', weights_only=True)  # 서버와 같은 방식으로 다시 읽기
    print('저장:', fname, f"{os.path.getsize(f'{TRAIN_DIR}/{fname}') / 1e6:.1f}MB", '· weights_only=True 로 읽힘')

json.dump({'made': str(datetime.datetime.now())[:16], 'compare': cmp_df.reset_index().to_dict('records'),
           'hist_finetune': HIST_FT, 'hist_aihub_only': HIST_AI,
           'test_drivers': int(ta.driver.nunique()), 'test_eyes': int(len(ta))},
          open(f'{TRAIN_DIR}/aihub_compare.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: aihub_compare.json')

### 3-7. 결과 (10.27 Colab T4)

**AI-Hub test = 실제 도로 운전자만, 눈 33,190개 / 기준 0.5**

| 모델 | MRL test 정확도 | AI-Hub test 정확도 | AI-Hub 균형 정확도 | AI-Hub 감음 재현율 / 뜸 재현율 | 감음 정밀도 | 팀 사진 (KPI ①, 10/28) |
|---|---|---|---|---|---|---|
| MRL만 (`eye_cnn.pt`) | **95.9%** | 76.2% | 82.5% | 91.7% / **73.2%** | **40.0%** | 완전히 감은 눈 100% |
| **미세조정** (`eye_cnn_aihub.pt`) | **96.2%** | **95.2%** | **93.8%** | **91.7% / 95.9%** | **81.2%** | 완전히 감은 눈 100% · 진짜 졸음 취소 0건 → **채택** |
| AI-Hub만 (`eye_cnn_aihub_only.pt`) | 92.9% | 95.0% | 93.4% | 91.1% / 95.8% | 80.8% | (안 잼) |

- **MRL만은 한국인 실제 운전자의 뜬 눈 4개 중 1개 이상을 '감음'으로 봄** (뜸 재현율 73.2%) → '감음' 판정 중 실제 감은 눈은 40% 뿐 = **괜한 졸음 경고가 쏟아짐** → 강사님 지적을 숫자로 확인
- 같은 코드로 다른 런타임에서 다시 학습한 MRL 모델 (MRL test 96.9%) 은 반대로 **감은 눈을 3개 중 1개 놓침** (감음 재현율 66.5%, 뜸 94.0%, 균형 80.2%) → **MRL 만으로 학습하면 한국인 실제 운전자에서 어느 쪽으로 틀릴지조차 들쭉날쭉** (균형 정확도 80~82%)
- **미세조정: 균형 정확도 82.5% → 93.8%, 뜸 재현율 73.2% → 95.9%, 감음 정밀도 40.0% → 81.2%, 감음 재현율 91.7% 유지, MRL test 95.9% → 96.2% (오히려 조금 오름)**
- AI-Hub만은 AI-Hub 에서 비슷하지만 MRL test 92.9% 로 떨어짐 → **미세조정 모델 사용**
- 학습: 미세조정 4 에포크 가중치 최고 (val 손실 0.1475, val 정확도 94.7%) / AI-Hub만 1 에포크 최고 (val 손실 0.1440) → 4 에포크에서 조기 종료. 에포크당 약 92~97초

**나눠 보기 (AI-Hub test 균형 정확도)**

| 구분 | MRL만 | 미세조정 | AI-Hub만 |
|---|---|---|---|
| 버스 · 승용 · 택시 | 82.6% · 83.2% · 75.7% | 93.1% · 95.2% · 89.8% | 93.0% · 94.8% · 89.4% |
| **트럭 (우리 대상)** | 86.0% | **96.5%** | 95.4% |
| 안경 안 씀 / 씀 | 83.7% / 80.6% | 95.3% / 91.3% | 94.9% / 90.9% |
| **마스크 안 씀 / 씀** | 82.5% / 82.3% | **94.1% / 93.2%** | 93.6% / 93.1% |
| 남 / 여 | 82.3% / 83.9% | 93.5% / 97.9% | 93.1% / 97.7% |
| 20 · 30 · 40 · 50 · 60 · 70대 | 68.4 · 81.0 · 84.4 · 82.2 · 83.8 · 78.2% | 94.7 · 93.4 · 95.0 · 93.4 · 93.0 · 87.5% | 95.0 · 93.0 · 94.1 · 92.9 · 93.4 · 90.0% |

- 마스크를 써도 (얼굴만 찾으면) 차이 1%p — 마스크는 눈 판정에 거의 영향 없음
- 안경이 가장 어려움 (약 4%p 낮음, 렌즈 반사 — MRL 때와 같은 경향). 70대는 사람 수가 적어 흔들릴 수 있음
- 틀린 눈 1,598 / 33,190 (4.8%)

**기준값별 (미세조정, 서버 `CNN_CLOSED_PROB` 후보)**

| 기준 | 감음 재현율 | 뜸 재현율 | 균형 정확도 | 감음 정밀도 |
|---|---|---|---|---|
| 0.3 | 94.1% | 93.9% | 94.0% | 75.0% |
| **0.4 (채택)** | 93.2% | 95.1% | **94.1%** | 78.6% |
| 0.5 | 91.7% | 95.9% | 93.8% | 81.2% |
| 0.6 | 89.9% | 96.6% | 93.2% | 83.6% |
| 0.7 | 87.9% | 97.2% | 92.5% | 85.9% |

- 졸음을 놓치지 않는 쪽 + 균형 정확도 최고 → **0.4 채택** (10/28, 팀 사진 KPI ① 함께 확인)
- MRL만은 기준을 바꿔도 균형 정확도 77~84% (0.3: 감음 96.3% · 뜸 58.4% / 0.7: 감음 82.0% · 뜸 85.4%) — 기준값으로는 해결 안 됨

**⚠️ 한계**
- AI-Hub 173 도 **적외선** 영상 (눈동자 반사점 · `Channel` 1) → "외국인 → 한국인" 은 해결, "적외선 → 폰 컬러 카메라" 차이는 남음 → 6번 구간에서 컬러 사진으로 확인
- 마스크 · 고개 돌림 사진 일부는 MediaPipe 가 얼굴을 못 찾아 학습 · 평가 모두에서 빠짐 (서버도 같음 → 앱 판단대로 확정)
- 출처: AI-Hub 「졸음운전 예방을 위한 운전자 상태 정보 영상」 (한국지능정보사회진흥원). 원본 · 자른 눈 사진은 재배포하지 않음

### 3-8. Grad-CAM — CNN 이 눈의 어디를 보고 판단하나 (기획안 4-4)

- **Grad-CAM:** 모델이 '감음' 또는 '뜸'으로 판정할 때 그 판정에 크게 영향을 준 자리를 색으로 칠한 그림 (빨강일수록 많이 봄)
  - 판정한 쪽의 근거를 칠함 — 감음으로 본 눈은 '감음' 근거, 뜸으로 본 눈은 '뜸' 근거 (기준 0.4, 서버와 같음)
- **층:** ResNet18 의 `layer2` (입력 64×64 → 8×8 지도). `layer3` 은 4×4, `layer4` 는 2×2 라 너무 뭉개짐 → PC 에서 비교해 보고 `layer2` 로 정함
- **눈 사진:** 미세조정 모델 · AI-Hub test 눈 (3-5 와 같은 눈만, 얼굴 X) — 맞힌 감은 눈 · 맞힌 뜬 눈 · 틀린 눈 각 6개
- **확인할 것:** 감은 눈은 **눈꺼풀이 닫힌 선**, 뜬 눈은 **눈동자 · 흰자**를 봐야 정상. 눈썹 · 사진 가장자리 · 안경테를 보면 엉뚱한 근거
- **PC 시험 (11/02, AI-Hub 548 컬러 사진 8장):** 감은 눈 → 눈꺼풀 선 · 뜬 눈 → 눈동자에 빨강 → 눈을 보고 판단하는 것 확인 (사진은 AI-Hub 라 노션 · GitHub 에 안 올림)
- 실행: 3-1 ~ 3-5 다음 (`STATE` · `APREP` · `AIDX` · `P_AI` 필요). GitHub 올리기 전 출력 지우기

In [ ]:
import torch.nn.functional as Fnn

TH = 0.4                                   # 서버 CNN_CLOSED_PROB


def gradcam(m, X, layer):
    # X: prep_eye 결과 (N, 64, 64) → (판정한 쪽 근거 지도 (N, 64, 64) 0~1, 감음 확률 (N,))
    acts, grads = [], []
    h1 = layer.register_forward_hook(lambda mod, i, o: acts.append(o))
    h2 = layer.register_full_backward_hook(lambda mod, gi, go: grads.append(go[0]))
    x = torch.from_numpy(np.asarray(X, np.float32))[:, None].to(device)
    logit = m(x)
    sign = torch.where(torch.sigmoid(logit) >= TH, 1.0, -1.0)   # 감음으로 봤으면 감음 쪽, 뜸이면 뜸 쪽
    m.zero_grad(); (logit * sign).sum().backward()
    h1.remove(); h2.remove()
    a, g = acts[0], grads[0]
    cam = torch.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = Fnn.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[:, 0]
    cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
    return cam.detach().cpu().numpy(), torch.sigmoid(logit).detach().cpu().numpy()


cam_model = EyeCNN(pretrained=False).to(device).eval()
cam_model.net.load_state_dict(STATE['미세조정'])
yt, pt = AY[AIDX['test']], P_AI['미세조정']
rng = np.random.default_rng(0)
groups = {'맞힌 감은 눈': np.flatnonzero((yt == 1) & (pt >= TH)), '맞힌 뜬 눈': np.flatnonzero((yt == 0) & (pt < TH)),
          '틀린 눈': np.flatnonzero((yt == 1) != (pt >= TH))}
fig, axes = plt.subplots(len(groups) * 2, 6, figsize=(10, 3.6 * len(groups)))
for g, (name, idx) in enumerate(groups.items()):
    sel = rng.choice(idx, min(6, len(idx)), replace=False)
    X = APREP['test'][sel]
    cam, p = gradcam(cam_model, X, cam_model.net.layer2)
    for c in range(len(sel)):
        axes[2 * g, c].imshow(X[c], cmap='gray')
        axes[2 * g, c].set_title(f"{name}\n정답 {'감음' if yt[sel[c]] else '뜸'} · p={p[c]:.2f}", fontsize=7)
        axes[2 * g + 1, c].imshow(X[c], cmap='gray'); axes[2 * g + 1, c].imshow(cam[c], cmap='jet', alpha=0.45)
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Grad-CAM (미세조정 모델, layer2) — 빨강일수록 판정에 많이 쓴 자리', fontsize=10)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/gradcam_aihub.png', dpi=80); plt.show()

## 4. 팀 사진 · 영상 프레임 CNN 확률 → 서버 값과 맞는지 확인 (KPI ① 보조)

- **결과 (10/28):** 팀 사진 · 영상 프레임 **458장 모두 서버 값과 일치** → 노트북 학습 모델 · 전처리가 서버에 그대로 들어갔음 확인
- **KPI ① 비교는 서버 CSV 하나로** (10/28 A 와 정함): A 의 `kpi_eye_accuracy.ipynb` 는 서버 `face_service` 가 만든 `result_mrl.csv` · `result_aihub.csv` 만 읽음
  - 서버 확인 · 취소 비교는 서버 최종 판정(`method` · `result`)이 있어야 해서 이 구간 확률만으로는 못 만듦
- **이 구간은 확인용:** 같은 사진을 서버 코드를 옮긴 계산으로 다시 돌려 `cnn.csv` (`file, label, p_mrl, p_aihub`) → 서버 `eye_closed_prob` 와 같은지 봄 (4-3)
  - `result_mrl.csv` ↔ `p_mrl`, `result_aihub.csv` ↔ `p_aihub` (서버 CSV 는 모델마다 따로)
  - 서버는 소수 3자리, 여기는 4자리 → 차이 0.001 이하는 같은 값
- 계산은 **서버 `face_service` 와 똑같이**: 사진(`IMREAD_COLOR`) → MediaPipe 눈 점(픽셀) → `crop_eye` (None 이면 CNN 없음, 눈 크기는 폭 기준으로만) → `prep_eye` → **두 눈 평균** 감음 확률
  - CNN 없음 (얼굴 못 찾음 · 눈이 너무 작음 · 사진 밖) = 빈칸
- `label` = 파일 이름 속 `closed` / `half` / `open` (서버 `_label_of` 와 같은 규칙)
- **폴더 · 이름:** A 의 `tools/kpi_frames.py` 가 사진 · 영상 프레임을 `images/` **한 폴더에** 넣음 (이름 규칙은 노션 「팀원 사진 · 영상 촬영」)
  - 사람 `a` · `b` · `c` · `d` · `t`(팀장) — 영상 프레임 `a_bright_closed_v01.jpg` · 사진 `a_bright_open_s01.jpg` · 조건 사진 `a_bright_mask_open_s01.jpg` (`mask` · `sun` · `turn`)
  - 하위 폴더 `images/_full/` (원본 크기 프레임) · `images/_check/` (모아보기) 는 안 읽음 → 같은 프레임을 두 번 세지 않게
- 🔴 팀 사진은 **CNN 학습에 안 씀** (평가만). 사진 · CSV 는 드라이브에만, GitHub 에 올리기 전 출력 지우기
- **실행:** 이 구간만 따로 돌려도 됨 (1 ~ 3번 불필요). `/content` 에 `eye_preprocess.py` (main 최신) 업로드 · GPU 필요 없음

### 4-1. 준비 — 경로 · MediaPipe · 모델 두 개

In [ ]:
import os, sys, glob, csv, collections, urllib.request
import numpy as np, cv2, torch, torchvision   # pandas 없이 (서버 venv 에서도 돌게)

try:                                   # Colab 이면 드라이브
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/sleep_detector'
    sys.path.insert(0, '/content')     # eye_preprocess.py 업로드 위치
    WORK = '/content'
except ImportError:                    # PC (서버 venv) 면 저장소 · D:\aihub173
    ROOT = os.environ.get('SD_ROOT', r'D:\aihub173')
    sys.path.insert(0, os.path.abspath(os.path.join('..', 'sleep_detector_api', 'services')))
    WORK = ROOT

IMAGE_DIR = f'{ROOT}/kpi_eye/images'                  # ← kpi_frames.py 결과 폴더 (사진 _s · 영상 프레임 _v 같이, 하위 폴더는 안 읽음)
OUT_CSV = f'{ROOT}/kpi_eye/cnn.csv'                   # ← 확인용 (A 노트북은 안 읽음)
SERVER_CSVS = {'p_mrl': f'{ROOT}/kpi_eye/result_mrl.csv',      # ← A 노트북 DATA_DIR 의 서버 CSV
               'p_aihub': f'{ROOT}/kpi_eye/result_aihub.csv'}
MODELS = {'p_mrl': f'{ROOT}/eye_cnn/eye_cnn.pt',      # MRL만
          'p_aihub': f'{ROOT}/eye_cnn/eye_cnn_aihub.pt'}  # 미세조정
if not os.path.exists(MODELS['p_mrl']):               # PC: D:\aihub173\models\
    MODELS = {k: f'{ROOT}/models/{os.path.basename(v)}' for k, v in MODELS.items()}

try:
    import mediapipe as mp
except ImportError:
    %pip -q install mediapipe==1.1.0
    import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from eye_preprocess import crop_eye, prep_eye

FACE_TASK = f'{WORK}/face_landmarker.task'
if not os.path.exists(FACE_TASK):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task', FACE_TASK)
det = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_buffer=open(FACE_TASK, 'rb').read()), num_faces=1))

LEFT_EYE = [33, 160, 158, 133, 153, 144]     # face_service.py 와 같은 번호
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
SIZE = 64                                    # config.CNN_INPUT_SIZE


def load_cnn(path):                          # 서버 _get_cnn 과 같음
    ck = torch.load(path, map_location='cpu', weights_only=True)
    m = torchvision.models.resnet18(weights=None)
    m.fc = torch.nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(ck['state_dict'])
    return m.eval()


NETS = {k: load_cnn(v) for k, v in MODELS.items()}
photos = sorted(f for f in glob.glob(f'{IMAGE_DIR}/*') if f.lower().endswith(('.jpg', '.jpeg', '.png')))   # _full/ · _check/ 은 안 읽음
n_v = sum('_v' in os.path.basename(f).lower() for f in photos)
print('모델', {k: os.path.basename(v) for k, v in MODELS.items()}, '/ 사진', len(photos) - n_v, '장 · 영상 프레임', n_v, '장')
if not os.path.isdir(IMAGE_DIR):
    print('⚠️ 폴더가 없습니다:', IMAGE_DIR)

### 4-2. 사진마다 두 모델 감음 확률 → `cnn.csv`

In [ ]:
def label_of(name):                          # 서버 _label_of 와 같음
    name = name.lower()
    return next((l for l in ('closed', 'half', 'open') if l in name), '')


@torch.no_grad()
def closed_probs(path):
    """사진 1장 → {'p_mrl': …, 'p_aihub': …} (서버 _cnn_closed_prob 과 같은 계산) / CNN 없음 → 빈 dict, 이유"""
    bgr = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)   # 서버 _read_image_bgr
    if bgr is None:
        return {}, '읽기 실패'
    h, w, _ = bgr.shape
    res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks:
        return {}, '얼굴 없음'
    lm = res.face_landmarks[0]
    eyes = [np.array([(lm[i].x * w, lm[i].y * h) for i in idx]) for idx in (LEFT_EYE, RIGHT_EYE)]
    # 눈 크기는 crop_eye 의 '폭' 기준으로만 (서버 10/28: 눈 점 '높이' 8px 검사는 감은 눈에서만 CNN 이 꺼지는 버그라 삭제)
    crops = [crop_eye(bgr, pts) for pts in eyes]
    if any(c is None for c in crops):
        return {}, 'crop_eye None (눈이 너무 작음 · 사진 밖)'
    x = torch.from_numpy(np.stack([prep_eye(c, SIZE) for c in crops]))[:, None].expand(-1, 3, -1, -1)
    return {k: round(float(torch.sigmoid(m(x).squeeze(1)).mean()), 4) for k, m in NETS.items()}, ''


rows, why = [], collections.Counter()
for f in photos:
    probs, reason = closed_probs(f)
    name = os.path.basename(f)
    rows.append({'file': name, 'label': label_of(name), 'p_mrl': probs.get('p_mrl', ''), 'p_aihub': probs.get('p_aihub', '')})
    if reason:
        why[reason] += 1
with open(OUT_CSV, 'w', newline='', encoding='utf-8-sig') as fp:   # A 노트북 · 서버 result.csv 와 같은 인코딩
    wr = csv.DictWriter(fp, fieldnames=['file', 'label', 'p_mrl', 'p_aihub'])
    wr.writeheader(); wr.writerows(rows)
print('저장:', OUT_CSV, len(rows), '장')

# 확인만 (비교 표는 A 노트북에서): 라벨 없는 사진 · CNN 없는 사진 · 라벨별 확률 중간값
print('라벨 없는 사진:', [r['file'] for r in rows if not r['label']][:10])
print('CNN 없음:', sum(why.values()), '장', dict(why))
for src, is_v in (('사진', False), ('영상', True)):
    for lab in ('open', 'half', 'closed'):
        g = [r for r in rows if r['label'] == lab and r['p_mrl'] != '' and ('_v' in r['file'].lower()) == is_v]
        if g:
            print(f"{src} {lab:6s} {len(g):3d}장  감음 확률 중간값  MRL만 {np.median([r['p_mrl'] for r in g]):.3f} · 미세조정 {np.median([r['p_aihub'] for r in g]):.3f}")

### 4-3. 서버 CSV 와 맞는지 확인 (`result_mrl.csv` ↔ `p_mrl` · `result_aihub.csv` ↔ `p_aihub`)

In [ ]:
TOL = 0.001                                  # 서버 3자리 · 여기 4자리 반올림 차이


def num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


mine = {r['file']: r for r in rows}
for col, path in SERVER_CSVS.items():
    if not os.path.exists(path):
        print(f'{col}: 서버 CSV 없음 ({os.path.basename(path)}) → 건너뜀')
        continue
    with open(path, encoding='utf-8-sig') as fp:
        srv = {r['file']: num(r.get('eye_closed_prob')) for r in csv.DictReader(fp)}
    both = sorted(set(srv) & set(mine))
    same, diff, only_one = 0, [], []
    for f in both:
        a, b = num(mine[f][col]), srv[f]
        if a is None and b is None:
            same += 1
        elif a is None or b is None:
            only_one.append((f, a, b))
        elif abs(a - b) <= TOL:
            same += 1
        else:
            diff.append((f, a, b))
    print(f'{col} ↔ {os.path.basename(path)}: 같은 사진 {len(both)}장 · 같음 {same} · 다름 {len(diff)} · 한쪽만 CNN 값 {len(only_one)}'
          f' / 서버에만 {len(set(srv) - set(mine))}장 · 여기만 {len(set(mine) - set(srv))}장')
    for f, a, b in (diff + only_one)[:10]:
        print(f'   {f}: 여기 {a} · 서버 {b}')

## 5. W&B 기록 (프로젝트 `eye-cnn`)

- 강사님 요청: CNN 학습 · 시험 결과를 W&B 에 남기고, 차트 · 표를 캡처해 노션에 정리 (run 링크 같이)
- **10.27 학습 3개는 다시 학습 안 함** — 드라이브의 `last.pt` (MRL만 학습 곡선) · `aihub_compare.json` (미세조정 · AI-Hub만 곡선 + 시험 결과) 을 그대로 올림 (5-1)
- 실행 순서: **1-1 → 2-1 → 2-1b (`USE_WANDB = True`) → 5-1 · 5-2** (사진 · 데이터는 안 읽어도 됨)
- run 이름의 시드는 **2-1 의 `SEED` (= 0)** 그대로 → `b_mrl_lr3e-4_s0` · `b_ft_lr3e-5_s0` · `b_aihub-only_lr3e-4_s0`
- 🔴 사진 · 사진마다 한 줄인 표 · AI-Hub 원본 · `.npz` · 키는 안 올림. GitHub 에 올리기 전 **모든 출력 지우기**

**W&B 에 있는 run (`eye-cnn`)**

| run | 내용 |
|---|---|
| `b_mrl_lr3e-4_s0` · `b_ft_lr3e-5_s0` · `b_aihub-only_lr3e-4_s0` | 10/27 학습 3개 — 에포크별 학습 곡선 + 시험 결과 (5-1) |
| `b_ft_tables` | 세 모델 비교 · 기준값별 · 나눠 보기 표 (5-2) |
| `b_mrl_lr3e-4_s1` · `s2` | 시드 비교 (5-3) |
| `b_eval548_g1` | AI-Hub 548 컬러 사진 시험 (6번, PC 에서 올림) |

### 5-1. 10.27 학습 3개 올리기 (다시 학습 X)

- x축 = 에포크, summary = MRL test · AI-Hub test 시험 결과 (`aihub_compare.json` 의 `compare`)
- `last.pt` 가 `eye_cnn.pt` 를 만든 학습인지 먼저 확인 (에포크 수 · 최저 val 손실이 `eye_cnn.pt` 에 적힌 값과 같아야 함)
- id 고정 (`<이름>_1027`) → 다시 실행해도 run 이 늘어나지 않음

In [ ]:
assert USE_WANDB, '2-1b 에서 USE_WANDB = True 로 바꾸고 다시 실행'
CMP = json.load(open(f'{TRAIN_DIR}/aihub_compare.json', encoding='utf-8'))
MRL_HIST = torch.load(f'{TRAIN_DIR}/last.pt', map_location='cpu', weights_only=False)['hist']   # 2-5 의 last.pt
ck = torch.load(f'{TRAIN_DIR}/eye_cnn.pt', map_location='cpu', weights_only=True)
same = len(MRL_HIST) == ck['epochs'] and abs(min(r['val_loss'] for r in MRL_HIST) - ck['best_val_loss']) < 1e-4
print('MRL만 에포크', len(MRL_HIST), '/ eye_cnn.pt', ck['epochs'], '→', 'last.pt = eye_cnn.pt 학습 ✅' if same else '❌ 다른 학습의 last.pt — MRL만 곡선은 올리지 말 것')


def log_old(key, lr, model_key, hist, cfg):
    name = wb_id(key, lr)
    wandb.init(project='eye-cnn', name=name, id=f'{name}_1027', group='train-1027', resume='allow',
               config=dict(model='resnet18', input=SIZE, batch=BATCH, seed=SEED, prep='prep_eye', **cfg),
               settings=wandb.Settings(console='off'))
    for row in hist:
        wandb.log(row, step=int(row['epoch']))
    res = next(r for r in CMP['compare'] if r['모델'] == model_key)
    wandb.summary.update({k: v for k, v in res.items() if k != '모델'})
    wandb.summary.update(dict(stopped_epoch=len(hist), best_val_loss=min(r['val_loss'] for r in hist)))
    wandb.finish()


if same:
    log_old('mrl', LR, 'MRL만', MRL_HIST,
            dict(data='MRL', init='ImageNet', split='사람 단위', lr=LR, epochs=EPOCHS, patience=PATIENCE))
log_old('ft', LR / 10, '미세조정', CMP['hist_finetune'],
        dict(data='MRL→AI-Hub', init='MRL만', split='운전자 단위', lr=LR / 10, epochs=5, patience=2))
log_old('aihub-only', LR, 'AI-Hub만', CMP['hist_aihub_only'],
        dict(data='AI-Hub', init='ImageNet', split='운전자 단위', lr=LR, epochs=EPOCHS, patience=PATIENCE))

### 5-2. 표 올리기 — 세 모델 비교 · 기준값별 · 나눠 보기

- 숫자는 3-5 결과 (`aihub_compare.json` · 3-7 표 그대로). 사진 없음, 요약 표만

In [ ]:
wandb.init(project='eye-cnn', name='b_ft_tables', id='b_ft_tables_1027', group='train-1027', resume='allow',
           settings=wandb.Settings(console='off'))

cmp_t = pd.DataFrame(CMP['compare'])[['모델', 'MRL_acc', 'AIHub_acc', 'AIHub_balanced_acc', 'AIHub_closed_recall',
                                    'AIHub_open_recall', 'AIHub_closed_precision']]
thr = pd.DataFrame({'기준': [0.3, 0.4, 0.5, 0.6, 0.7],                       # 3-7 기준값별 (미세조정, AI-Hub test)
                    '감음 재현율': [0.941, 0.932, 0.917, 0.899, 0.879],
                    '뜸 재현율': [0.939, 0.951, 0.959, 0.966, 0.972],
                    '균형 정확도': [0.940, 0.941, 0.938, 0.932, 0.925],
                    '감음 정밀도': [0.750, 0.786, 0.812, 0.836, 0.859]})
split = pd.DataFrame([                                                     # 3-7 나눠 보기 (AI-Hub test 균형 정확도)
    ('차종', '버스', 0.826, 0.931, 0.930), ('차종', '승용', 0.832, 0.952, 0.948), ('차종', '택시', 0.757, 0.898, 0.894),
    ('차종', '트럭', 0.860, 0.965, 0.954),
    ('안경', '안 씀', 0.837, 0.953, 0.949), ('안경', '씀', 0.806, 0.913, 0.909),
    ('마스크', '안 씀', 0.825, 0.941, 0.936), ('마스크', '씀', 0.823, 0.932, 0.931),
    ('성별', '남', 0.823, 0.935, 0.931), ('성별', '여', 0.839, 0.979, 0.977),
    ('나이', '20대', 0.684, 0.947, 0.950), ('나이', '30대', 0.810, 0.934, 0.930), ('나이', '40대', 0.844, 0.950, 0.941),
    ('나이', '50대', 0.822, 0.934, 0.929), ('나이', '60대', 0.838, 0.930, 0.934), ('나이', '70대', 0.782, 0.875, 0.900),
], columns=['구분', '그룹', 'MRL만', '미세조정', 'AI-Hub만'])

wandb.log({'세 모델 비교 (MRL test · AI-Hub test)': wandb.Table(dataframe=cmp_t),
           '미세조정 기준값별 (AI-Hub test)': wandb.Table(dataframe=thr),
           '기준값별 곡선': wandb.plot.line_series(xs=thr['기준'].tolist(),
                                                ys=[thr['감음 재현율'].tolist(), thr['뜸 재현율'].tolist(), thr['균형 정확도'].tolist()],
                                                keys=['감음 재현율', '뜸 재현율', '균형 정확도'], title='미세조정 — 기준값별', xname='기준값'),
           '나눠 보기 (균형 정확도)': wandb.Table(dataframe=split)})
wandb.finish()

### 5-3. (선택) 시드 비교 — MRL만을 시드만 바꿔 다시 학습

- 보여 줄 것: **MRL만은 학습할 때마다 AI-Hub 결과가 들쭉날쭉** (3-7: 같은 코드로 다시 학습하니 감음 재현율 91.7% → 66.5%), 미세조정은 안정적
- 방법: 2-1 에서 **`SEED = 1` 만** 바꿈 (다른 설정 절대 안 바꿈), 2-1b `USE_WANDB = True` → 2-2 ~ 2-6 실행. **2-8 은 건너뜀** (`eye_cnn.pt` 를 덮어쓰지 않게 assert 있음). 그다음 `SEED = 2` 로 한 번 더
- 체크포인트는 `last_s1.pt` · `best_state_s1.pt` 로 따로 저장 → 시드 0 학습 파일은 그대로
- AI-Hub test 결과까지 보려면 3-1 ~ 3-3 실행 뒤 아래 셀 (이 시드의 MRL만 모델을 AI-Hub test 로 재서 같은 run summary 에)
- 증강 난수는 torch 난수(`SEED` 고정)에서 뽑음 → 같은 시드면 같은 증강, 에포크마다는 다른 증강. 남는 차이는 GPU 연산(아주 작음)뿐이라 '시드만 바꾼 비교'. ⚠️ 중간에 끊겨 `last.pt` 에서 이어 하면 난수 흐름이 달라짐 → 시드 비교는 끊기지 않게 한 번에
- W&B 화면: Workspace → `val_loss` · `val_acc` 패널에 `b_mrl_lr3e-4_s0`(5-1 `_1027`) · `s1` · `s2` 가 겹쳐 보임

**결과 (10/29)**

| MRL만 학습 | MRL test | AI-Hub 감음 재현율 | AI-Hub 뜸 재현율 | 감음 정밀도 | 균형 정확도 |
|---|---|---|---|---|---|
| 시드 0 (`eye_cnn.pt`) | 95.9% | **91.7%** | 73.2% | 40.0% | 82.5% |
| 시드 1 | 96.9% | 80.3% | 87.5% | 55.5% | 83.9% |
| 시드 2 | **97.1%** | **79.3%** | 89.1% | 58.6% | 84.2% |
| **미세조정 (참고)** | 96.2% | 91.7% | 95.9% | 81.2% | **93.8%** |

- MRL test 는 세 번 모두 96~97% 로 거의 같음 → 공개 데이터만 보면 전부 좋은 모델
- 한국인 운전자에서는 감음 재현율 79~92% · 뜸 재현율 73~89% → **졸음 놓침 ↔ 괜한 경고, 틀리는 방향이 매번 다름**
- MRL test 최고(시드 2) = 감은 눈을 가장 많이 놓침 → **공개 데이터 test 점수로 모델을 고르면 안 됨**
- 시드를 골라도 균형 정확도 82~84% → 미세조정(93.8%)에 못 미침 → 원인은 운이 아니라 **데이터 차이**
- ⚠️ 시드 0 은 증강 난수 고정 전 코드로 학습 → '학습마다 흔들리는 정도'로 해석

In [ ]:
assert SEED != 0, '시드 비교용 — 2-1 에서 SEED 를 1 · 2 로'
m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(torch.load(BEST, map_location=device, weights_only=True))
res = {f'AIHub_{k}': v for k, v in metrics(predict_arr(m, APREP['test']), AY[AIDX['test']]).items()}   # metrics · predict_arr 는 3-3
print(res)
wb_summary('mrl', LR, res)

## 6. AI-Hub 548 컬러(RGB) 사진 시험 — 지금 모델이 컬러 카메라에서도 맞나

- **왜 (11/02 팀장 결정):** 지금 모델(`eye_cnn_aihub.pt`)은 MRL · AI-Hub 173 둘 다 **적외선**으로만 학습 → 폰 같은 **컬러 카메라**에서 맞는지 따로 확인 필요. 특히 반쯤 감은 눈
- **데이터:** AI-Hub 「안구 움직임 영상 데이터」 (dataSetSn=548) — 한국인 1,510명, 웹캠 **RGB 1280×720 얼굴 전체** (거리 30 · 50cm, 정면 F · 왼쪽 L · 오른쪽 R 보기, 안경 · 마스크 섞임)
  - 라벨: CVAT XML, 눈마다 16점 다각형 `left_eyelid` · `right_eyelid` + `status` = **open / half / closed**
  - ⚠️ **시선 추적 데이터라 감은 눈이 아주 적음** — Validation G1 RGB 28,080장 중 반쯤 감음 478 · 감음 505 / Training 은 약 3%
  - 일부 '감음' 라벨은 눈을 위로 굴리거나 깜빡이는 중간 사진 (눈으로 확인)
- **순서 (팀장):** ① 지금 모델로 **시험만** → ② 결과가 나쁠 때만 (컬러 90% 미만 등) 추가 학습, 최대 2일 → ③ 팀 사진(KPI ①)에서 지금보다 안 나빠질 때만 교체 (서버 설정은 팀장)
- **판정은 서버와 똑같이:** `face_service._real` → `_normalize(reason="microsleep")` 그대로 부름 (CNN 0.4 · EAR 0.23 · 취소는 둘 다 '확실히 뜸'일 때만)
- **사진 고르기:** 감음 · 반쯤 감음 **전부** + 사람 · 거리마다 뜸 **25장** (시드 0). 사진 라벨 = 두 눈 다 closed → 감음, 한쪽이라도 half · closed → 반쯤 감음
- **받기 · 시험은 PC 에서** (`notebooks/aihub548.py`) — AI-Hub 는 해외 IP 를 막아 Colab 에서 못 받음 (3번과 같음)
  1. `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval` → `D:\aihub548\result_g1.csv` (사진마다 한 줄, **저장소 밖에만**)
  2. `set PYTHONUTF8=1` 후 `... aihub548.py wandb` → W&B `eye-cnn` 의 run **`b_eval548_g1`** (숫자 · 묶음 표만)
  3. 추가 학습용: `... aihub548.py download --keys 55882` (TS_G1_1, 88GB, `D:\aihub548\`)
- 🔴 **재배포 금지:** AI-Hub 원본 · 꺼낸 사진 · 사진마다 한 줄 결과는 저장소 밖 · 본인 드라이브에만. GitHub · 단톡 X

### 6-1. PC 결과 불러오기 — 라벨별

- PC 에서 만든 `D:\aihub548\result_g1.csv` (사진마다 한 줄: 파일 이름 · 사람 번호 · 거리 · 보는 방향 · 라벨 · 안경 · 감음 확률 · EAR · 서버 판정) 를 드라이브 `sleep_detector/aihub548/` 에 올리고 실행
  - 사진은 없고 숫자만 있음. 그래도 AI-Hub 결과라 **본인 드라이브에만** (GitHub · 단톡 X)
- 이 구간만 따로 돌려도 됨 (1 ~ 5번 불필요, GPU 필요 없음)
- 서버 판정을 다시 계산하지 않음 — PC 에서 서버 `face_service` 가 낸 값을 그대로 보여 줌

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:                                   # Colab 이면 드라이브
    from google.colab import drive
    drive.mount('/content/drive')
    CSV548 = '/content/drive/MyDrive/sleep_detector/aihub548/result_g1.csv'   # ← PC 의 D:\aihub548\result_g1.csv 를 올린 곳
except ImportError:                    # PC 면 그대로
    CSV548 = r'D:\aihub548\result_g1.csv'

# 그림 한글 글꼴 (1-1 을 안 돌렸을 때만 설치)
import subprocess, matplotlib as mpl, matplotlib.font_manager as fm
if mpl.rcParams['font.family'] != ['NanumGothic'] and os.path.isdir('/usr/share'):
    NANUM = '/usr/share/fonts/truetype/nanum'
    if not fm.findSystemFonts([NANUM]):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    for f in fm.findSystemFonts([NANUM]):
        fm.fontManager.addfont(f)
    if fm.findSystemFonts([NANUM]):
        mpl.rcParams['font.family'] = 'NanumGothic'
        mpl.rcParams['axes.unicode_minus'] = False

r548 = pd.read_csv(CSV548)
r548['face'] = r548['face'].astype(str) == 'True'
for c in ('closed', 'cancel'):
    r548[c] = r548[c].astype(str) == 'True'
LAB = {'open': '뜸', 'half': '반쯤 감음', 'closed': '감음'}
r548['라벨'] = r548['label'].map(LAB)
F548 = r548[r548['face'] & r548['prob'].notna()]   # 얼굴 찾고 CNN 값이 있는 사진


def summ(d, keys):
    g = d.groupby(keys, sort=False)
    out = pd.DataFrame({'사진': g.size()})
    f = d[d['face'] & d['prob'].notna()].groupby(keys, sort=False)
    out['얼굴 찾음'] = (f.size() / out['사진']).round(3)
    out['감음으로 판정'] = f['closed'].mean().round(3)
    out['서버가 경고 취소'] = f['cancel'].mean().round(3)
    return out


print(f"사람 {r548['person'].nunique()}명 · 사진 {len(r548)}장 (PC 판정: CNN 0.4 · EAR 0.23, 취소 = CNN · EAR 둘 다 확실히 뜸)")
display(summ(r548, ['라벨']).reindex(LAB.values()))
c = F548[F548['label'] == 'closed']
print(f"감은 눈: CNN {c['closed'].mean():.1%} · EAR(< 0.23) {(c['ear'] < 0.23).mean():.1%} · 둘 중 하나 {((c['prob'] >= 0.4) | (c['ear'] < 0.23)).mean():.1%}")
o = F548[F548['label'] == 'open']['closed'].mean()
print(f"CNN 균형 정확도 = (뜬 눈 맞힘 {1 - o:.1%} + 감은 눈 맞힘 {c['closed'].mean():.1%}) / 2 = {(1 - o + c['closed'].mean()) / 2:.1%}")

### 6-2. 나눠 보기 · 기준값별

- 거리(30 / 50cm) · 안경 · 보는 방향(정면 F · 왼쪽 L · 오른쪽 R)별 — 특정 조건에서만 틀리는지
- CNN 기준값을 바꾸면 감은 눈 · 뜬 눈 결과가 어떻게 바뀌는지 (서버 기준 후보)

In [ ]:
VIEW = {'F': '정면', 'L': '왼쪽', 'R': '오른쪽'}
r548['보는 방향'] = r548['view'].map(VIEW)
r548['거리'] = r548['dist'].astype(str) + 'cm'
r548['안경'] = r548['glasses'].map({'yes': '씀', 'no': '안 씀'})
for col in ('거리', '안경', '보는 방향'):
    display(summ(r548[r548['label'] != 'half'], [col, '라벨']))

TH = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
rows = []
for th in TH:
    rate = {k: (F548.loc[F548['label'] == k, 'prob'] >= th).mean() for k in LAB}
    rows.append({'기준': th, '뜸 → 감음 (괜한 경고)': rate['open'], '반쯤 → 감음': rate['half'],
                 '감음 → 감음': rate['closed'], '균형 정확도': (1 - rate['open'] + rate['closed']) / 2})
sweep = pd.DataFrame(rows).set_index('기준')
display(sweep.round(3))

fig, ax = plt.subplots(figsize=(9, 3.6))
for col, st in (('감음 → 감음', 'o-'), ('반쯤 → 감음', 's--'), ('뜸 → 감음 (괜한 경고)', '^:'), ('균형 정확도', 'd-')):
    ax.plot(sweep.index, sweep[col] * 100, st, label=col)
ax.axvline(0.4, color='gray', lw=0.8); ax.text(0.405, 3, '지금 서버 0.4', fontsize=8, color='gray')
ax.axhline(90, color='r', lw=0.6, ls=':'); ax.text(0.42, 91.5, '팀장 기준 90%', fontsize=8, color='r')
ax.set_xlabel('CNN 감음 기준값'); ax.set_ylabel('%'); ax.set_ylim(0, 102)
ax.set_title('AI-Hub 548 컬러 사진 — 기준값별 (Validation G1, 65명)'); ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.show()

### 6-3. 결과 정리 — Validation G1 (65명, 4,233장, 11/02 PC)

**라벨별 (얼굴 찾은 사진 기준, 지금 서버 설정)**

| 라벨 | 사진 | 얼굴 찾음 | 감음으로 판정 | 서버가 경고 취소 (확실히 뜸) |
|---|---|---|---|---|
| 뜸 | 3,250 | 97.4% | **0.1%** | 96.2% |
| 반쯤 감음 | 478 | 97.5% | 9.0% | 66.3% |
| **감음** | 505 | 93.5% | **59.5%** | 10.6% |

- **뜬 눈은 거의 완벽** — 컬러 웹캠 · 마스크 · 좌우 보기에서도 괜한 경고 0.1%
- **감은 눈은 CNN 이 59.5% 만 '감음'** → CNN 균형 정확도 **79.7%** (팀장 기준 90% 미만 → **추가 학습 대상**)
  - EAR(< 0.23) 은 같은 감은 눈의 79.7%, CNN · EAR 중 하나라도 감음이면 82.0%
  - 거리(30 / 50cm) · 안경 · 보는 방향별 차이 작음 → 특정 조건이 아니라 **컬러 사진 전반에서 감은 눈을 덜 확신**
- **실제 앱에서는:** 미세수면 의심을 서버가 취소하는 건 진짜 감은 눈의 10.6% (둘 다 '확실히 뜸'일 때만 취소하는 규칙 덕분). 약한 곳은 사진 1장 테스트 · 반쯤 감은 눈 (PERCLOS 가 보완)

**CNN 기준값별 (같은 사진)**

| 기준 | 뜸 → 감음 (괜한 경고) | 반쯤 → 감음 | 감음 → 감음 | 균형 정확도 |
|---|---|---|---|---|
| **0.4 (지금)** | 0.1% | 9.0% | 59.5% | 79.7% |
| 0.3 | 0.2% | 11.2% | 64.0% | 81.9% |
| 0.2 | 0.3% | 15.7% | 72.2% | 86.0% |
| 0.1 | 0.8% | 24.5% | 82.6% | 90.9% |
| 0.05 | 1.6% | 35.2% | 91.5% | 95.0% |

- 순서는 맞게 매기는데 **확신이 낮음** (기준만 낮춰도 90% 넘음). 다만 기준을 바꾸면 적외선(AI-Hub 173) · 팀 사진(KPI ①) 결과가 달라지므로 바로 안 바꿈
- ⚠️ **AI-Hub 548 은 사람이 아니라 사진 단위로 나뉘어 있음** — Validation G1 65명(526~590번)이 Training G1 646명 안에 **전원** 들어 있음 (11/02 라벨 파일로 확인)
  → 안 빼면 같은 사람으로 학습 · 시험 = 점수 부풀림

**다음: 추가 학습(미세조정) 계획 — 시험지를 섞지 않게**

| 데이터 | 쓰는 곳 | 비고 |
|---|---|---|
| 548 Training G1 (TS_G1_1) 중 **Validation 에 없는 사람** | 학습 (train 85%) | `aihub548.py crop` 이 Validation(VL) 사람 151명을 자동으로 뺌 (사람 = 그룹 + 번호) |
| 위 사람 중 사람 단위로 뗀 15% | **val — 기준값(`CNN_CLOSED_PROB`) 고르기 · 조기 종료** | 팀 사진으로 기준값을 고르지 않음 |
| 548 Validation G1 65명 | **시험 — 교체 판단** (6-1 과 같은 방식) | 학습 · val 에 한 번도 안 쓴 사람 |
| AI-Hub 173 test (적외선) | 시험 — 적외선에서 안 나빠졌나 | 3-7 과 비교 |
| 팀 사진 (KPI ①) | **마지막 확인 한 번만** | 반복해서 보면 시험지를 외운 셈이 됨 |

- 눈 하나 = 한 줄, 라벨 = 그 눈 다각형의 status (감음 1 / 뜸 0 / 반쯤 -1 — **반쯤 감음은 학습에서 뺌**, 11/03 → 7번)
- 자르기는 서버와 같은 MediaPipe 눈 점 → `crop_eye` (3번과 같음), 기간 최대 2일 (팀장)
- W&B: `eye-cnn` / `b_eval548_g1` (group `eval`) — 라벨 × 거리 · 안경 · 보기 표, 기준값 표

## 7. AI-Hub 548 컬러 추가 학습(미세조정) → `eye_cnn_548.pt`

- **왜:** 6번 시험에서 컬러 사진의 감은 눈을 CNN 이 59.5% 만 '감음' (Validation G1 균형 정확도 79.7%, Validation 전부 151명 83.1%) → 팀장 기준 90% 미만 → 추가 학습 (최대 2일)
- **학습 방식은 3번과 같음:** 라벨 두 가지 (**감음 1 / 뜸 0**), 출력 1개 = 감음 확률 → 서버 `face_service` · 웹캠 도구 · KPI 노트북 코드는 그대로, `config.CNN_MODEL_PATH` 파일만 바꾸면 됨
  - **반쯤 감음(`y = -1`)은 학습 · val 에서 뺌** (11/03) — MRL · AI-Hub 173 도 반쯤 칸 없이 학습. 세 번째 칸을 만들면 출력이 바뀌어 서버 기준값 · 취소 규칙 · KPI 채점이 다 바뀜
  - 반쯤 감은 눈은 **시험에서만** '반쯤 → 감음' 비율로 봄 (7-4 · PC 시험)
- **시작 가중치:** 지금 채택 모델 `eye_cnn_aihub.pt` (MRL → 173). 학습률 `LR / 10`, 최대 5 에포크, val 손실 2번 연속 안 좋아지면 멈춤 (3-4 와 같음)
- **분할:** PC `aihub548.py crop` 이 만든 `split` 칸 그대로 — Validation 151명을 뺀 324명을 사람 단위로 train 275명 / val 49명 (시드 0)
  - val = 조기 종료 · 기준값 후보 보기. 팀 사진으로 기준값을 고르지 않음
- **교체 판단 (시험지):** ① **Validation 151명** — PC `aihub548.py eval --set val --model eye_cnn_548.pt` (서버 판정 그대로, 지금 83.1% 와 비교) ② **AI-Hub 173 test** (적외선, 7-4 — 안 나빠졌나) ③ MRL test ④ 팀 사진 (KPI ①) **마지막 한 번만**
- **실행 순서:** 1-1 · 1-2 · 1-3 → 2-1 ~ 2-4 → 3-1 ~ 3-3 (**3-4 학습은 안 돌림** — 173 test 와 `predict_arr` · `metrics` 만 씀) → **7-1 부터**
- **PC 준비:** `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py crop` → `D:\aihub548\eye_npz\` 안 파일을 드라이브 `sleep_detector/aihub548_eye/` 에 (본인 드라이브만)
- 🔴 **재배포 금지:** AI-Hub 원본 · 자른 눈 사진(`.npz`)은 본인 드라이브에만. 숫자 · 학습된 모델은 공유 가능
- 🔴 GitHub 에 올리기 전 **모든 출력 지우기** (7-1 눈 사진 그림)

### 7-1. 자른 눈 사진 읽기 (PC 의 `aihub548.py crop` 결과) — 반쯤 감음 빼기

- 드라이브 `sleep_detector/aihub548_eye/` 의 `crops_*.npz` + `summary.json`
- 눈 하나 = 한 줄: 96×96 흑백 눈 사진 + `y` (감음 1 / 뜸 0 / 반쯤 -1) · `status` · 사람(`driver`, 예: `G1_526`) · `split` · 거리(30 / 50cm) · 보는 방향(F · L · R) · 안경 · 성별 · 나이
- PC 에서 이미 버린 것: 얼굴 못 찾음 · MediaPipe 눈 중심이 라벨 다각형에서 먼 눈
- 반쯤 감음은 학습에서 빼고, **val 사람의 반쯤 감은 눈만 `HALF548` 로 따로** 둠 (7-4 시험용)

In [ ]:
CACHE548 = '/content/drive/MyDrive/sleep_detector/aihub548_eye'   # 본인 드라이브에만
S548 = json.load(open(f'{CACHE548}/summary.json', encoding='utf-8'))
missing = [n for n in S548['chunks'] if not os.path.exists(f'{CACHE548}/{n}')]
assert not missing, f'드라이브에 없는 파일 {len(missing)}개: {missing[:3]}'

parts = [np.load(f'{CACHE548}/{n}') for n in S548['chunks']]
ys = [z['y'] for z in parts]
C548 = np.concatenate([z['X'][y >= 0] for z, y in zip(parts, ys)])   # 반쯤 감음(-1) 빼기 — 라벨 두 가지 (3번과 같음). 묶음마다 걸러 큰 복사본 없이
HALF548 = np.stack([prep_eye(e, SIZE) for z, y in zip(parts, ys) for e in z['X'][(y < 0) & (z['split'] == 'val')]])
m_all = pd.DataFrame({k: np.concatenate([z[k] for z in parts]) for k in ('fname', 'side', 'y', 'status', 'driver', 'split', 'dist', 'view', 'glasses', 'gender', 'age')})
m548 = m_all[m_all['y'].to_numpy() >= 0].reset_index(drop=True)
assert len(C548) == len(m548)
Y548 = m548['y'].to_numpy(np.float32)
I548 = {k: np.flatnonzero(m548['split'].to_numpy() == k) for k in ('train', 'val')}
P548 = {'val': np.stack([prep_eye(C548[j], SIZE) for j in I548['val']])}
assert not set(m548.driver[I548['train']]) & set(m548.driver[I548['val']]), 'train · val 에 같은 사람'

c = S548['counts']
print(f"PC 에서 만든 날: {S548['made']} / 사진 {c['files']}장 → 얼굴 없음 {c.get('noface', 0)} ({c.get('noface', 0) / c['files']:.1%})"
      f" · 라벨과 안 맞는 눈 {c.get('no_match', 0)} → 반쯤 감음 {c.get('eyes_half', 0)} 뺌 → 쓰는 눈 {len(m548)} (val 반쯤 감음 {len(HALF548)}개는 시험용)")
print(f'메모리 {C548.nbytes / 1e9:.2f}GB / Validation 이라 뺀 사람 {S548["dropped_val_people"]}명')
display(m548.groupby('split').agg(사람=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row, yv in enumerate((0, 1)):
    for ax, j in zip(axes[row], np.random.default_rng(0).choice(np.flatnonzero(Y548 == yv), 8, replace=False)):
        ax.imshow(C548[j], cmap='gray'); ax.set_title(f"{'감음' if yv else '뜸'} {m548.dist[j]}cm {m548.view[j]}", fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

### 7-2. 학습 함수 — 미세조정 (`eye_cnn_aihub.pt` → 548)

- train: `augment`(2-4 와 같음) → `prep_eye`, 감음 · 뜸을 **반반 뽑기** (`WeightedRandomSampler`, 3-3 과 같음 — 548 은 감음이 약 9%)
- 끊기면 다시 실행 → 드라이브의 `aihub548_ft_last.pt` 에서 이어서

In [ ]:
class Ds548(torch.utils.data.Dataset):
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = prep_eye(augment(C548[j], np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))), SIZE)
        return torch.from_numpy(x)[None], torch.tensor(Y548[j])


y548 = Y548[I548['train']]
w548 = np.where(y548 == 1, 0.5 / y548.mean(), 0.5 / (1 - y548.mean()))
loader548 = torch.utils.data.DataLoader(Ds548(I548['train']), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                        sampler=torch.utils.data.WeightedRandomSampler(w548, len(w548), replacement=True))


def bce(p, y):   # 2-5 와 같음 (2-5 를 안 돌렸을 때도 쓰게)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def fit548(init_state, lr, epochs, patience):
    m = EyeCNN(pretrained=False).to(device)
    m.net.load_state_dict(init_state)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    last, best_p = f'{TRAIN_DIR}/aihub548_ft{RUN_SFX}_last.pt', f'{TRAIN_DIR}/aihub548_ft{RUN_SFX}_best.pt'
    start, h, best_l, bad_n = 0, [], float('inf'), 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, h, best_l, bad_n = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
        print(f'548: {start + 1}번째 에포크부터 이어서')
    run = wb_start('ft548', lr, epochs, patience, split='사람 단위 (Validation 151명 뺌)', data='MRL→AI-Hub 173→AI-Hub 548 (반쯤 감음 뺌)',
                   init='eye_cnn_aihub.pt')
    yv = Y548[I548['val']]
    for ep in range(start, epochs):
        if bad_n >= patience:
            print('조기 종료'); break
        t0 = time.time(); m.train(); tl = n = 0
        for x, y in loader548:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        p = predict_arr(m, P548['val']); vl = bce(p, yv); pred = p >= 0.5
        row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(float((pred == yv).mean()), 4),
                   val_closed_recall=round(float(pred[yv == 1].mean()), 4), val_open_recall=round(float((~pred[yv == 0]).mean()), 4),
                   sec=round(time.time() - t0))
        h.append(row); print('548', row)
        if run:
            wandb.log(row, step=row['epoch'])
        if vl < best_l:
            best_l, bad_n = vl, 0; torch.save(m.net.state_dict(), best_p)
        else:
            bad_n += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=h, best=best_l, bad=bad_n), last)
    if run:
        wandb.summary.update(dict(stopped_epoch=len(h), best_val_loss=round(best_l, 4))); wandb.finish()
    return torch.load(best_p, map_location='cpu', weights_only=True), h


STATE['지금 (173 미세조정)'] = torch.load(f'{TRAIN_DIR}/eye_cnn_aihub.pt', map_location='cpu', weights_only=True)['state_dict']
print('train 눈', len(I548['train']), '/ 감음', f'{y548.mean():.1%}', '→ 반반 뽑기')

### 7-3. 학습 실행

In [ ]:
STATE['548 추가'], HIST548 = fit548(STATE['지금 (173 미세조정)'], LR / 10, 5, 2)

### 7-4. 비교 — 548 val · 반쯤 감은 눈 · AI-Hub 173 test · MRL test

- 기준 **0.4** (지금 서버 `CNN_CLOSED_PROB`). 548 val 은 감음이 적어 **균형 정확도**로 봄
- 548 val 은 기준값 후보도 봄 (교체할 때 기준을 다시 고를지). 최종 판단은 PC Validation 151명 시험 (7-5 아래)
- 반쯤 감은 눈 (val 사람, 학습에 안 씀): '감음' 으로 보는 비율 — 서버는 감음 확률이 0.3 미만이어야 취소하므로 0.3 아래 비율(취소될 수 있는 비율)도 같이
- AI-Hub 173 test 는 적외선 — 3-7 의 0.4 균형 정확도 94.1% 에서 크게 안 떨어져야 함

In [ ]:
TH = 0.4                                                                # 서버 CNN_CLOSED_PROB
rows548, P5 = [], {}
for k in ('지금 (173 미세조정)', '548 추가'):
    m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(STATE[k])
    P5[k] = predict_arr(m, P548['val'])
    ph = predict_arr(m, HALF548)
    rows548.append({'모델': k, **{f'548val_{a}': b for a, b in metrics(P5[k], Y548[I548['val']], TH).items()},
                    '548val_반쯤→감음': round(float((ph >= TH).mean()), 4), '548val_반쯤_0.3미만': round(float((ph < TH - 0.1).mean()), 4),
                    **{f'AIHub_{a}': b for a, b in metrics(predict_arr(m, APREP['test']), AY[AIDX['test']], TH).items()},
                    **{f'MRL_{a}': b for a, b in metrics(predict_arr(m, PREP['test']), Y[IDX['test']], TH).items()}})
cmp548 = pd.DataFrame(rows548).set_index('모델')
if SEED == 0:
    wb_summary('ft548', LR / 10, cmp548.loc['548 추가'].to_dict())
display(cmp548[['548val_balanced_acc', '548val_closed_recall', '548val_open_recall', '548val_반쯤→감음', '548val_반쯤_0.3미만',
                'AIHub_balanced_acc', 'AIHub_closed_recall', 'AIHub_open_recall', 'MRL_acc']])

vm = m548.iloc[I548['val']]
for col in ('dist', 'glasses', 'view'):
    display(pd.DataFrame({k: {g: metrics(P5[k][(vm[col] == g).to_numpy()], Y548[I548['val']][(vm[col] == g).to_numpy()], TH)['balanced_acc']
                              for g in sorted(vm[col].unique())} for k in P5}).rename_axis(col))

print('기준값별 (548 추가, 548 val)')
display(pd.DataFrame([{'기준': th, **metrics(P5['548 추가'], Y548[I548['val']], th)} for th in (0.2, 0.3, 0.4, 0.5, 0.6)]).set_index('기준'))

### 7-5. 저장 — `eye_cnn_548.pt` · 비교 숫자

- 형식은 `eye_cnn_aihub.pt` 와 같음 (출력 1개 = 감음 확률) → 서버 코드 그대로
- 비교 숫자 `aihub548_compare.json` 은 공유 가능 (사진 없음)

**저장 뒤 PC 에서 Validation 151명 시험 (교체 판단)**
1. 드라이브 `sleep_detector/eye_cnn/eye_cnn_548.pt` → PC `sleep_detector_api\ml_models\` (GitHub X)
2. `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval --set val --model eye_cnn_548.pt` → `D:\aihub548\result_val_eye_cnn_548.csv` (서버 판정 그대로, 서버 `config.py` 는 안 바꿈)
3. 지금 모델 `result_val.csv` (균형 83.1%, 뜬 눈 오판 0.04~0.11%) 와 비교 → 좋아지고 173 test · 팀 사진(KPI ①)이 안 나빠질 때만 교체 제안 (서버 설정은 팀장)

In [ ]:
torch.save(dict(
    state_dict=STATE['548 추가'], arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
    output='logit 1개, sigmoid = 감음 확률',
    prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
    crop='MediaPipe 눈 점 → eye_preprocess.crop_eye (서버와 같음)',
    data='MRL → AI-Hub 173 → AI-Hub 548 안구 움직임 영상 (컬러, Validation 151명 뺌, 사람 단위 분할, 반쯤 감음 뺌)',
    test={'AIHub548_val': {a[7:]: float(v) for a, v in cmp548.loc['548 추가'].items() if a.startswith('548val_') and not isinstance(v, str)},
          'AIHub': {a[6:]: float(v) for a, v in cmp548.loc['548 추가'].items() if a.startswith('AIHub_')},
          'MRL': {a[4:]: float(v) for a, v in cmp548.loc['548 추가'].items() if a.startswith('MRL_')}},
    threshold=TH, made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),
), f'{TRAIN_DIR}/eye_cnn_548.pt')
torch.load(f'{TRAIN_DIR}/eye_cnn_548.pt', map_location='cpu', weights_only=True)  # 서버와 같은 방식으로 다시 읽기
print('저장: eye_cnn_548.pt', f"{os.path.getsize(f'{TRAIN_DIR}/eye_cnn_548.pt') / 1e6:.1f}MB", '· weights_only=True 로 읽힘')

json.dump({'made': str(datetime.datetime.now())[:16], 'threshold': TH, 'compare': cmp548.reset_index().to_dict('records'), 'hist': HIST548,
           'npz_made': S548['made'], 'train_people': int(m548.driver[I548['train']].nunique()), 'val_people': int(vm.driver.nunique())},
          open(f'{TRAIN_DIR}/aihub548_compare.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: aihub548_compare.json')

### 7-6. 결과 (11/03 Colab T4 · PC) — **11/03 교체 결정: 서버 모델 `eye_cnn_548.pt`**

| 모델 | 548 val 균형 (0.4) | 548 val 반쯤 → 감음 | Validation 151명 균형 (PC, 서버 판정) | AI-Hub 173 test 균형 (0.4) | MRL test | 팀 사진 (KPI ①) |
|---|---|---|---|---|---|---|
| 지금 (`eye_cnn_aihub.pt`) | 76.1% (감음 52.3% · 뜸 99.8%) | 4.9% | 83.1% (감음 66.3% · 뜬 눈 오판 0.1%) | **94.1%** | **96.3%** | 감음 159/159 · 뜬 눈 오판 0/160 · 반쯤 → 감음 50/109 |
| **548 추가 (`eye_cnn_548.pt`)** | **97.9%** (감음 97.0% · 뜸 98.8%) | 46.4% | **99.4%** (감음 99.7% · 뜬 눈 오판 0.8%) | 83.0% (뜸 67.9%) ⚠️ | 80.8% ⚠️ | 감음 159/159 · 뜬 눈 오판 0/160 · 반쯤 → 감음 **78/109** |

- **데이터 누수 확인 (PC):** train · val · Validation 151명 사이 같은 사람 0명 · 같은 사진 0장, val 감은 눈 493개는 49명 중 43명에게서 (상위 5명 33%)
- 컬러 사진(548 · 팀 사진)에서는 좋아지거나 같음. 548 숫자가 높은 건 학습과 같은 촬영 환경이라서 (사람만 다름) — 지금 모델도 기준 0.05 면 95% 였음 (6-3, 순서는 이미 맞게 매김)
- **반쯤 감은 눈:** 학습에 안 넣었는데도 감음으로 보는 비율이 늘어남 (Validation 11.4% → 50.5%, 팀 사진 50 → 78/109) → 서버가 반쯤 감은 눈을 취소하는 비율 61.5% → 43.6%
- ⚠️ **적외선을 잊음:** AI-Hub 173 test 의 뜬 눈을 3개 중 1개 감음으로 봄 · MRL 96.3% → 80.8% (컬러만으로 이어서 학습한 영향). 앱은 폰 컬러 카메라만 씀 → **카메라에 맞는 모델을 고름** (11/03 팀장): 컬러 = `eye_cnn_548.pt`, 적외선 카메라를 쓰면 `eye_cnn_aihub.pt` (버리지 않고 보관)
- 팀 조건 사진 (11/03 PC, 시연 폰 10/29): 마스크 40장 두 모델 같음 (얼굴 찾음 38/40, 감음 19/19 · 뜬 눈 오판 1/19) / 선글라스 20장은 얼굴 · 눈 점은 잡히지만 눈이 거의 안 보임 → 뜬 눈을 감음으로 0/10 (지금) → 4/10 (548) — **판별기는 만들지 않고 발표 한계로 정리** (11/03 팀장). 서버는 폰(ML Kit)이 의심했을 때만 사진을 받으므로, **A 확인 (11/04): 폰(ML Kit, 기준 0.4)은 같은 선글라스 뜬 눈을 0/10 의심** → 서버로 안 가서 경고 안 남. 대신 선글라스 **감은 눈을 폰이 2/10 만 의심** → 진짜 한계는 놓침 쪽 (5명 × 2장, 한 장씩 판정)

## 8. 상황별 증강 + 모델 여러 개 앙상블 (11/03 팀장 요청)

- **왜:** 실제 앱은 낮 · 밤 · 가로등 · 역광 · 그림자 · 안경 · 흔들림 등 여러 상황에서 찍힘. 지금 증강(2-4)은 크기 · 회전 · 감마 · 흐림 · 잡음뿐 → **우리가 쓰려는 상황에 맞춘 증강**을 넣고, 구조가 다른 CNN 여러 개를 **앙상블**해서 비교
- **상황별 증강 (train 만, 컬러 눈 사진에 씌운 뒤 `prep_eye` 가 흑백으로)**

| 상황 (앱에서 언제) | 증강 |
|---|---|
| 햇빛 · 실내등 · 나트륨 가로등 · 내비 화면 빛 | **색온도** 3000K(따뜻) ~ 7500K(차가움) · **색조** (초록 / 자홍) |
| 밤 운전 · 어두운 차 안 | **어둡게 + 고감도 잡음** (밝기 잡음 + 색 잡음) |
| 역광 · 터널 출구 | **과노출** (밝은 곳이 하얗게 날아감) + 뿌연 대비 |
| 창틀 · 모자 챙 · A필러 | **반쪽 그림자** (방향이 무작위인 그러데이션) |
| 안경 | **안경 반사** (하얀 빛 번짐) |
| 차 진동 · 고개 움직임 | **흔들림** (한 방향 번짐) · 초점 흐림 |
| 폰이 멂 · 낮은 화질 | **저해상도** (줄였다 키움) |
| 앱이 서버로 보낼 때 | **JPEG 압축** |
| 폰 거치 각도 · 왼눈 / 오른눈 | 회전 · 크기 · 위치 · 좌우 반전 · 감마 (2-4 와 같음) |

  - **색온도를 하려면 컬러 눈 사진이 필요** → PC `aihub548.py crop --color` (7번과 같은 사람 · 같은 분할, 흑백 대신 BGR 저장)
  - '전체를 밝게 · 대비 몇 배' 같은 직선 변화는 `prep_eye` 의 사진마다 정규화에서 사라짐 → 어둡게는 잡음과, 밝게는 날아감(잘림)과 같이 넣음
- **모델 (반쯤 감음은 7번처럼 뺌, 출력 1개 = 감음 확률)**

| 이름 | 구조 | 시작 | 학습 |
|---|---|---|---|
| 지금 (`eye_cnn_548.pt`) | ResNet18 | — (7번 결과, 기본 증강) | 비교 기준 |
| `resnet18` | ResNet18 | `eye_cnn_aihub.pt` (7번과 같은 시작) | **상황 증강**, LR/10 · 5 에포크 (7번과 증강만 다름 → 증강 효과) |
| `effb0` | EfficientNet-B0 | ImageNet | 상황 증강, LR · 12 에포크 · patience 3 (3번 'AI-Hub만' 방식) |
| `mnv3` | MobileNetV3-Large | ImageNet | 상황 증강, 위와 같음 (가벼움 — 서버 CPU) |
| **앙상블** | 위 3개 감음 확률 평균 | | |

- **시험지:** ① 548 val (49명) 깨끗한 사진 ② 같은 val 에 상황을 하나씩 **강하게** 씌운 상황별 시험지 (⚠️ 학습 증강과 같은 방식이라 상황 증강 모델에 유리 — 참고용) ③ **PC: Validation 151명 · 팀 사진** (서버 판정 그대로, 진짜 판단) ④ 서버 CPU 속도
- **서버에 넣으려면:** 지금 `face_service._get_cnn` 은 ResNet18 하나만 읽음 → EfficientNet · 앙상블을 채택하면 A 와 상의해 `.pt` 의 `arch` 로 구조를 고르게 바꿔야 함 (PC 시험은 `aihub548.py` 가 대신 처리)
- **실행 순서:** 1-1 → 2-1 · 2-1b → **8-1 부터** (MRL · 173 은 필요 없음. 3-1 ~ 3-2 를 돌려 두면 8-6 에 AI-Hub 173 test 도 나옴)
- **PC 준비:** `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py crop --color` → `D:\aihub548\eye_npz_color\` 안 파일을 드라이브 `sleep_detector/aihub548_eye_color/` 에 (본인 드라이브만)
- 🔴 재배포 금지 (AI-Hub 눈 사진은 본인 드라이브에만) · GitHub 에 올리기 전 **모든 출력 지우기** (8-1 · 8-2 눈 사진 그림)

### 8-1. 컬러 눈 사진 읽기 (PC 의 `aihub548.py crop --color` 결과) — 반쯤 감음 빼기

- 7-1 과 같은 모양, 눈 사진만 96×96×3 (BGR). 사람 · `split` 은 7번과 같음
- 반쯤 감음은 학습에서 빼고, val 사람의 반쯤 감은 눈은 `HALF8` 로 따로 (8-6 시험용)

In [ ]:
import collections
import matplotlib.pyplot as plt   # 2-4 를 안 돌려도 되게
CACHE8 = '/content/drive/MyDrive/sleep_detector/aihub548_eye_color'   # 본인 드라이브에만
S8 = json.load(open(f'{CACHE8}/summary.json', encoding='utf-8'))
assert S8.get('color'), 'summary.json 이 컬러 자르기 결과가 아님 — PC 에서 crop --color'
missing = [n for n in S8['chunks'] if not os.path.exists(f'{CACHE8}/{n}')]
assert not missing, f'드라이브에 없는 파일 {len(missing)}개: {missing[:3]}'

parts = [np.load(f'{CACHE8}/{n}') for n in S8['chunks']]
ys = [z['y'] for z in parts]
C8 = np.concatenate([z['X'][y >= 0] for z, y in zip(parts, ys)])     # 반쯤 감음(-1) 빼기 (7번과 같음). 묶음마다 걸러 큰 복사본 없이 (Colab 메모리)
HALF8 = np.concatenate([z['X'][(y < 0) & (z['split'] == 'val')] for z, y in zip(parts, ys)])
m_all = pd.DataFrame({k: np.concatenate([z[k] for z in parts]) for k in ('fname', 'side', 'y', 'status', 'driver', 'split', 'dist', 'view', 'glasses')})
m8 = m_all[m_all['y'].to_numpy() >= 0].reset_index(drop=True)
assert len(C8) == len(m8)
Y8 = m8['y'].to_numpy(np.float32)
I8 = {k: np.flatnonzero(m8['split'].to_numpy() == k) for k in ('train', 'val')}
assert C8.ndim == 4 and C8.shape[-1] == 3, C8.shape
assert not set(m8.driver[I8['train']]) & set(m8.driver[I8['val']]), 'train · val 에 같은 사람'
print(f"PC 에서 만든 날: {S8['made']} / 눈 {len(m8)} (반쯤 감음 {S8['counts'].get('eyes_half', 0)} 뺌, val 반쯤 {len(HALF8)}개는 시험용) · 메모리 {C8.nbytes / 1e9:.2f}GB")
display(m8.groupby('split').agg(사람=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))

### 8-2. 상황별 증강 함수 + 눈으로 보기

- `scene_aug(눈 BGR, rng)` = 상황 증강을 확률로 몇 개 골라 씌우고 → 기하 증강 (2-4 와 같은 범위). 결과는 uint8 BGR → `prep_eye` 가 흑백 · 64×64 · 정규화
- `SCENES` = 시험지용으로 상황 하나씩 **강하게** (같은 함수, 값 고정)
- 아래 그림: 줄 = 상황, 칸 = 같은 눈 (위 줄 = 원본). 이상하게 보이는 상황이 있으면 값을 줄임

In [ ]:
def color_temp(x, t):        # t: -1 = 따뜻함(약 3000K, 주황) … 0 = 그대로 … +1 = 차가움(약 7500K, 파랑). x 는 float BGR
    g = (1 + 0.45 * t, 1 + 0.2 * t, 1.0) if t < 0 else (1.0, 1 - 0.1 * t, 1 - 0.2 * t)
    return x * np.array(g, np.float32)


def tint(x, s):              # 초록(+) / 자홍(-) 끼 — LED · 화면 빛
    return x * np.array((1.0, 1 + s, 1.0), np.float32)


def night(x, rng, k, sigma):  # 어둡게(k 배) + 고감도 잡음 (밝기 잡음 + 색 잡음)
    x = x * k
    return x + rng.normal(0, sigma, x.shape[:2])[..., None] + rng.normal(0, sigma * 0.5, x.shape)


def overexpose(x, g, haze):  # 과노출: g 배 (255 넘는 곳은 하얗게 날아감) + 뿌옇게
    return np.clip(x * g, 0, 255) * (1 - haze) + 255 * haze


def shadow(x, rng, depth):   # 반쪽 그림자: 무작위 방향으로 depth 만큼 어두워지는 그러데이션
    h, w = x.shape[:2]
    a = rng.uniform(0, 2 * np.pi)
    yy, xx = np.mgrid[0:h, 0:w]
    proj = ((xx - w / 2) * np.cos(a) + (yy - h / 2) * np.sin(a)) / (w / 2)
    return x * (1 - depth * np.clip(proj + rng.uniform(-0.3, 0.3), 0, 1))[..., None]


def glare(x, rng, a):        # 안경 반사: 하얀 타원 빛 번짐 (세기 a)
    h, w = x.shape[:2]
    blob = np.zeros((h, w), np.float32)
    cv2.ellipse(blob, (int(rng.uniform(0.2, 0.8) * w), int(rng.uniform(0.2, 0.7) * h)),
                (int(rng.uniform(0.06, 0.2) * w), int(rng.uniform(0.03, 0.1) * h)), rng.uniform(0, 180), 0, 360, 1.0, -1)
    blob = cv2.GaussianBlur(blob, (0, 0), w * 0.03)
    return x + a * 255 * blob[..., None]


def motion(x, rng, n):       # 흔들림: 길이 n 픽셀, 무작위 방향
    k = np.zeros((n, n), np.float32); k[n // 2] = 1
    k = cv2.warpAffine(k, cv2.getRotationMatrix2D((n / 2 - 0.5, n / 2 - 0.5), rng.uniform(0, 180), 1), (n, n))
    return cv2.filter2D(x, -1, k / max(k.sum(), 1e-6))


def lowres(x, s):            # 저해상도: s 픽셀로 줄였다가 원래 크기로
    h, w = x.shape[:2]
    return cv2.resize(cv2.resize(x, (s, s), interpolation=cv2.INTER_AREA), (w, h), interpolation=cv2.INTER_LINEAR)


def jpeg(x, q):              # JPEG 압축 (품질 q)
    ok, buf = cv2.imencode('.jpg', np.clip(x, 0, 255).astype(np.uint8), [cv2.IMWRITE_JPEG_QUALITY, int(q)])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR).astype(np.float32)


def geometry(x, rng):        # 2-4 augment 와 같은 범위: 좌우 반전 · 회전 ±10° · 크기 ±15% · 위치 ±5% · 감마
    h, w = x.shape[:2]
    if rng.random() < 0.5:
        x = np.ascontiguousarray(x[:, ::-1])
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), rng.uniform(0.85, 1.15))
    M[:, 2] += rng.uniform(-0.05, 0.05, 2) * (w, h)
    x = cv2.warpAffine(x, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return 255.0 * (np.clip(x, 0, 255) / 255.0) ** rng.uniform(0.6, 1.6)


def scene_aug(img, rng):
    # 눈 사진(uint8 BGR) → 상황 증강 몇 개 + 기하 증강 → uint8 BGR. 순서: 빛(색 · 밝기) → 그림자 · 반사 → 카메라(흔들림 · 해상도 · 잡음 · 압축)
    x = img.astype(np.float32)
    if rng.random() < 0.6: x = color_temp(x, rng.uniform(-1, 1))
    if rng.random() < 0.2: x = tint(x, rng.uniform(-0.15, 0.15))
    r = rng.random()
    if r < 0.25:   x = night(x, rng, rng.uniform(0.15, 0.6), rng.uniform(3, 15))
    elif r < 0.4:  x = overexpose(x, rng.uniform(1.3, 2.2), rng.uniform(0, 0.25))
    if rng.random() < 0.3:  x = shadow(x, rng, rng.uniform(0.3, 0.7))
    if rng.random() < 0.15: x = glare(x, rng, rng.uniform(0.4, 1.0))
    r = rng.random()
    if r < 0.15:   x = motion(x, rng, int(rng.choice([3, 5, 7, 9])))
    elif r < 0.3:  x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.6, 1.5))
    if rng.random() < 0.2:  x = lowres(x, int(rng.integers(20, 48)))
    x = geometry(x, rng)
    if rng.random() < 0.3:  x = jpeg(x, rng.integers(30, 85))
    return np.clip(x, 0, 255).astype(np.uint8)


SCENES = {   # 시험지: 상황 하나씩 강하게 (값 고정, 기하 증강 없음)
    '깨끗함': lambda x, r: x,
    '따뜻한 색 3000K': lambda x, r: color_temp(x, -1.0),
    '차가운 색 7500K': lambda x, r: color_temp(x, 1.0),
    '밤 (어둡게 + 잡음)': lambda x, r: night(x, r, 0.25, 10),
    '역광 (과노출)': lambda x, r: overexpose(x, 1.9, 0.15),
    '반쪽 그림자': lambda x, r: shadow(x, r, 0.6),
    '안경 반사': lambda x, r: glare(x, r, 0.8),
    '흔들림': lambda x, r: motion(x, r, 7),
    '저해상도': lambda x, r: lowres(x, 24),
    'JPEG 30': lambda x, r: jpeg(x, 30),
}


def scene_img(img, name, seed):
    return np.clip(SCENES[name](img.astype(np.float32), np.random.default_rng(seed)), 0, 255).astype(np.uint8)


# 눈으로 보기 — 줄 = 상황, 칸 = 같은 눈 6개 (뜸 3 · 감음 3). 사진은 흑백(prep_eye 전)으로 보여 줌 = CNN 이 보는 것
show = np.r_[np.random.default_rng(0).choice(I8['val'][Y8[I8['val']] == 0], 3, replace=False),
             np.random.default_rng(0).choice(I8['val'][Y8[I8['val']] == 1], 3, replace=False)]
fig, axes = plt.subplots(len(SCENES) + 1, 6, figsize=(9, 1.6 * (len(SCENES) + 1)))
for r, name in enumerate(list(SCENES) + ['무작위 (학습 때)']):
    for c, j in enumerate(show):
        im = scene_aug(C8[j], np.random.default_rng(c)) if name == '무작위 (학습 때)' else scene_img(C8[j], name, c)
        axes[r, c].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[r, c].axis('off')
        if c == 0: axes[r, c].set_title(name, fontsize=8, loc='left')
plt.tight_layout(); plt.show()

### 8-3. 시험지 만들기 — 548 val × 상황 (약 1분)

- val 눈마다 상황을 씌우고 `prep_eye` 한 결과를 한 번만 만들어 둠 (사진마다 씨앗 고정 → 매번 같은 시험지)

In [ ]:
t0 = time.time()
P8 = {name: np.stack([prep_eye(scene_img(C8[j], name, int(j)), SIZE).astype(np.float16) for j in I8['val']]) for name in SCENES}   # float16: 메모리 절반
P8_HALF = np.stack([prep_eye(e, SIZE).astype(np.float16) for e in HALF8])
yv8 = Y8[I8['val']]
print(f'시험지 {len(P8)}개 × 눈 {len(yv8)} (감음 {int(yv8.sum())}) · 반쯤 감음 {len(P8_HALF)} — {time.time() - t0:.0f}초 · 메모리 {sum(v.nbytes for v in P8.values()) / 1e9:.2f}GB')

### 8-4. 모델 · 학습 함수

- `make_net(arch)` — PC `aihub548.py make_net` 과 같음 (출력 1개 = 감음 logit, 흑백을 3채널로 복사)
- train: `scene_aug` → `prep_eye`, 감음 · 뜸 반반 뽑기 (7-2 와 같음)
- 끊기면 다시 실행 → 드라이브의 `aug8_<이름>_last.pt` 에서 이어서

In [ ]:
import torchvision


def make_net(arch, pretrained=False):
    if arch == 'resnet18':
        m = torchvision.models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        m.fc = torch.nn.Linear(m.fc.in_features, 1)
    elif arch == 'efficientnet_b0':
        m = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[1] = torch.nn.Linear(m.classifier[1].in_features, 1)
    elif arch == 'mobilenet_v3_large':
        m = torchvision.models.mobilenet_v3_large(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[3] = torch.nn.Linear(m.classifier[3].in_features, 1)
    return m


class Net8(torch.nn.Module):
    # (N, 1, 64, 64) prep_eye 결과 → (N,) 감음 logit. self.net 의 state_dict 를 저장 (서버 · PC 는 make_net 으로 읽음)
    def __init__(self, arch, pretrained=False):
        super().__init__()
        self.net = make_net(arch, pretrained)
    def forward(self, x):
        return self.net(x.expand(-1, 3, -1, -1)).squeeze(1)


class Ds8(torch.utils.data.Dataset):
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = prep_eye(scene_aug(C8[j], np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))), SIZE)
        return torch.from_numpy(x)[None], torch.tensor(Y8[j])


y8 = Y8[I8['train']]
w8 = np.where(y8 == 1, 0.5 / y8.mean(), 0.5 / (1 - y8.mean()))
loader8 = torch.utils.data.DataLoader(Ds8(I8['train']), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                      sampler=torch.utils.data.WeightedRandomSampler(w8, len(w8), replacement=True))
loss_fn = torch.nn.BCEWithLogitsLoss()


@torch.no_grad()
def predict8(m, X, bs=512):
    # prep_eye 결과 (N, 64, 64) → 감음 확률
    m.eval(); out = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[i:i + bs], np.float32))[:, None].to(device)   # 시험지는 float16 으로 보관 (8-3)
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(m(x).float()).cpu())
    return torch.cat(out).numpy()


def metrics8(p, y, th=0.4):
    pr = p >= th
    rc, ro = float(pr[y == 1].mean()), float((~pr[y == 0]).mean())
    return dict(balanced_acc=round((rc + ro) / 2, 4), closed_recall=round(rc, 4), open_recall=round(ro, 4),
                closed_precision=round(float(y[pr].mean()) if pr.any() else float('nan'), 4))


def bce8(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def fit8(name, arch, init_state, lr, epochs, patience, loader=None, data='AI-Hub 548 컬러 (반쯤 감음 뺌)'):
    m = Net8(arch, pretrained=init_state is None).to(device)
    if init_state is not None:
        m.net.load_state_dict(init_state)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    last, best_p = f'{TRAIN_DIR}/aug8_{name}{RUN_SFX}_last.pt', f'{TRAIN_DIR}/aug8_{name}{RUN_SFX}_best.pt'
    start, h, best_l, bad_n = 0, [], float('inf'), 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, h, best_l, bad_n = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
        print(f'{name}: {start + 1}번째 에포크부터 이어서')
    run = wb_start(f'aug8-{name}', lr, epochs, patience, arch=arch, split='사람 단위 (Validation 151명 뺌)', aug='상황 증강 (8-2)',
                   data=data, init='eye_cnn_aihub.pt' if init_state is not None else 'ImageNet')
    for ep in range(start, epochs):
        if bad_n >= patience:
            print('조기 종료'); break
        t0 = time.time(); m.train(); tl = n = 0
        for x, y in (loader or loader8):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        p = predict8(m, P8['깨끗함']); vl = bce8(p, yv8)
        row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), **{f'val_{k}': v for k, v in metrics8(p, yv8).items()},
                   sec=round(time.time() - t0))
        h.append(row); print(name, row)
        if run:
            wandb.log(row, step=row['epoch'])
        if vl < best_l:
            best_l, bad_n = vl, 0; torch.save(m.net.state_dict(), best_p)
        else:
            bad_n += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=h, best=best_l, bad=bad_n), last)
    if run:
        wandb.summary.update(dict(stopped_epoch=len(h), best_val_loss=round(best_l, 4))); wandb.finish()
    return torch.load(best_p, map_location='cpu', weights_only=True), h


ARCH8 = {'resnet18': 'resnet18', 'effb0': 'efficientnet_b0', 'mnv3': 'mobilenet_v3_large'}
STATE8, HIST8 = {}, {}
print('train 눈', len(I8['train']), '/ 감음', f'{y8.mean():.1%}', '→ 반반 뽑기')

### 8-5. 학습 실행 (세 모델 — 하나씩 끝나면 드라이브에 저장, 끊기면 이 셀 다시)

- `resnet18`: 7번과 같은 시작 · 같은 설정, 증강만 상황 증강 → 7번 결과(`eye_cnn_548.pt`)와 비교하면 **증강 효과**
- `effb0` · `mnv3`: ImageNet 에서 시작 (3번 'AI-Hub만' 과 같은 설정)

In [ ]:
init_aihub = torch.load(f'{TRAIN_DIR}/eye_cnn_aihub.pt', map_location='cpu', weights_only=True)['state_dict']
STATE8['resnet18'], HIST8['resnet18'] = fit8('resnet18', 'resnet18', init_aihub, LR / 10, 5, 2)
STATE8['effb0'], HIST8['effb0'] = fit8('effb0', 'efficientnet_b0', None, LR, EPOCHS, PATIENCE)
STATE8['mnv3'], HIST8['mnv3'] = fit8('mnv3', 'mobilenet_v3_large', None, LR, EPOCHS, PATIENCE)

### 8-6. 비교 — 상황별 시험지 · 반쯤 감은 눈 · 서버 CPU 속도 (기준 0.4)

- 줄 = 모델 (지금 `eye_cnn_548.pt` · 상황 증강 3개 · 앙상블), 칸 = 상황별 균형 정확도
- ⚠️ 상황별 시험지는 학습 증강과 같은 함수 → 상황 증강 모델에 유리. **깨끗함** 칸과 PC 시험(8-8)이 공정한 비교
- `AIHub173` 칸: 3-1 ~ 3-2 를 돌렸을 때만 (적외선 — 7번에서 떨어졌던 것)
- CPU 속도 = 서버처럼 눈 2개 한 번 (스레드 1개 / 전부)

In [ ]:
models8 = {'지금 (eye_cnn_548)': ('resnet18', torch.load(f'{TRAIN_DIR}/eye_cnn_548.pt', map_location='cpu', weights_only=True)['state_dict'])}
models8.update({k: (ARCH8[k], STATE8[k]) for k in STATE8})
P_SC, P_HALF, P_173, LAT = {}, {}, {}, {}
has173 = 'APREP' in globals() and 'test' in APREP
for k, (arch, st) in models8.items():
    m = Net8(arch).to(device); m.net.load_state_dict(st)
    P_SC[k] = {sc: predict8(m, X) for sc, X in P8.items()}
    P_HALF[k] = predict8(m, P8_HALF)
    if has173:
        P_173[k] = predict8(m, APREP['test'])
    cm = Net8(arch).eval(); cm.net.load_state_dict(st)                # CPU 속도 (서버는 GPU 없음)
    x2 = torch.from_numpy(P8['깨끗함'][:2].astype(np.float32))[:, None]
    for th in (1, os.cpu_count()):
        torch.set_num_threads(th)
        with torch.no_grad():
            for _ in range(3): cm(x2)
            t0 = time.perf_counter()
            for _ in range(20): cm(x2)
        LAT.setdefault(k, {})[f'{th}스레드_ms'] = round((time.perf_counter() - t0) / 20 * 1000, 1)
ens = [k for k in STATE8]                                              # 앙상블 = 상황 증강 3개 평균
if len(ens) > 1:
    E = f'앙상블 ({" + ".join(ens)})'
    P_SC[E] = {sc: np.mean([P_SC[k][sc] for k in ens], 0) for sc in P8}
    P_HALF[E] = np.mean([P_HALF[k] for k in ens], 0)
    if has173:
        P_173[E] = np.mean([P_173[k] for k in ens], 0)
    LAT[E] = {c: round(sum(LAT[k][c] for k in ens), 1) for c in LAT[ens[0]]}

tab = pd.DataFrame({k: {sc: metrics8(P_SC[k][sc], yv8)['balanced_acc'] for sc in P8} for k in P_SC}).T
tab['반쯤→감음'] = [round(float((P_HALF[k] >= 0.4).mean()), 4) for k in tab.index]
if has173:
    tab['AIHub173'] = [metrics8(P_173[k], AY[AIDX['test']])['balanced_acc'] for k in tab.index]
tab['CPU_ms(1스레드)'] = [LAT[k]['1스레드_ms'] for k in tab.index]
display(tab)
print('깨끗함 자세히 (감음 재현율 · 뜸 재현율 · 감음 정밀도)')
display(pd.DataFrame({k: metrics8(P_SC[k]['깨끗함'], yv8) for k in P_SC}).T)

fig, ax = plt.subplots(figsize=(11, 3.8))
sc_names = list(P8)
for i, k in enumerate(P_SC):
    ax.bar(np.arange(len(sc_names)) + i * 0.8 / len(P_SC), tab.loc[k, sc_names] * 100, 0.8 / len(P_SC), label=k)
ax.set_xticks(np.arange(len(sc_names)) + 0.4 - 0.4 / len(P_SC)); ax.set_xticklabels(sc_names, rotation=30, ha='right', fontsize=8)
ax.set_ylim(50, 100); ax.set_ylabel('균형 정확도 (%)'); ax.axhline(90, color='r', lw=0.6, ls=':')
ax.set_title('548 val (49명) — 상황별 시험지 (기준 0.4)'); ax.legend(fontsize=7, loc='lower left')
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/aug8_scenes.png', dpi=90); plt.show()
if SEED == 0:
    for k in STATE8:
        wb_summary(f'aug8-{k}', LR / 10 if k == 'resnet18' else LR, {f'scene_{sc}': tab.loc[k, sc] for sc in sc_names})

### 8-7. 저장 — `eye_cnn8_<이름>.pt` · 비교 숫자

- 형식은 `eye_cnn_548.pt` 와 같고 `arch` 가 들어 있음 (PC `aihub548.py` 가 `arch` 로 구조를 고름)
- 비교 숫자 `aug8_compare.json` 은 공유 가능 (사진 없음)

In [ ]:
for k in STATE8:
    torch.save(dict(
        state_dict=STATE8[k], arch=ARCH8[k], in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
        output='logit 1개, sigmoid = 감음 확률',
        prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
        crop='MediaPipe 눈 점 → eye_preprocess.crop_eye (서버와 같음)',
        aug='상황 증강 scene_aug (eye_cnn.ipynb 8-2: 색온도 · 색조 · 밤 · 역광 · 그림자 · 안경 반사 · 흔들림 · 저해상도 · JPEG + 기하)',
        data='AI-Hub 548 안구 움직임 영상 (컬러, Validation 151명 뺌, 사람 단위 분할, 반쯤 감음 뺌)' + (' ← eye_cnn_aihub.pt 에서 시작' if k == 'resnet18' else ' ← ImageNet'),
        test={sc: float(tab.loc[k, sc]) for sc in sc_names}, threshold=0.4, cpu_latency=LAT[k],
        made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),
    ), f'{TRAIN_DIR}/eye_cnn8_{k}.pt')
    torch.load(f'{TRAIN_DIR}/eye_cnn8_{k}.pt', map_location='cpu', weights_only=True)
    print('저장:', f'eye_cnn8_{k}.pt', f"{os.path.getsize(f'{TRAIN_DIR}/eye_cnn8_{k}.pt') / 1e6:.1f}MB")
json.dump({'made': str(datetime.datetime.now())[:16], 'threshold': 0.4, 'table': tab.reset_index().to_dict('records'), 'hist': HIST8,
           'npz_made': S8['made']}, open(f'{TRAIN_DIR}/aug8_compare.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: aug8_compare.json')

### 8-8. PC 시험 (진짜 판단) · 결과 (학습 뒤 채움)

1. 드라이브 `sleep_detector/eye_cnn/eye_cnn8_*.pt` 3개 → PC `sleep_detector_api\ml_models\` (GitHub X)
2. Validation 151명 (서버 판정 그대로) — 하나씩 + 앙상블
   - `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval --set val --model eye_cnn8_resnet18.pt` (`effb0` · `mnv3` 도 같은 방식)
   - `... eval --set val --model eye_cnn8_resnet18.pt,eye_cnn8_effb0.pt,eye_cnn8_mnv3.pt` → `result_val_eye_cnn8_resnet18_ens3.csv`
3. 팀 사진 (밝은 곳 · 어두운 곳 · 고개 · 마스크 · 선글라스) — `... aihub548.py team --model …` (지금 모델은 `--model` 없이)

**결과 (11/03 Colab T4 · PC)**

고르기용 — 548 val 49명 (8-6 · 9-4) · Validation 151명 · 팀 사진 (PC, 서버 판정 그대로, 기준 0.4)

| 모델 | 548 val 깨끗함 | 548 밤 | 상황 평균 | 173 적외선 | 151명 균형 | 151명 감은 눈 취소 | 151명 뜬 눈 오경보 | 팀 반쯤 → 감음 | CPU ms |
|---|---|---|---|---|---|---|---|---|---|
| 지금 (`eye_cnn_548.pt`) | 97.8% | 92.7% | 96.9% | 83.0% | 99.4% | 0.0% | 3.2% | 78/109 | 21 |
| `resnet18` (상황 증강) | 98.2% | 96.4% | 97.7% | 83.3% | 99.2% | 0.2% | 3.9% | 81/109 | 29 |
| `effb0` | 98.5% | 96.7% | 98.2% | 76.6% | 99.4% | 0.6% | 3.3% | 77/109 | 24 |
| `mnv3` | 98.2% | 92.1% | 97.1% | 77.0% | — | — | — | — | 12 |
| 앙상블 (3개) | 98.5% | 96.5% | 98.1% | 78.8% | 99.4% | 0.2% | 3.3% | 77/109 | 65 |
| `resnet18_ir` (+ 173 30%, 9-4) | 97.5% | — | 97.2% | **93.7%** | 99.3% | 0.2% | 3.5% | 76/109 | 21 |

- 라벨 섞기 (9-1): 진짜 val 49.2% → 새는 곳 없음 / 좌우 반전 평균 (9-3): ±0.3%p → 안 씀
- 한국인 데이터에서 후보 다섯 개가 오차 수준으로 같음 → 도전 모델 = `resnet18_ir` (적외선 회복 · 상황 증강 · 서버 코드 그대로). EfficientNet · 앙상블은 적외선 77~79% · 서버 수정 필요 · 앙상블 CPU 3배라 제외

**마지막 시험 (한 번만) — AI-Hub 71421 「디스플레이 중심 안구 움직임」 Validation 6명 (PC `aihub548.py extra`, 서버 판정 그대로)**
- 판정 규칙 (시험 전에 정함): 감은 눈 취소율 · 뜬 눈 오경보율이 둘 다 지금 모델보다 나쁘지 않을 때만 교체 제안
- 사진: 감은 눈 261 · 뜬 눈 795 (노트북 · 모니터 · 차량 LCD · 태블릿), 라벨 = 홍채 없음 + 눈꺼풀 세로/가로 < 0.15 → 감음 / 홍채 있음 + 0.25 이상 → 뜸

| 모델 | 감은 눈 → 감음 | 감은 눈 취소율 | CNN 뜬 눈 → 감음 | 뜬 눈 오경보율 | CNN 균형 |
|---|---|---|---|---|---|
| **지금 (`eye_cnn_548.pt`)** | 261/261 | **0.0%** | **5.4%** | **41.9%** | **97.3%** |
| `resnet18_ir` | 261/261 | 0.0% | 21.1% | 48.8% | 89.4% |

- **판정: 지금 모델 유지** (11/03) — 뜬 눈 오경보가 도전 모델에서 더 나쁨
- 뜬 눈 오경보 대부분은 EAR 쪽 (화면을 내려다보는 영상이라 뜬 눈도 EAR 이 0.26 미만). CNN 오판 43장 중 33장이 한 사람(152번)
- 상황 증강은 합성 시험지(밤 92.7% → 96.4%)에서는 좋았지만 처음 보는 데이터에서는 이득이 없었음


## 9. 의심 확인 · 추가 개선 (11/03)

- **왜:** 548 미세조정 뒤 점수가 크게 올라서 (Validation 151명 83.1% → 99.4%) **정말 눈을 보고 맞히는지** 확인하고, 남은 개선을 시험
- 이미 PC 에서 확인한 것 (7-6): 같은 사람 · 같은 사진 0, 사람 단위 부트스트랩 95% 구간 **99.2 ~ 99.7%** (지금 모델 79.7 ~ 86.3%), 학습에 없던 그룹 G2 194/194 · U1 233/234
  - PC 틀린 사진 (`aihub548.py wrong`): 60장 중 57장이 '뜸' 라벨을 감음으로 본 것 — 눈으로 보면 상당수가 **실제로 감았거나 가늘게 뜬 눈** (라벨 잡음) · 마스크에서 눈 점이 엇나간 사진 · 아래를 보는 눈
- 이 구간: **9-1** 라벨 섞기 (새는 곳이 있으면 섞어도 높게 나옴) · **9-2** Grad-CAM (어디를 보나) · **9-3** 좌우 반전 평균 · **9-4** 적외선(AI-Hub 173) 섞어 학습 · **9-5** 틀린 사진 모아 보기 · **9-6** 저장
- **실행 순서:** 8번을 8-6 까지 돌린 뒤 (`P8` · `STATE8` · `models8`) → 9-1. **9-4 만** 3-1 ~ 3-2 (AI-Hub 173 `CROPS` · `AIDX`) 가 필요
- 🔴 GitHub 에 올리기 전 **모든 출력 지우기** (9-2 · 9-5 눈 사진)

### 9-1. 라벨 섞기 시험 — 새는 곳이 있나 (약 5분)

- train 라벨만 무작위로 섞어 ResNet18 (ImageNet) 을 3 에포크 학습 → 진짜 라벨의 val 로 채점
- **정상:** 눈과 라벨의 관계를 못 배우므로 균형 정확도 **50% 근처**. 70~80% 넘게 나오면 사진 이름 · 순서 · 사람 같은 다른 길로 답이 새고 있다는 뜻
- 감음 비율은 그대로 (섞기만 함) → 반반 뽑기도 같음

In [ ]:
Y_SHUF = Y8.copy()
Y_SHUF[I8['train']] = np.random.default_rng(0).permutation(Y8[I8['train']])
print('섞은 뒤 train 라벨이 진짜와 같은 비율:', f"{(Y_SHUF[I8['train']] == Y8[I8['train']]).mean():.1%}", '(감음 9% 라 우연히 같은 게 많음)')


class DsLab(torch.utils.data.Dataset):          # Ds8 와 같고 라벨 배열만 따로
    def __init__(self, idx, labels):
        self.idx, self.labels = idx, labels
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = prep_eye(scene_aug(C8[j], np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))), SIZE)
        return torch.from_numpy(x)[None], torch.tensor(self.labels[j])


ys = Y_SHUF[I8['train']]
loader_shuf = torch.utils.data.DataLoader(DsLab(I8['train'], Y_SHUF), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                          sampler=torch.utils.data.WeightedRandomSampler(np.where(ys == 1, 0.5 / ys.mean(), 0.5 / (1 - ys.mean())), len(ys), replacement=True))
STATE_SHUF, HIST_SHUF = fit8('shuffled', 'resnet18', None, LR, 3, 3, loader=loader_shuf, data='AI-Hub 548 컬러 — train 라벨 무작위로 섞음 (새는지 확인)')
m = Net8('resnet18').to(device); m.net.load_state_dict(STATE_SHUF)
p_shuf = predict8(m, P8['깨끗함'])
print('라벨 섞어 학습 → 진짜 val:', metrics8(p_shuf, yv8), '/ 예측 확률 범위', p_shuf.min().round(3), '~', p_shuf.max().round(3))

### 9-2. Grad-CAM — 모델이 눈의 어디를 보나 (3-8 과 같은 방법)

- 빨간 곳 = 판정(감음이면 감음 쪽, 뜸이면 뜸 쪽)에 가장 크게 기여한 곳. **눈꺼풀 · 눈동자** 에 모여야 정상. 눈썹 · 사진 가장자리 · 마스크 · 안경테에 모이면 엉뚱한 단서
- 줄 = 모델, 칸 = 같은 val 눈 8개 (맞힌 감음 2 · 맞힌 뜸 2 · 지금 모델이 틀린 것 4)

In [ ]:
import torch.nn.functional as Fnn

CAM_LAYER = {'resnet18': lambda n: n.layer4, 'efficientnet_b0': lambda n: n.features[-1], 'mobilenet_v3_large': lambda n: n.features[-1]}


def gradcam8(m, layer, X, th=0.4):
    # prep_eye 결과 (N, 64, 64) → (판정한 쪽 근거 지도 (N, 64, 64) 0~1, 감음 확률 (N,))
    acts, grads = [], []
    h1 = layer.register_forward_hook(lambda mod, i, o: acts.append(o))
    h2 = layer.register_full_backward_hook(lambda mod, gi, go: grads.append(go[0]))
    x = torch.from_numpy(np.asarray(X, np.float32))[:, None].to(device).requires_grad_(True)
    m.eval(); logit = m(x)
    sign = torch.where(torch.sigmoid(logit) >= th, 1.0, -1.0)
    m.zero_grad(); (logit * sign).sum().backward()
    h1.remove(); h2.remove()
    cam = torch.relu((grads[0].mean(dim=(2, 3), keepdim=True) * acts[0]).sum(1, keepdim=True))
    cam = Fnn.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[:, 0]
    cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
    return cam.detach().cpu().numpy(), torch.sigmoid(logit).detach().cpu().numpy()


p_now = P_SC['지금 (eye_cnn_548)']['깨끗함']
rng = np.random.default_rng(0)
ok_c = rng.choice(np.flatnonzero((yv8 == 1) & (p_now >= 0.4)), 2, replace=False)
ok_o = rng.choice(np.flatnonzero((yv8 == 0) & (p_now < 0.4)), 2, replace=False)
bad = np.flatnonzero((p_now >= 0.4) != (yv8 == 1)); bad = bad[np.argsort(-np.abs(p_now[bad] - 0.4))][:4]
pick9 = np.r_[ok_c, ok_o, bad]
fig, axes = plt.subplots(len(models8), len(pick9), figsize=(1.6 * len(pick9), 1.8 * len(models8)), squeeze=False)
for r, (k, (arch, st)) in enumerate(models8.items()):
    m = Net8(arch).to(device); m.net.load_state_dict(st)
    cams, probs = gradcam8(m, CAM_LAYER[arch](m.net), P8['깨끗함'][pick9])
    for c, j in enumerate(pick9):
        ax = axes[r, c]
        ax.imshow(P8['깨끗함'][j], cmap='gray'); ax.imshow(cams[c], cmap='jet', alpha=0.4); ax.axis('off')
        ax.set_title(f"{'감음' if yv8[j] else '뜸'} p={probs[c]:.2f}", fontsize=7, color='k' if (probs[c] >= 0.4) == (yv8[j] == 1) else 'r')
    axes[r, 0].text(-8, 32, k, fontsize=7, rotation=90, va='center', ha='right')
plt.suptitle('Grad-CAM (빨간 제목 = 틀림) — 칸 1~2 맞힌 감음 · 3~4 맞힌 뜸 · 5~8 지금 모델이 틀린 눈', fontsize=9)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/aug9_gradcam.png', dpi=90); plt.show()

### 9-3. 좌우 반전 평균 (TTA) — 학습 없이

- 눈을 좌우로 뒤집어 한 번 더 넣고 확률 평균 (서버 계산 2배). 깨끗함 · 상황별 평균 균형 정확도가 오르는지
- PC 에서도: `aihub548.py eval --set val --model … --tta` · `team --tta`

In [ ]:
@torch.no_grad()
def predict8_tta(m, X):
    return (predict8(m, X) + predict8(m, np.ascontiguousarray(X[:, :, ::-1]))) / 2


P_TTA = {}
for k, (arch, st) in models8.items():
    m = Net8(arch).to(device); m.net.load_state_dict(st)
    P_TTA[k] = {sc: predict8_tta(m, X) for sc, X in P8.items()}
if len(ens) > 1:
    P_TTA[E] = {sc: np.mean([P_TTA[k][sc] for k in ens], 0) for sc in P8}
tta = pd.DataFrame({k: {'깨끗함': metrics8(P_SC[k]['깨끗함'], yv8)['balanced_acc'], '깨끗함 + 반전': metrics8(P_TTA[k]['깨끗함'], yv8)['balanced_acc'],
                        '상황 평균': np.mean([metrics8(P_SC[k][sc], yv8)['balanced_acc'] for sc in P8]),
                        '상황 평균 + 반전': np.mean([metrics8(P_TTA[k][sc], yv8)['balanced_acc'] for sc in P8])} for k in P_TTA}).T.round(4)
display(tta)

### 9-4. 적외선(AI-Hub 173) 섞어 학습 — 7번에서 떨어진 적외선을 지키면서 컬러도

- **3-1 ~ 3-2 를 먼저** 돌려 둠 (`CROPS` · `AIDX` · `AY` · `APREP`)
- 한 에포크 = 548 train 장수, 그중 **173 이 30%** (각 데이터 안에서 감음 · 뜸 반반). 173 은 흑백이라 3채널로 복사해 같은 `scene_aug` (색 증강은 흑백에서 거의 효과 없음)
- `resnet18_ir`: `eye_cnn_aihub.pt` 에서 시작, LR/10 · 5 에포크 (8-5 `resnet18` 과 데이터만 다름)
- 비교: 548 val 깨끗함 · 상황 평균 · **AI-Hub 173 test** (7번 83.0% → 회복되는지)

In [ ]:
assert 'CROPS' in globals() and 'AIDX' in globals(), '3-1 ~ 3-2 를 먼저 실행'
IR_FRAC = 0.3


class DsMix(torch.utils.data.Dataset):          # 앞쪽 = 548 train, 뒤쪽 = 173 train
    def __init__(self):
        self.a, self.b = I8['train'], AIDX['train']
    def __len__(self):
        return len(self.a) + len(self.b)
    def __getitem__(self, i):
        rng = np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))
        if i < len(self.a):
            j = self.a[i]; img, y = C8[j], Y8[j]
        else:
            j = self.b[i - len(self.a)]; img, y = cv2.cvtColor(CROPS[j], cv2.COLOR_GRAY2BGR), AY[j]
        return torch.from_numpy(prep_eye(scene_aug(img, rng), SIZE))[None], torch.tensor(y, dtype=torch.float32)


ya, yb = Y8[I8['train']], AY[AIDX['train']]
wa = np.where(ya == 1, 0.5 / ya.mean(), 0.5 / (1 - ya.mean())) * (1 - IR_FRAC) / len(ya)
wb = np.where(yb == 1, 0.5 / yb.mean(), 0.5 / (1 - yb.mean())) * IR_FRAC / len(yb)
loader_mix = torch.utils.data.DataLoader(DsMix(), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                         sampler=torch.utils.data.WeightedRandomSampler(np.r_[wa, wb], len(ya), replacement=True))
STATE8['resnet18_ir'], HIST8['resnet18_ir'] = fit8('resnet18_ir', 'resnet18', init_aihub, LR / 10, 5, 2, loader=loader_mix,
                                                   data=f'AI-Hub 548 컬러 + AI-Hub 173 적외선 {IR_FRAC:.0%} (반쯤 감음 뺌)')
ARCH8['resnet18_ir'] = 'resnet18'
models8['resnet18_ir'] = ('resnet18', STATE8['resnet18_ir'])

rows_ir = {}
for k in ('지금 (eye_cnn_548)', 'resnet18', 'resnet18_ir'):
    arch, st = models8[k]
    m = Net8(arch).to(device); m.net.load_state_dict(st)
    p_sc = {sc: predict8(m, X) for sc, X in P8.items()}
    rows_ir[k] = {'548 깨끗함': metrics8(p_sc['깨끗함'], yv8)['balanced_acc'],
                  '548 상황 평균': round(float(np.mean([metrics8(p, yv8)['balanced_acc'] for p in p_sc.values()])), 4),
                  'AIHub173 test': metrics8(predict8(m, APREP['test']), AY[AIDX['test']])['balanced_acc'],
                  '반쯤→감음': round(float((predict8(m, P8_HALF) >= 0.4).mean()), 4)}
display(pd.DataFrame(rows_ir).T)
print('참고: 3-7 eye_cnn_aihub.pt 의 AI-Hub 173 test (기준 0.4) = 0.941')

### 9-5. 틀린 사진 모아 보기 — 라벨 잡음인가, 진짜 약점인가

- `WHO` 모델이 548 val 에서 틀린 눈 (확신하고 틀린 것부터) — **컬러 원본** (CNN 은 흑백으로 봄)
- 아래 표: 틀린 눈이 어떤 조건(거리 · 보는 방향 · 안경)에 몰렸나
- PC 의 Validation 151명 틀린 사진: `aihub548.py wrong --csv result_val_<모델>.csv`

In [ ]:
WHO = E if len(ens) > 1 else '지금 (eye_cnn_548)'          # ← 보고 싶은 모델 이름 (8-6 표의 줄 이름)
p_w = P_SC[WHO]['깨끗함']
wr = np.flatnonzero((p_w >= 0.4) != (yv8 == 1)); wr = wr[np.argsort(-np.abs(p_w[wr] - 0.4))]
vm8 = m8.iloc[I8['val']].reset_index(drop=True)
print(f'{WHO}: 틀린 눈 {len(wr)} / {len(yv8)} — 감음을 뜸으로 {int((yv8[wr] == 1).sum())} · 뜸을 감음으로 {int((yv8[wr] == 0).sum())}')
fig, axes = plt.subplots(3, 8, figsize=(14, 6))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, wr[:24]):
    ax.imshow(cv2.cvtColor(C8[I8['val'][i]], cv2.COLOR_BGR2RGB))
    ax.set_title(f"{'감음' if yv8[i] else '뜸'} p={p_w[i]:.2f}\n{vm8.dist[i]}cm {vm8.view[i]}{' 안경' if vm8.glasses[i] else ''}", fontsize=7)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/aug9_wrong.png', dpi=80); plt.show()
wdf = vm8.assign(wrong=((p_w >= 0.4) != (yv8 == 1)))
for col in ('dist', 'view', 'glasses'):
    display(wdf.groupby(col).agg(눈=('wrong', 'size'), 틀림=('wrong', 'sum'), 틀린비율=('wrong', 'mean')).round(4))

### 9-6. 저장 — `eye_cnn8_resnet18_ir.pt` · 의심 확인 숫자

- 형식은 8-7 과 같음. `aug9_check.json` = 라벨 섞기 · 좌우 반전 · 적외선 섞기 숫자 (사진 없음)
- PC: `aihub548.py eval --set val --model eye_cnn8_resnet18_ir.pt` · `team --model eye_cnn8_resnet18_ir.pt` (+ `--tta`)

In [ ]:
if 'resnet18_ir' in STATE8:
    torch.save(dict(
        state_dict=STATE8['resnet18_ir'], arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
        output='logit 1개, sigmoid = 감음 확률',
        prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
        crop='MediaPipe 눈 점 → eye_preprocess.crop_eye (서버와 같음)',
        aug='상황 증강 scene_aug (eye_cnn.ipynb 8-2)',
        data=f'AI-Hub 548 컬러 + AI-Hub 173 적외선 {IR_FRAC:.0%} (반쯤 감음 뺌) ← eye_cnn_aihub.pt 에서 시작',
        test={k: float(v) for k, v in rows_ir['resnet18_ir'].items()}, threshold=0.4,
        made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),
    ), f'{TRAIN_DIR}/eye_cnn8_resnet18_ir.pt')
    torch.load(f'{TRAIN_DIR}/eye_cnn8_resnet18_ir.pt', map_location='cpu', weights_only=True)
    print('저장: eye_cnn8_resnet18_ir.pt')
json.dump({'made': str(datetime.datetime.now())[:16], 'shuffled_label_val': metrics8(p_shuf, yv8), 'hist_shuffled': HIST_SHUF,
           'tta': tta.reset_index().to_dict('records'), 'ir_mix': rows_ir if 'rows_ir' in globals() else None},
          open(f'{TRAIN_DIR}/aug9_check.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: aug9_check.json')

## 10. PC 시험 모아 보기 — 중간 점검 피드백 형식 (11/04)

- **왜:** 지금까지 PC 에서 돌린 시험 (`notebooks/aihub548.py` eval · team · extra) 결과가 노트북 여기저기 표로만 있음 → **CSV 를 직접 읽어 같은 형식으로** 다시 냄. 중간 점검 피드백(11/04, `eye_aihub.pt` 기준)을 따름
  1. **개선 판단 기준:** 기준값을 0.05 ~ 0.95 로 바꿔 가며 151명에서 (뜬 눈 오경보율, 반쯤 감은 눈 취소율) 곡선 → 기존 · 새 모델을 한 그림에. **새 곡선이 아래면 개선**. 숫자 하나는 "오경보 3% 일 때 반쯤 감은 눈 취소율 ○○%" (오경보 고정)
  2. **보고 형식 고정:** 전체 / 판정됨 / 판정 불가 (얼굴 못 찾음 · CNN 값 없음) / 취소 · 오경보 / 비율 / 95% 범위
  3. **조건 사진은 조건별로:** 고개 돌림 · 숙임 · 마스크 · 선글라스 각각 감은 눈 취소 / 뜬 눈 오경보 / 판정 불가
  4. **데이터를 뺄 때는 근거:** CEW 를 뺀 이유를 '외국인 눈' 이 아니라 확인한 원인으로 (CNN 단독 · EAR 단독 · 얼굴 검출 · 눈 크기 · 548 을 같은 크기로 줄여 보기)
- **데이터 역할 (피드백 · 7-5 와 같음):** 기준값을 고른 데이터로 성능을 재면 실제보다 좋게 나옴

| 데이터 | 역할 | 쓰는 곳 |
|---|---|---|
| 548 학습용 묶음의 val 15% (사람 단위) | **고르기** | 기준값 · 여유값 · 조기 종료 · 모델 비교 (Colab 7 · 8 · 9번) |
| 548 Validation 151명 | **성능 측정** | 최종 숫자 보고 (이 구간 10-2 · 10-3). 곡선은 기준값을 고르지 않고 모델끼리 비교만 |
| 팀 사진 | **시연 확인** | "우리 폰 · 조명에서 되는가" (10-4) |
| AI-Hub 71421 Validation 13명 중 6명 | 마지막 한 번 | 후보 교체 판단 (11/03, 규칙 미리 정함 → 10-6) |
| CEW | 참고 | 판단에 안 씀 — 왜 안 맞는지 원인만 확인 (10-7) |

- **판정 규칙 = 서버 그대로** (`config.py`): 감음 = CNN ≥ 0.4 / **경고 취소(확실히 뜸) = CNN < 0.3 그리고 EAR ≥ 0.26** (0.23 + 0.03). 판정 불가(얼굴 못 찾음)는 서버가 취소하지 않음 → 앱 경고 그대로
  - **감은 눈 · 반쯤 감은 눈 취소율** = 진짜 감은 눈의 경고를 서버가 취소한 비율 (놓침) / **뜬 눈 오경보율** = 뜬 눈인데 취소 못 해 경고가 나간 비율
- **PC 에서 돌린 것 (전부 저장소 밖 CSV, 사진마다 한 줄):**

| 명령 (`sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py ...`) | 결과 CSV | 이 구간 |
|---|---|---|
| `eval --set val` (11/02, `eye_aihub.pt`) | `D:\aihub548\result_val.csv` | 10-2 · 10-3 |
| `eval --set val --model eye_cnn_548.pt` (11/04 다시 — 눈 폭 칸 추가) | `D:\aihub548\result_val_eye_cnn_548.csv` | 10-2 · 10-3 · 10-7 |
| `eval --set val --model eye_cnn_548.pt --scale 배율` (11/04) | `D:\aihub548\result_val_eye_cnn_548_s배율.csv` | 10-7 |
| `eval --set val --model eye_cnn8_*.pt` (11/03 후보) | `D:\aihub548\result_val_eye_cnn8_*.csv` | 8-8 |
| `team --model eye_cnn_548.pt` · `team --model eye_cnn_aihub.pt` (11/04 다시) | `D:\aihub173\kpi_eye\team_eye_cnn_548.csv` · `team_eye_cnn_aihub.csv` | 10-4 · 10-5 |
| `extra --model eye_cnn_548.pt` · `--model eye_cnn8_resnet18_ir.pt` (11/03, 71421) | `D:\aihub71421\extra_sel_*.csv` | 10-6 |
| `extra --dir D:\cew\faces\dataset_B_FacialImages --pad 0.4 --model ...` (11/04 다시) | `D:\cew\faces\extra_dataset_B_FacialImages_*.csv` | 10-7 |
| `wrong --csv result_val_eye_cnn_548.csv` (11/03) | `D:\aihub548\wrong_*.png` (눈만) | 9-5 |

- **실행:** Colab — 위 CSV 를 드라이브 `sleep_detector/pc_results/` 에 같은 이름으로 올리고 10-1 부터 (GPU 필요 없음, 1 ~ 9번 불필요) / PC — 그대로 (경로 자동)
- 🔴 **재배포 금지:** 사진마다 한 줄 결과도 AI-Hub 결과 → 저장소 밖 · 본인 드라이브에만 (GitHub · 단톡 X). 팀 사진 CSV 도 팀 안에서만

### 10-0. PC 작업 기록 — 548 학습 묶음 늘리기 (11/03 ~ 11/04)

- 받기 (`download --keys 55884 55886`): TS_G2_1 (120GB) · TS_U_1 (70GB) — AI-Hub 는 이어받기(Range)가 안 되어 끊기면 처음부터
- **11/04 받은 파일 두 개가 망가져 있었음 → 고쳐서 씀** (자르기가 00:48 에 실패해서 발견)
  - **TS_G2_1.zip:** 조각(.part) 합칠 때 29 ~ 30GB 사이 **0.80GB 빠짐** → 그 뒤 항목 위치가 다 어긋남 (zip 목차는 원래 크기 기준). 목차 위치와 실제 파일 머리를 맞춰 보며 **빠진 구간만 버리고 새 목차를 파일 끝에 붙임** (CRC 확인)
    - 버린 항목 1,976개 = **전부 사람 079 의 DEPTH(깊이) 사진** → 자르기에 쓰는 RGB · meta.xml 은 **0장 손실**
  - **TS_U_1.Zip:** 이름은 .Zip 인데 실제로는 **7z** 압축 → Windows 기본 `tar` (libarchive) 로 RGB 사진 · meta.xml 만 꺼내 압축 없는 zip 으로 다시 묶음 (원본은 `TS_U_1.7z` 로 이름만 바꿔 둠)
- 자르기 (`crop` 흑백 · `crop --color`): 사람 분할은 `D:\aihub548\people_split.json` 에 고정 — 예전 324명은 그대로, **새 사람만** 85 : 15. Validation 151명은 자동으로 뺌
  - **결과 (11/04 17:18):** 학습 zip 3개 (TS_G1_1 · TS_G2_1 · TS_U_1) → 사진 55,101장 → 눈 **뜸 90,458 · 감음 8,502 · 반쯤 5,726** (흑백, 컬러는 뜸 90,509) / 얼굴 못 찾음 1,703장 · 눈 짝 안 맞음 2,110
  - 사람: Validation 이라 뺀 151명 · 남은 **763명 = train 648 / val 115** (겹침 0). **G1 예전 324명은 train 275 / val 49 그대로** (11/03 과 같음), 새 사람 G2 195명 (171 / 24) · U1 244명 (202 / 42)
  - 반쯤 뺀 학습용: train 뜸 76,876 · 감음 7,340 / val 뜸 13,582 · 감음 1,162 (11/03 의 324명 · 눈 44,707 보다 감은 눈 약 2.5배)
  - 드라이브: `sleep_detector/aihub548_eye` ← `eye_npz` (561MB) · `aihub548_eye_color` ← `eye_npz_color` (2.0GB), 옛 파일은 지우고 바꿔 넣기. **다시 학습 전 드라이브 `eye_cnn/` 의 `aihub548_ft_last.pt` · `aug8_*_last.pt` 같은 이어받기 파일을 지우거나 이름 바꾸기** (안 그러면 옛 학습을 이어감)

### 10-1. PC 결과 불러오기 · 보고 형식 함수

- `report(d, 묶음)`: 묶음 × 라벨마다 **전체 / 판정됨 / 판정 불가 (얼굴 못 찾음 · CNN 값 없음) / 취소 · 오경보 / 비율 / 95% 범위**
  - 뜬 눈 → **오경보** (취소 못 함), 반쯤 감음 · 감음 → **취소** (놓침)
  - 95% 범위: 사람 번호가 있고 20명 이상이면 **사람 단위 부트스트랩** (한 사람 사진끼리 닮아서 사진 단위보다 넓게 나옴, 7-6 과 같음), 아니면 Wilson
- 판정은 CSV 의 `prob` (CNN 감음 확률) · `ear` 로 **서버 규칙을 다시 계산** — 기준값을 바꿔 볼 수 있게. 기준 0.4 에서 CSV 의 서버 판정과 같은지 먼저 확인

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PC = {  # PC 경로 — Colab 이면 드라이브 sleep_detector/pc_results/ 에 같은 이름으로
    'val_old': r'D:\aihub548\result_val.csv',                      # 548 Validation 151명 · eye_aihub.pt (11/02)
    'val_new': r'D:\aihub548\result_val_eye_cnn_548.csv',          # 같은 사진 · eye_cnn_548.pt (11/04, 눈 폭 칸)
    'val_small': r'D:\aihub548\result_val_eye_cnn_548_s{SCALE}.csv',  # 같은 사진을 줄여서 (10-7)
    'team_old': r'D:\aihub173\kpi_eye\team_eye_cnn_aihub.csv',     # 팀 사진 · 조건 사진
    'team_new': r'D:\aihub173\kpi_eye\team_eye_cnn_548.csv',
    'x71421_new': r'D:\aihub71421\extra_sel_eye_cnn_548.csv',      # 71421 마지막 시험 (11/03)
    'x71421_ir': r'D:\aihub71421\extra_sel_eye_cnn8_resnet18_ir.csv',
    'x71421_old': r'D:\aihub71421\extra_sel_eye_cnn_aihub.csv',
    'cew_new': r'D:\cew\faces\extra_dataset_B_FacialImages_eye_cnn_548.csv',
    'cew_old': r'D:\cew\faces\extra_dataset_B_FacialImages_eye_cnn_aihub.csv',
}
SCALE = '0.33'   # 10-7: 548 을 CEW 눈 크기로 줄인 배율 (눈 폭 가운데값 548 45.6px → CEW 15.2px)
PC['val_small'] = PC['val_small'].format(SCALE=SCALE)

try:                                   # Colab 이면 드라이브
    from google.colab import drive
    drive.mount('/content/drive')
    PCR = '/content/drive/MyDrive/sleep_detector/pc_results'
    PATH = {k: f'{PCR}/{os.path.basename(v)}' for k, v in PC.items()}
except ImportError:                    # PC 면 그대로
    PATH = dict(PC)

# 그림 한글 글꼴 (1-1 을 안 돌렸을 때만 설치 · PC 는 맑은 고딕)
import subprocess, matplotlib as mpl, matplotlib.font_manager as fm
if os.name == 'nt':
    mpl.rcParams['font.family'] = 'Malgun Gothic'
    mpl.rcParams['axes.unicode_minus'] = False
elif mpl.rcParams['font.family'] != ['NanumGothic'] and os.path.isdir('/usr/share'):
    NANUM = '/usr/share/fonts/truetype/nanum'
    if not fm.findSystemFonts([NANUM]):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    for f in fm.findSystemFonts([NANUM]):
        fm.fontManager.addfont(f)
    if fm.findSystemFonts([NANUM]):
        mpl.rcParams['font.family'] = 'NanumGothic'
        mpl.rcParams['axes.unicode_minus'] = False

EAR_T, EAR_M, CNN_T, CNN_M = 0.23, 0.03, 0.4, 0.1   # 서버 config.py 와 같음 (EAR 기준 · 여유, CNN 기준 · 여유)
LAB = {'open': '뜸', 'half': '반쯤 감음', 'closed': '감음'}


def load(k):
    if not os.path.exists(PATH[k]):
        print('없음 (건너뜀):', PATH[k])
        return None
    d = pd.read_csv(PATH[k])
    d['face'] = d['face'].astype(str) == 'True'
    d['ok'] = d['face'] & d['prob'].notna()     # 판정됨 = 얼굴 찾고 CNN 값 있음
    return d


def cancel(d, t=CNN_T, how='server'):
    # 경고 취소(확실히 뜸) 다시 계산 — server: CNN < t - 0.1 그리고 EAR ≥ 0.26 / cnn: CNN 만 / ear: EAR 만
    c, e = d['prob'] < t - CNN_M, d['ear'] >= EAR_T + EAR_M
    return {'server': c & e, 'cnn': c, 'ear': e}[how] & d['ok']


def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - h), min(1.0, c + h)


def ci95(d, hit, B=2000, seed=0):
    # d 의 판정된 사진 중 hit 비율의 95% 범위 — 사람 20명 이상이면 사람 단위 부트스트랩, 아니면 Wilson
    if 'person' in d and d['person'].nunique() >= 20:
        g = pd.DataFrame({'p': d['person'].to_numpy(), 'h': hit.to_numpy()}).groupby('p')['h'].agg(['sum', 'count'])
        rng = np.random.default_rng(seed)
        idx = rng.integers(0, len(g), (B, len(g)))
        r = g['sum'].to_numpy()[idx].sum(1) / np.maximum(g['count'].to_numpy()[idx].sum(1), 1)
        return np.percentile(r, 2.5), np.percentile(r, 97.5)
    return wilson(int(hit.sum()), len(hit))


def report(d, keys=(), t=CNN_T, how='server'):
    # 보고 형식 (피드백): 묶음 × 라벨 → 전체 / 판정됨 / 판정 불가 (얼굴 못 찾음 · CNN 없음) / 취소 · 오경보 / 비율 / 95% 범위
    rows = []
    for k, g in d.groupby(list(keys) + ['label'], sort=False):
        k = k if isinstance(k, tuple) else (k,)
        lab = k[-1]
        f = g[g['ok']]
        c = cancel(f, t, how)
        hit = ~c if lab == 'open' else c
        lo, hi = ci95(f, hit)
        rows.append({**dict(zip(keys, k[:-1])), '라벨': LAB.get(lab, lab), '전체': len(g), '판정됨': len(f),
                     '판정 불가 (얼굴 못 찾음)': int((~g['face']).sum()), '판정 불가 (CNN 없음)': int((g['face'] & g['prob'].isna()).sum()),
                     '지표': '뜬 눈 오경보' if lab == 'open' else '취소 (놓침)', '건수': int(hit.sum()),
                     '비율': f'{hit.mean():.1%}' if len(f) else '—', '95% 범위': f'{lo:.1%} ~ {hi:.1%}' if len(f) else '—'})
    out = pd.DataFrame(rows)
    order = {v: i for i, v in enumerate(LAB.values())}
    return out.sort_values(list(keys) + ['라벨'], key=lambda s: s.map(order) if s.name == '라벨' else s).reset_index(drop=True)


D = {k: load(k) for k in PATH}
for k, d in D.items():   # 기준 0.4 에서 다시 계산한 취소 = CSV 의 서버 판정
    if d is not None and 'cancel' in d:
        same = (cancel(d) == (d['cancel'].astype(str) == 'True')).mean()
        print(f'{k:<11} 사진 {len(d):5d} · 판정됨 {int(d["ok"].sum()):5d} · 서버 판정과 같음 {same:.2%}')

### 10-2. 기준값 곡선 — 151명, 기존 `eye_aihub.pt` ↔ 새 `eye_cnn_548.pt` (피드백 1)

- CNN 기준값 `t` 를 0.05 ~ 0.95 로 바꿔 가며 (취소는 CNN < t − 0.1) **가로 = 뜬 눈 오경보율, 세로 = 반쯤 감은 눈 취소율**. 둘 다 낮을수록 좋음 → **아래 · 왼쪽 곡선이 개선**
  - 왼쪽 그림 = **서버 규칙** (CNN · EAR 둘 다 확실히 뜸일 때만 취소) / 오른쪽 = **CNN 단독** (EAR 조건 없이 — 모델끼리만 비교)
  - ★ = 지금 서버 기준 0.4
- **오경보 고정 숫자:** 오경보가 목표 이하가 되는 가장 낮은 기준에서 반쯤 감은 눈 취소율 (서버 규칙 3% · CNN 단독 1%)
  - ⚠️ 이 기준은 151명으로 **고른 것이라 서버 기준 후보가 아님** — 모델 비교용 숫자. 서버 기준은 548 val 15% 로 골랐음 (7-4)
- 같은 사진 9,337장 (감음 · 반쯤 전부 + 사람 · 거리마다 뜸 25장), 판정됨 9,090장

In [ ]:
V = {'기존 eye_aihub.pt': D['val_old'], '새 eye_cnn_548.pt': D['val_new']}
TS = np.round(np.arange(0.05, 0.951, 0.05), 2)
FINE = np.round(np.arange(0.11, 1.001, 0.01), 2)   # 0.10 이하는 취소가 아예 없음 (CNN < 0)


def point(d, t, how):
    f = d[d['ok']]
    c = cancel(f, t, how)
    return dict(오경보=1 - c[f['label'] == 'open'].mean(), 반쯤취소=c[f['label'] == 'half'].mean(), 감음취소=c[f['label'] == 'closed'].mean())


curve = pd.DataFrame([{'모델': n, '규칙': how, '기준': t, **point(d, t, how)} for n, d in V.items() for how in ('server', 'cnn') for t in TS])
display((curve.pivot_table(index='기준', columns=['규칙', '모델'], values=['오경보', '반쯤취소', '감음취소']) * 100).round(1))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, how, title, xmax in ((axs[0], 'server', '서버 규칙 (CNN · EAR 둘 다 확실히 뜸일 때만 취소)', 6), (axs[1], 'cnn', 'CNN 단독', 3)):
    for (n, d), st in zip(V.items(), ('o--', 's-')):
        pts = pd.DataFrame([point(d, t, how) for t in FINE])
        ax.plot(pts['오경보'] * 100, pts['반쯤취소'] * 100, st, ms=3, label=n)
        p = point(d, CNN_T, how)
        ax.plot(p['오경보'] * 100, p['반쯤취소'] * 100, '*', ms=14, color=ax.lines[-1].get_color())
    ax.set_xlim(0, xmax); ax.set_ylim(0, 100)
    ax.set_xlabel('뜬 눈 오경보율 (%)'); ax.set_ylabel('반쯤 감은 눈 취소율 (%)'); ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle('548 Validation 151명 — 기준값 0.11 ~ 1.0 (★ = 지금 0.4), 아래 곡선이 개선')
plt.tight_layout(); plt.show()


def at_fa(d, how, target):
    # 오경보 ≤ target 이 되는 가장 낮은 기준 → 그때 반쯤 · 감은 눈 취소율 (사람 단위 95% 범위)
    for t in FINE:
        p = point(d, t, how)
        if p['오경보'] <= target:
            f = d[d['ok']]
            h = f[f['label'] == 'half']
            lo, hi = ci95(h, cancel(h, t, how))
            return {'기준': t, '오경보': f"{p['오경보']:.1%}", '반쯤 감은 눈 취소율': f"{p['반쯤취소']:.1%}", '95% 범위': f'{lo:.1%} ~ {hi:.1%}', '감은 눈 취소율': f"{p['감음취소']:.1%}"}
    return {'기준': '없음', '오경보': f"최소 {min(point(d, t, how)['오경보'] for t in FINE):.1%}"}


display(pd.DataFrame([{'규칙': r, '목표 오경보': f'{g:.0%}', '모델': n, **at_fa(d, how, g)}
                      for r, how, g in (('서버 규칙', 'server', 0.03), ('CNN 단독', 'cnn', 0.01)) for n, d in V.items()]))

### 10-3. 151명 보고 표 — 지금 서버 설정 (기준 0.4), 피드백 2 형식

- 같은 사진을 두 모델로. 묶음 = 그룹 (G1 · G2 · U — 사람 번호 앞부분). **95% 범위는 사람 단위 부트스트랩**

In [ ]:
for n, d in V.items():
    d = d.assign(그룹=d['person'].str.split('_').str[0])
    print(f'== {n} ==')
    display(report(d))
    display(report(d, ['그룹']))

### 10-4. 팀 사진 · 조건 사진 — 조건별 (피드백 3)

- 기본 사진 (밝은 곳 · 어두운 곳) 은 라벨별, 조건 사진 (고개 돌림 · 고개 숙임 · 마스크 · 선글라스) 은 **조건마다 따로** — 감은 눈 취소 / 뜬 눈 오경보 / 판정 불가
- 사진 수: 기본 428장 (사람 5명 × 밝은 · 어두운 × 뜸 16 · 반쯤 11 · 감음 16, 한 장 빠짐) + 조건 90장 (고개 30 · 마스크 40 · 선글라스 20) = **518장**
  - 예전 발표 표의 '458장' 은 기본 428장 + 고개 30장이었음 — 표에 들어간 428장 (159 · 109 · 160) 과 **30장 차이 = 고개 사진 (뜬 눈만, 따로 봄)** → 얼굴 못 찾음 · 라벨 제외가 아님
- 팀 사진은 사람이 5명뿐 → 95% 범위는 Wilson (사진 단위, 실제보다 좁을 수 있음)

In [ ]:
COND = {'turn': '고개 돌림', 'down': '고개 숙임', 'mask': '마스크', 'sun': '선글라스'}
for n, k in (('기존 eye_aihub.pt', 'team_old'), ('새 eye_cnn_548.pt', 'team_new')):
    t = D[k]
    if t is None:
        continue
    t = t.assign(조건=t['cond'].map(COND).fillna(t['group']))
    print(f'== {n} — 기본 사진 (밝은 곳 + 어두운 곳) ==')
    display(report(t[t['cond'].isna()]))
    display(report(t[t['cond'].isna()], ['조건']))
    print(f'== {n} — 조건 사진 ==')
    display(report(t[t['cond'].notna()], ['조건']))

### 10-5. CNN 개인 기준 (팀장 제안, 11/03) — 효과 확인

- **제안:** 운행 전 찍는 뜬 눈 5장의 CNN 감음 확률 + 여유값 → 사람마다 CNN 기준 (EAR 개인 기준처럼)
- **시험:** 팀 기본 사진에서 사람마다 **밝은 곳 뜬 눈 s01 ~ s05** 를 운행 전 사진으로 → 개인 기준 = min(0.4, 5장 최대 + 여유값). 나머지 사진으로 반쯤 감은 눈 → 감음 · 뜬 눈 → 감음 (오판) 을 고정 0.4 와 비교
  - 개인 기준을 정하는 데 쓴 5장은 채점에서 뺌 (뜬 눈 160 → 135장)

In [ ]:
t = D['team_new']
t = t[t['cond'].isna() & t['ok']].assign(사람=lambda x: x['file'].str[0])
calib = t[t['file'].str.contains('_bright_open_s0[1-5]', regex=True)]
rest = t.drop(calib.index)
base = calib.groupby('사람')['prob'].max()
print('사람마다 운행 전 뜬 눈 5장 CNN 최대:', base.round(3).to_dict())
rows = [{'방식': '고정 0.4 (지금)', '반쯤 → 감음': f"{int((rest.loc[rest['label'] == 'half', 'prob'] >= CNN_T).sum())}/{int((rest['label'] == 'half').sum())}",
         '뜬 눈 → 감음 (오판)': f"{int((rest.loc[rest['label'] == 'open', 'prob'] >= CNN_T).sum())}/{int((rest['label'] == 'open').sum())}",
         '감은 눈 → 감음': f"{int((rest.loc[rest['label'] == 'closed', 'prob'] >= CNN_T).sum())}/{int((rest['label'] == 'closed').sum())}"}]
for m in (0.1, 0.2, 0.25, 0.3):
    thr = rest['사람'].map(np.minimum(CNN_T, base + m))
    rows.append({'방식': f'개인 기준 (5장 최대 + {m})', **{c: f"{int((rest.loc[rest['label'] == l, 'prob'] >= thr[rest['label'] == l]).sum())}/{int((rest['label'] == l).sum())}"
                                                       for c, l in (('반쯤 → 감음', 'half'), ('뜬 눈 → 감음 (오판)', 'open'), ('감은 눈 → 감음', 'closed'))}})
display(pd.DataFrame(rows))

### 10-6. 마지막 시험 — AI-Hub 71421 「디스플레이 중심 안구 움직임」 (11/03, 한 번만)

- Validation 13명 중 6명 (149 ~ 154번), 감은 눈 261 · 뜬 눈 795 (노트북 · 모니터 · 차량 LCD · 스마트폰 · 태블릿). 라벨 = 홍채 없음 + 눈꺼풀 세로/가로 < 0.15 → 감음 / 홍채 있음 + 0.25 이상 → 뜸
- 판정 규칙 (시험 전에 정함): 도전 모델 (`resnet18_ir`) 이 감은 눈 취소율 · 뜬 눈 오경보율 둘 다 지금 모델보다 나쁘지 않을 때만 교체 → **지금 모델 유지** (8-8)
- 기존 `eye_aihub.pt` 는 비교용으로만 (11/04, 고르기에 안 씀)
- 사람 6명뿐 → 95% 범위 Wilson. 사람 · 기기별로도

In [ ]:
DEV = {'S': '스마트폰', 'T': '태블릿', 'L': '노트북', 'V': '차량 LCD', 'M': '모니터'}
for n, k in (('새 eye_cnn_548.pt (지금)', 'x71421_new'), ('도전 resnet18_ir', 'x71421_ir'), ('기존 eye_aihub.pt (비교만)', 'x71421_old')):
    x = D[k]
    if x is None:
        continue
    b = x['file'].str.split('/').str[-1].str.split('_')
    x = x.assign(사람=b.str[2], 기기=b.str[5].map(DEV))
    print(f'== {n} ==')
    display(report(x))
    f = x[x['ok']]
    print(f"  CNN 단독 뜬 눈 → 감음 {(f.loc[f['label'] == 'open', 'prob'] >= CNN_T).mean():.1%} · EAR 단독 뜬 눈 EAR < 0.26 {(f.loc[f['label'] == 'open', 'ear'] < EAR_T + EAR_M).mean():.1%}")
    display(report(x, ['사람']))
    display(report(x, ['기기']))

### 10-7. CEW 원인 확인 — 왜 뜬 눈 오경보가 높나 (피드백 4)

- 11/03 에는 "외국인 눈이라 판단에 안 씀" 으로 뺐음 → **확인된 원인이 아님** (피드백). 아래 순서로 원인을 찾음
  1. **CNN 단독 · EAR 단독** 뜬 눈 오경보율 → 어느 쪽이 무너지나
  2. **얼굴 검출 실패율**
  3. **눈 crop 원래 크기** (확대 전 눈 폭 px = 두 눈 눈꼬리 사이 평균) 분포 — CEW 는 100×100 얼굴 사진이라 눈이 수 픽셀일 수 있음
  4. **548 을 CEW 눈 크기로 줄여서** 같은 시험 (`eval --scale`) → 오경보가 오르면 원인은 **해상도**
- CEW: 뜬 눈 1,231 · 감은 눈 1,192 (Closed Eyes in the Wild, 얼굴 100×100 → 사방 40% 늘려 MediaPipe)

In [ ]:
rows = []
for n, k in (('CEW · 기존 eye_aihub.pt', 'cew_old'), ('CEW · 새 eye_cnn_548.pt', 'cew_new'), ('548 151명 · 새 eye_cnn_548.pt', 'val_new'),
             (f'548 151명 ×{SCALE} 줄임 · 새 eye_cnn_548.pt', 'val_small')):
    d = D[k]
    if d is None:
        continue
    f = d[d['ok']]
    o = f[f['label'] == 'open']
    rows.append({'데이터 · 모델': n, '얼굴 못 찾음': f"{(~d['face']).mean():.1%}", '뜬 눈 판정됨': len(o),
                 '서버 규칙 오경보': f"{(~cancel(o)).mean():.1%}", 'CNN 단독 오경보 (CNN ≥ 0.3)': f"{(~cancel(o, how='cnn')).mean():.1%}",
                 'EAR 단독 오경보 (EAR < 0.26)': f"{(~cancel(o, how='ear')).mean():.1%}",
                 '감은 눈 취소': f"{cancel(f[f['label'] == 'closed']).mean():.1%}",
                 '눈 폭 가운데값 (px)': f"{d.loc[d['ok'], 'eye_w'].median():.1f}" if 'eye_w' in d else '—'})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(9, 3.4))
for n, k in (('CEW', 'cew_new'), ('548 151명', 'val_new'), (f'548 ×{SCALE}', 'val_small')):
    d = D[k]
    if d is not None and 'eye_w' in d:
        ax.hist(d.loc[d['ok'], 'eye_w'], bins=np.arange(0, 80, 1), alpha=0.5, label=n, density=True)
ax.set_xlabel('눈 폭 (px, 확대 전 — 서버가 64×64 로 늘리기 전)'); ax.set_ylabel('비율'); ax.legend(); ax.set_title('눈 crop 원래 크기')
plt.tight_layout(); plt.show()

# 눈 폭 구간별 뜬 눈 오경보 (548 원래 + 줄인 것 합쳐서) — 눈이 작아질수록 오르나
both = pd.concat([D[k].assign(데이터=k) for k in ('val_new', 'val_small') if D[k] is not None])
o = both[both['ok'] & (both['label'] == 'open')]
o = o.assign(눈폭=pd.cut(o['eye_w'], [0, 10, 15, 20, 30, 45, 200]))
display(o.groupby('눈폭', observed=True).apply(lambda g: pd.Series({'뜬 눈': len(g), '서버 오경보': f"{(~cancel(g)).mean():.1%}",
                                                               'CNN 단독': f"{(~cancel(g, how='cnn')).mean():.1%}", 'EAR 단독': f"{(~cancel(g, how='ear')).mean():.1%}"})))

### 10-8. 결과 정리 (11/04 PC)

- 다시 계산한 서버 판정 = CSV 의 서버 판정 **100% 같음** (10개 CSV 전부). 151명 `eye_cnn_548` 은 11/04 에 눈 폭 칸을 넣어 다시 돌림 → 11/03 결과와 확률 · EAR · 판정 **100% 같음**

**① 기준값 곡선 (10-2, 151명) — 새 곡선이 아래 = 개선**

| 비교 (오경보 고정) | 기존 `eye_aihub.pt` | 새 `eye_cnn_548.pt` |
|---|---|---|
| 서버 규칙, 오경보 3% → 반쯤 감은 눈 취소율 | 55.9% (43.4 ~ 67.0%, 기준 0.25) | **50.2%** (35.8 ~ 63.6%, 기준 0.62) |
| 같은 지점 감은 눈 취소율 | 6.6% | **0.7%** |
| CNN 단독, 오경보 1% → 반쯤 감은 눈 취소율 | 64.3% (53.2 ~ 73.2%, 기준 0.18) | **46.4%** (30.4 ~ 60.2%, 기준 0.38) |
| 같은 지점 감은 눈 취소율 | 10.9% | **0.0%** |

- ⚠️ 위 기준값 (0.25 · 0.62 · 0.18 · 0.38) 은 **151명 곡선 위의 점 (보고용)** — 서버 기준값으로 쓰지 않음. 서버 t 는 548 학습용 val 115명에서만 고름 (12번, 11/04 팀장 규칙)
- 0.15 ~ 0.95 모든 기준에서 새 곡선이 아래. 단 **반쯤 감은 눈은 사람 단위 95% 범위가 겹침** → "개선 경향" 까지. **감은 눈 취소 (9.2% → 0.0%, 기준 0.4) 는 확실한 개선**
- 서버 규칙은 EAR 조건 때문에 오경보가 **2.8% 아래로 안 내려감** (EAR 단독 오경보 2.8%) → 오경보를 더 줄이려면 CNN 이 아니라 EAR 쪽

**② 151명 보고 표 (10-3, 기준 0.4)**

| 모델 | 뜬 눈 오경보 | 반쯤 감은 눈 취소 | 감은 눈 취소 | 판정 불가 (얼굴 못 찾음) |
|---|---|---|---|---|
| 기존 | 2.8% (209/7,380, 1.6 ~ 4.3%) | 61.5% (498/810, 49.9 ~ 71.9%) | 9.2% (83/900, 6.7 ~ 12.0%) | 247/9,337 (CNN 없음 0) |
| **새** | 3.2% (236/7,380, 2.0 ~ 4.7%) | **43.6%** (353/810, 27.6 ~ 58.0%) | **0.0%** (0/900) | 같음 |

**③ 팀 사진 (10-4, 518장)** — 기본 428장 + 조건 90장. 예전 표의 '빠진 30장' = **고개 사진 (뜬 눈만)**, 얼굴 못 찾음 · 라벨 제외 아님. 얼굴 못 찾음은 마스크 2장뿐

| 조건 (새 모델) | 감은 눈 취소 | 뜬 눈 오경보 | 판정 불가 |
|---|---|---|---|
| 기본 (밝은 곳 + 어두운 곳) | 0/159 | 1/160 (0.6%) | 0 |
| 고개 돌림 | — | **0/20** | 0 |
| 고개 숙임 | — | **10/10 (100%)** | 0 |
| 마스크 | 0/19 | 4/19 (21.1%) | 2 (감음 1 · 뜸 1) |
| 선글라스 | 0/10 | 6/10 (60.0%) | 0 |

- 기본 사진 반쯤 감은 눈 취소: 기존 29/109 (26.6%) → 새 **25/109 (22.9%)**. 조건 사진 서버 오경보는 두 모델이 같음
- 조건 사진 뜬 눈이 왜 경고되나 (취소 못 한 이유, 새 모델 · 뜬 눈 중 CNN ≥ 0.3 / EAR < 0.26): **고개 숙임 CNN 10 · EAR 10 / 10 — 둘 다 무너짐** (기존 모델 CNN 8/10) · 선글라스 CNN 5 · EAR 5 / 10 · 마스크 CNN 1 · EAR 4 / 19 (EAR 쪽)
- 예전 메모의 '고개 뜬 눈 오판 9 ~ 10/30' 은 돌림 · 숙임을 합친 숫자 → **숙임만 문제** (돌림은 0/20)

**④ CNN 개인 기준 (10-5)** — 5명 모두 운행 전 뜬 눈 CNN 최대 0.000 ~ 0.005 → 개인 기준 = 기준을 낮춘 것과 같음. 여유값 0.25: 반쯤 78 → 80/109 · 뜬 눈 오판 0 → 1/135 → **고정 0.4 유지** (11/03 결론 그대로)

**⑤ 71421 (10-6)** — 감은 눈 취소 세 모델 모두 0/261

| 모델 | 뜬 눈 오경보 (서버) | CNN 단독 뜬 눈 → 감음 | EAR 단독 (< 0.26) |
|---|---|---|---|
| 새 `eye_cnn_548` (지금) | 41.9% (38.5 ~ 45.3%) | 5.4% | 41.0% |
| 도전 `resnet18_ir` | 48.8% | 21.1% | 41.0% |
| 기존 `eye_aihub` (비교만) | 41.5% (38.1 ~ 45.0%) | **1.5%** | 41.0% |

- ⚠️ **이 데이터에서는 기존 모델의 CNN 이 더 좋음** (뜬 눈 오판 1.5% vs 5.4%). 서버 결과(오경보)는 둘이 같음 — 오경보 대부분이 EAR (화면을 내려다봐서 뜬 눈도 EAR 작음)
- 사람별 차이가 큼: 152번 오경보 69.6% · 155번 6.8% / 기기별: 차량 LCD 63.2% · 노트북 49.5% · 태블릿 10.2%

**⑥ CEW 원인 (10-7, 피드백 4)**

| 데이터 · 모델 | 얼굴 못 찾음 | 서버 오경보 | CNN 단독 | EAR 단독 | 눈 폭 가운데값 |
|---|---|---|---|---|---|
| CEW · 기존 | 0.8% | 41.9% | 33.0% | 32.0% | 16px |
| CEW · 새 | 0.8% | 68.5% | **67.9%** | 32.0% | 16px |
| 548 151명 · 새 | 2.6% | 3.2% | 0.9% | 2.8% | 46px |
| 548 151명 **0.33배로 줄임** · 새 | 2.0% | 7.8% | **6.3%** | 2.9% | 15px |

1. **CNN · EAR 둘 다 무너짐** — CNN 단독 67.9% (새) · 33.0% (기존), EAR 단독 32.0%. 548 은 각각 0.9% · 2.8%
2. 얼굴 검출 실패는 0.8% 뿐 → 원인 아님
3. **CEW 눈은 548 의 1/3 크기** (15 ~ 16px vs 46px)
4. **548 을 같은 크기로 줄이면 CNN 단독 오경보 0.9% → 6.3% (7배)** — 눈 폭 10px 이하 18.0% · 10 ~ 15px 9.9% · 30px 이상 0.5 ~ 1.3% → **해상도는 원인 중 하나 (확인됨)**
   - 하지만 줄인 548 (6.3%) 은 CEW (67.9%) 의 **1/10** — 해상도만으로는 설명 안 됨. EAR 도 줄인 548 은 2.9% 인데 CEW 는 32.0% → **EAR 이 높은 것도 해상도 때문이 아님**
- **결론 (뺀 근거):** "외국인 눈이라서" 가 아니라 **"CEW 는 눈이 15px 로 작고 (해상도 영향 확인: 548 을 줄이면 CNN 오경보 7배), 그 밖의 사진 차이 (인터넷 사진 · 압축 · 촬영 각도 · 라벨 기준 — 아직 하나로 못 좁힘) 로 EAR 까지 무너지는 데이터"**. 우리 앱은 폰 앞카메라로 얼굴이 가까워 **팀 사진 눈 폭 가운데값 83px** (548 은 46px, CEW 는 16px) → CEW 같은 작은 눈은 앱에서 드묾 → 판단 데이터에서 뺌
- 새 모델이 CEW 에서 기존보다 나쁜 것 (33.0% → 67.9%) 은 548 미세조정으로 **처음 보는 종류의 사진에서 과한 확신** — 발표 한계로 적음

## 11. 정규화 비교 + 사람 단위 k-fold (11/04 피드백)

> ⚠️ **보류 (12번 실험 뒤)** — 11/04 팀장 결정: 정규화 대신 **세로 누르기(vscale)** 로 먼저 실험 → **12번**. 서버 모델 결정이 먼저 (팀 확정 규칙 12)
>
> ✅ **11/05 실행 (12번 뒤, 규칙 12)** → 판정 `z` (지금 정규화 유지) — 결과 11-7

- **왜 (피드백):** ① **정규화** — 적외선 / 컬러, 외국인 / 한국인 사진의 차이를 입력 단계에서 줄이기 ② **k-fold** — val 한 번(49명)이 아니라 사람을 바꿔 가며 여러 번 재서 평균 · 흔들림으로 비교 ③ **CEW 는 정규화 · k-fold 뒤에** 다시 시험
- **우리 데이터 = 2 × 2** (CEW 칸만 학습 데이터가 없음 → 마지막 시험지)

| | 외국인 | 한국인 |
|---|---|---|
| **적외선** | MRL (학습) | AI-Hub 173 (학습) |
| **컬러** | **CEW (마지막 시험만)** | AI-Hub 548 (학습) |

- **지금 정규화** (`prep_eye`): 흑백 → 64×64 → **사진마다 평균 0 · 표준편차 1** — 전체 밝기 · 대비는 이미 맞춰짐. 남는 차이 = 부분 대비(적외선은 눈동자가 밝게 · 컬러는 어둡게), 밝기 분포 모양, 해상도(CEW 눈 15px vs 548 46px, 10-7)
- **비교하는 정규화 4가지** (`prep_eye` 앞에 한 단계 더, 64×64 흑백에서):

| 이름 | 하는 일 | 노리는 차이 |
|---|---|---|
| `z` (지금) | 추가 없음 | 기준 |
| `clahe` | CLAHE — 4×4 칸마다 대비 맞추기 (clip 2.0) | 적외선 ↔ 컬러 부분 대비 · 그림자 |
| `hist` | 히스토그램 평활화 — 밝기 분포를 고르게 | 밝기 분포 모양 |
| `clahe_res` | CLAHE + 해상도 맞춤 (20px 로 줄였다 64 로) | + CEW 처럼 작은 눈 (해상도) |

- **학습 방식 (모든 후보 똑같이):** MRL + 173 + 548 을 **처음부터 같이** (ImageNet 가중치) — 데이터마다 1/3 씩, 그 안에서 감음 · 뜸 반반 뽑기. 에포크 4번 고정 · 에포크당 60,000장 (조기 종료 없음 → 시험 fold 로 아무것도 안 고름)
  - 지금 서버 모델(MRL → 173 → 548 이어서 학습)과 순서가 다름 → 정규화끼리 **같은 조건에서 비교**하는 것이 목적
- **k-fold (사람 단위 5-fold):** 데이터마다 사람을 5묶음으로 (장수 비슷하게) → 4묶음으로 학습, 남은 1묶음으로 시험 × 5번. 한 사람 사진은 항상 한 묶음에만
  - **원래 시험지는 계속 뺌:** MRL test (1-11) · 173 test (3-2) · 548 Validation 151명 (PC) · 팀 사진 · 71421 · CEW → k-fold 에 안 들어감
- **판정 규칙 (돌리기 전에 정함):** 지표 = fold 시험의 **데이터별 균형 정확도 (기준 0.4)** → 세 데이터 평균 · **가장 나쁜 데이터** (도메인 차이에 강한가)
  - 후보가 `z` 보다 **가장 나쁜 데이터 균형 정확도가 5-fold 평균으로 높고, 그 차이가 fold 간 표준편차보다 클 때만** 채택. 아니면 `z` 유지 (서버 코드 안 바꿈)
  - 채택되면 11-6 에서 전체로 다시 학습 → PC 에서 548 Validation 151명 · 팀 사진 · **CEW 한 번** (서버 판정 그대로)
- **실행 순서:** 1-1 → 1-2 → 1-3 → 2-1 → 2-2 → 2-3 → 2-4 (MRL) → 3-1 → 3-2 (173) → 7-1 (548) → 11-1 ~ 11-5. 끊기면 11-4 다시 (끝난 fold 는 건너뜀)
  - 시간 (T4): 학습 한 번 약 3분 × 4가지 × 5 fold ≈ **1시간 ~ 1시간 반**. 빨리 보려면 11-4 의 `K_RUN = 2`

### 11-1. 세 데이터를 64×64 흑백으로 모으기 + 사람 단위 5-fold

In [ ]:
assert all(k in globals() for k in ('df', 'IMGS', 'CROPS', 'meta', 'C548', 'm548')), '2-2 · 2-3 (MRL), 3-1 · 3-2 (173), 7-1 (548) 을 먼저 실행'
K = 5
DOM = ['MRL', 'AIHub173', 'AIHub548']
DOM_KO = {'MRL': '적외선 · 외국인', 'AIHub173': '적외선 · 한국인', 'AIHub548': '컬러 · 한국인'}


def to64(img):   # prep_eye 와 같은 흑백 · 크기 맞추기 (정규화 전 단계까지, uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY if img.shape[2] == 4 else cv2.COLOR_BGR2GRAY)
    return cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA if min(img.shape) >= SIZE else cv2.INTER_LINEAR)


# 원래 시험지는 뺌: MRL test · 173 test (548 은 PC 에서 Validation 151명을 이미 뺌)
src = {'MRL': (np.flatnonzero(df['split'].to_numpy() != 'test'), lambda j: IMGS[j], Y, df['subject'].to_numpy()),
       'AIHub173': (np.flatnonzero(meta['split'].to_numpy() != 'test'), lambda j: CROPS[j], AY, meta['driver'].astype(str).to_numpy()),
       'AIHub548': (np.arange(len(m548)), lambda j: C548[j], Y548, m548['driver'].astype(str).to_numpy())}
X11, Y11, D11, P11 = [], [], [], []
for d, (idx, get, y, who) in src.items():
    X11.append(np.stack([to64(get(j)) for j in idx]))
    Y11.append(y[idx].astype(np.float32)); D11 += [d] * len(idx); P11 += [f'{d}:{p}' for p in who[idx]]
X11, Y11, D11, P11 = np.concatenate(X11), np.concatenate(Y11), np.array(D11), np.array(P11)

# 사람 단위 5-fold — 데이터마다 사람을 장수 많은 순서로 가장 가벼운 묶음에 (장수 · 사람 수 비슷하게)
FOLD = np.zeros(len(Y11), np.int8)
for d in DOM:
    m = D11 == d
    ppl, cnt = np.unique(P11[m], return_counts=True)
    load = np.zeros(K)
    for p, c in sorted(zip(ppl, cnt), key=lambda t: -t[1]):
        f = int(load.argmin()); load[f] += c
        FOLD[m & (P11 == p)] = f
assert all(len(set(FOLD[P11 == p])) == 1 for p in np.unique(P11)), '한 사람이 여러 fold 에'
print(f'눈 {len(Y11)}개 · 메모리 {X11.nbytes / 1e9:.2f}GB')
display(pd.DataFrame({'데이터': D11, 'fold': FOLD, '사람': P11, 'y': Y11}).groupby(['데이터', 'fold'])
        .agg(사람=('사람', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3).unstack('fold'))

### 11-2. 정규화 함수 + 눈으로 보기 + 데이터끼리 얼마나 비슷해지나

- 그림: 데이터마다 뜬 눈 · 감은 눈을 정규화 4가지로
- 숫자: 정규화 뒤 **데이터별 평균 밝기 분포(32칸)** 사이의 거리 (Jensen-Shannon, 0 = 같음) — 작아질수록 '입력이 데이터끼리 비슷해짐'. 학습 전에 보는 참고 숫자 (판정은 11-5 의 k-fold)

In [ ]:
NORMS = ['z', 'clahe', 'hist', 'clahe_res']
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))


def norm_img(x, how):
    # 64×64 uint8 흑백 → 같은 모양 uint8. 뒤에 prep_eye (사진마다 평균 0 · 표준편차 1) 가 붙음
    if how == 'z':
        return x
    if how == 'clahe':
        return _CLAHE.apply(x)
    if how == 'hist':
        return cv2.equalizeHist(x)
    if how == 'clahe_res':
        small = cv2.resize(x, (20, 20), interpolation=cv2.INTER_AREA)
        return _CLAHE.apply(cv2.resize(small, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR))
    raise ValueError(how)


rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(DOM) * 2, len(NORMS), figsize=(2.2 * len(NORMS), 2.2 * len(DOM) * 2))
for r, (d, yv) in enumerate([(d, yv) for d in DOM for yv in (0, 1)]):
    j = rng.choice(np.flatnonzero((D11 == d) & (Y11 == yv)))
    for c, how in enumerate(NORMS):
        axes[r, c].imshow(prep_eye(norm_img(X11[j], how), SIZE), cmap='gray')
        axes[r, c].set_title(f"{d} {'감음' if yv else '뜸'} · {how}", fontsize=7); axes[r, c].axis('off')
plt.tight_layout(); plt.show()


def js(p, q):
    m = (p + q) / 2
    kl = lambda a, b: float(np.sum(a * np.log((a + 1e-12) / (b + 1e-12))))
    return (kl(p, m) + kl(q, m)) / 2


rows = []
for how in NORMS:
    hist = {}
    for d in DOM:
        pick = rng.choice(np.flatnonzero(D11 == d), min(3000, int((D11 == d).sum())), replace=False)
        v = np.concatenate([prep_eye(norm_img(X11[j], how), SIZE).ravel() for j in pick])
        h = np.histogram(np.clip(v, -3, 3), bins=32, range=(-3, 3))[0].astype(float)
        hist[d] = h / h.sum()
    rows.append({'정규화': how, **{f'{a} ↔ {b}': round(js(hist[a], hist[b]), 4) for i, a in enumerate(DOM) for b in DOM[i + 1:]}})
display(pd.DataFrame(rows).set_index('정규화'))

### 11-3. 학습 · 시험 함수 (fold 하나 × 정규화 하나)

- 증강(2-4 `augment`) → 정규화 → `prep_eye` 순서 (서버에서도 사진 → 정규화 → `prep_eye`)
- 시험 = 그 fold 사람들의 눈 (증강 없음). 데이터마다 **균형 정확도 · 감은 눈 → 감음 · 뜬 눈 → 감음 (오판)** (기준 0.4) + 뜬 눈 중 CNN 0.3 이상 (서버가 '확실히 뜸' 으로 못 보는 비율, 10번과 같은 뜻)

In [ ]:
EPOCHS11, PER_EPOCH11 = 4, 60000
RES11 = f'{TRAIN_DIR}/kfold11.json'   # fold · 정규화마다 결과 (끊겨도 이어서)
RESULTS11 = json.load(open(RES11, encoding='utf-8')) if os.path.exists(RES11) else {}


class Ds11(torch.utils.data.Dataset):
    def __init__(self, idx, how, train):
        self.idx, self.how, self.train = idx, how, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = X11[j]
        if self.train:
            x = augment(x, np.random.default_rng(int(torch.randint(0, 2**31, ()).item())))
        return torch.from_numpy(prep_eye(norm_img(x, self.how), SIZE))[None], torch.tensor(Y11[j])


def sampler11(idx):
    # 데이터마다 1/3, 그 안에서 감음 · 뜸 반반
    w = np.zeros(len(idx))
    for d in DOM:
        for yv in (0, 1):
            m = (D11[idx] == d) & (Y11[idx] == yv)
            if m.any():
                w[m] = 1 / (len(DOM) * 2 * m.sum())
    return torch.utils.data.WeightedRandomSampler(w, PER_EPOCH11, replacement=True)


@torch.no_grad()
def predict11(m, idx, how, bs=512):
    m.eval(); out = []
    dl = torch.utils.data.DataLoader(Ds11(idx, how, False), batch_size=bs, num_workers=NUM_WORKERS)
    for x, _ in dl:
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(m(x.to(device)).float()).cpu())
    return torch.cat(out).numpy()


def score11(p, y, th=0.4):
    pr = p >= th
    rc, ro = float(pr[y == 1].mean()), float((~pr[y == 0]).mean())
    return dict(bal=round((rc + ro) / 2, 4), closed_recall=round(rc, 4), open_fa=round(1 - ro, 4),
                open_not_sure=round(float((p[y == 0] >= th - 0.1).mean()), 4), n=int(len(y)))


def run11(how, k):
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    tr, te = np.flatnonzero(FOLD != k), np.flatnonzero(FOLD == k)
    m = EyeCNN(pretrained=True).to(device)
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS11)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    dl = torch.utils.data.DataLoader(Ds11(tr, how, True), batch_size=BATCH, sampler=sampler11(tr), num_workers=NUM_WORKERS,
                                     pin_memory=(device == 'cuda'))
    t0 = time.time()
    for ep in range(EPOCHS11):
        m.train()
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
        s.step()
    p = predict11(m, te, how)
    res = {d: score11(p[D11[te] == d], Y11[te][D11[te] == d]) for d in DOM}
    res['sec'] = round(time.time() - t0)
    return res


loss_fn = torch.nn.BCEWithLogitsLoss()
print('끝난 것:', sorted(RESULTS11))

### 11-4. 실행 — 정규화 4가지 × 5 fold (끊기면 이 셀 다시)

In [ ]:
K_RUN = K   # 빨리 보려면 2 (fold 0 · 1 만)
for k in range(K_RUN):
    for how in NORMS:
        key = f'{how}|{k}'
        if key in RESULTS11:
            continue
        RESULTS11[key] = run11(how, k)
        json.dump(RESULTS11, open(RES11, 'w', encoding='utf-8'), indent=1)
        r = RESULTS11[key]
        print(f"{how:<10} fold {k}: " + ' · '.join(f"{d} {r[d]['bal']:.3f}" for d in DOM) + f" ({r['sec']}초)")

### 11-5. 결과 — 평균 ± fold 간 표준편차 · 가장 나쁜 데이터 · 판정

In [ ]:
rows = []
for key, r in RESULTS11.items():
    how, k = key.split('|')
    for d in DOM:
        rows.append({'정규화': how, 'fold': int(k), '데이터': d, **{c: r[d][c] for c in ('bal', 'closed_recall', 'open_fa', 'open_not_sure')}})
R11 = pd.DataFrame(rows)
R11['정규화'] = pd.Categorical(R11['정규화'], NORMS, ordered=True)

ms = lambda s: f'{s.mean():.3f} ± {s.std(ddof=1) if len(s) > 1 else 0:.3f}'
display(R11.pivot_table(index='정규화', columns='데이터', values='bal', aggfunc=ms, observed=True)
        .rename(columns=lambda d: f'{d} ({DOM_KO[d]})'))
for c, t in (('closed_recall', '감은 눈 → 감음'), ('open_fa', '뜬 눈 → 감음 (오판)'), ('open_not_sure', '뜬 눈 CNN ≥ 0.3 (확실히 뜸 못 함)')):
    print(t); display(R11.pivot_table(index='정규화', columns='데이터', values=c, aggfunc=ms, observed=True))

# fold 마다 세 데이터 평균 · 가장 나쁜 데이터 → 그 값의 평균 ± 표준편차
per = R11.groupby(['정규화', 'fold'], observed=True)['bal'].agg(평균='mean', 가장나쁜='min').reset_index()
summ = per.groupby('정규화', observed=True).agg(평균=('평균', 'mean'), 평균_sd=('평균', 'std'), 가장나쁜=('가장나쁜', 'mean'),
                                                가장나쁜_sd=('가장나쁜', 'std'), fold=('fold', 'nunique')).round(4)
display(summ)

base = summ.loc['z']
pick = 'z'
for how in NORMS[1:]:
    if how in summ.index:
        diff = summ.loc[how, '가장나쁜'] - base['가장나쁜']
        # 같은 fold 끼리 짝지어 차이 → 그 차이의 fold 간 표준편차보다 커야 (11 시작에 정한 규칙)
        d = per[per['정규화'] == how].set_index('fold')['가장나쁜'] - per[per['정규화'] == 'z'].set_index('fold')['가장나쁜']
        ok = diff > 0 and len(d.dropna()) > 1 and diff > d.std()
        print(f"{how:<10} 가장 나쁜 데이터 {base['가장나쁜']:.3f} → {summ.loc[how, '가장나쁜']:.3f} ({diff:+.3f}, fold 차이 표준편차 {d.std():.3f}) → {'채택 후보' if ok else '아님'}")
        if ok and summ.loc[how, '가장나쁜'] > summ.loc[pick, '가장나쁜']:
            pick = how
print('\n판정:', pick, '(z 면 지금 정규화 유지 — 서버 코드 안 바꿈)')

fig, ax = plt.subplots(figsize=(7, 3.6))
for i, d in enumerate(DOM):
    g = R11[R11['데이터'] == d].groupby('정규화', observed=True)['bal']
    ax.errorbar(np.arange(len(NORMS)) + (i - 1) * 0.12, g.mean().reindex(NORMS), yerr=g.std().reindex(NORMS), fmt='o', capsize=3, label=f'{d} ({DOM_KO[d]})')
ax.set_xticks(range(len(NORMS))); ax.set_xticklabels(NORMS); ax.set_ylabel('균형 정확도 (기준 0.4)')
ax.set_title(f'정규화별 · 데이터별 — 사람 단위 {K}-fold 평균 ± 표준편차'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/kfold11.png', dpi=110); plt.show()

### 11-6. 채택된 정규화로 전체 다시 학습 → PC 시험 (CEW 포함 한 번)

- 11-5 판정이 `z` 가 아니면 실행. fold 없이 세 데이터 전부 (원래 시험지는 여전히 뺌) 로 같은 방식 학습 → `eye_cnn11_<정규화>.pt` (`norm` 칸에 이름)
- 그 뒤 PC: 서버 `prep_eye` 앞에 같은 정규화가 들어가야 함 → 서버 코드 바꾸기 전에 PC 시험 (548 Validation 151명 · 팀 사진 · 71421 · **CEW 한 번**) 으로 지금 모델(`eye_cnn_548.pt`)과 비교
- `z` 로 판정되면: 정규화는 지금 그대로가 맞다는 근거 → k-fold 숫자(11-5)를 발표에 쓰고, CEW 는 지금 모델 그대로 (10-7)

In [ ]:
NORM_PICK = pick   # 11-5 판정 (손으로 바꾸지 않기)
if NORM_PICK == 'z':
    print('z 유지 — 다시 학습 안 함')
else:
    torch.manual_seed(SEED)
    allidx = np.arange(len(Y11))
    m = EyeCNN(pretrained=True).to(device)
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS11)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    dl = torch.utils.data.DataLoader(Ds11(allidx, NORM_PICK, True), batch_size=BATCH, sampler=sampler11(allidx), num_workers=NUM_WORKERS,
                                     pin_memory=(device == 'cuda'))
    for ep in range(EPOCHS11):
        m.train()
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
        s.step(); print('에포크', ep + 1)
    PT11 = f'{TRAIN_DIR}/eye_cnn11_{NORM_PICK}.pt'
    torch.save({'state_dict': m.net.state_dict(), 'arch': 'resnet18', 'input_size': SIZE, 'norm': NORM_PICK,
                'made': str(datetime.date.today()), 'data': 'MRL + AI-Hub 173 + AI-Hub 548 (원래 시험지 뺌), 데이터마다 1/3',
                'kfold11': summ.reset_index().to_dict('records')}, PT11)
    print('저장:', PT11, '→ PC ml_models/ 에 (GitHub X)')

### 11-7. 결과 (11/05 Colab T4 — 정규화 4가지 × 사람 단위 5-fold, 11-5 판정 그대로)

**판정: `z` (지금 정규화 유지 — 서버 코드 안 바꿈, 11-6 안 돌림)**

균형 정확도 (기준 0.4, 5-fold 평균 ± fold 간 표준편차)

| 정규화 | AIHub173 (적외선 · 한국인) | AIHub548 (컬러 · 한국인) | MRL (적외선 · 외국인) | 세 데이터 평균 | 가장 나쁜 데이터 |
|---|---|---|---|---|---|
| **`z` (지금)** | **0.942 ± 0.007** | **0.987 ± 0.008** | 0.977 ± 0.011 | **0.9687 ± 0.0057** | **0.9422 ± 0.0073** |
| `clahe` | 0.940 ± 0.009 | 0.987 ± 0.009 | 0.978 ± 0.008 | 0.9681 ± 0.0054 | 0.9398 ± 0.0087 |
| `hist` | 0.938 ± 0.007 | 0.985 ± 0.009 | 0.976 ± 0.011 | 0.9666 ± 0.0061 | 0.9383 ± 0.0067 |
| `clahe_res` | 0.936 ± 0.007 | 0.986 ± 0.008 | 0.972 ± 0.014 | 0.9646 ± 0.0068 | 0.9364 ± 0.0072 |

- 판정 규칙 (가장 나쁜 데이터가 `z` 보다 높고, 그 차이가 같은 fold 끼리 짝지은 차이의 표준편차보다 클 때만 채택): `clahe` −0.002 (짝 차이 sd 0.002) · `hist` −0.004 (0.002) · `clahe_res` −0.006 (0.001) → **셋 다 아님**
- 가장 나쁜 데이터는 모든 정규화에서 **AIHub173 (적외선 · 한국인)**. 뜬 눈 오판도 173 이 가장 많음 (`z` 7.9% · 548 1.2% · MRL 3.0%)
- 같은 fold 끼리 보면 다른 정규화가 **조금씩 그러나 꾸준히 낮음** (짝 차이 sd 가 차이보다 작음) → "입력 단계에서 데이터 차이를 줄이는" 정규화는 이 세 데이터에서 도움이 안 됨. 사진마다 평균 0 · 표준편차 1 (`prep_eye`) 로 충분
- 감은 눈 → 감음 · 뜬 눈 CNN ≥ 0.3 도 같은 순서 (`z` 가 가장 좋거나 같음)
- fold 간 표준편차가 데이터마다 0.007 ~ 0.011 → **설정끼리 1%p 안쪽 차이는 사람을 바꾸면 뒤집힐 수 있는 크기** (12번 · 13번 비교를 읽을 때 기준)
- CEW (11번 시작 때 계획한 '정규화 뒤 CEW 한 번') 는 11/03 결정대로 판단에 안 씀 → 정규화를 안 바꾸므로 다시 재지 않음

## 12. 세로 누르기 (`vscale`) 설정 5개 × MRL → 173 → 548 다시 학습 (11/04 팀장 규칙)

- **팀장 규칙 (11/04):** 정규화(11번) 대신 **눈 사진 세로를 k배로 누르고 원래 크기로 채우는 방식**으로 실험 → 11번은 보류 (이 실험 뒤)
- **`vscale(img, k)`** (`eye_preprocess.py`, 서버 추론은 안 부름): 세로만 k배로 누르고 (가로 그대로) 위 · 아래 빈칸은 **원래 사진의 위쪽 · 아래쪽 줄을 그대로 복사**해 붙임 → 크게 뜬 눈이 조금 가늘게 뜬 눈처럼 보임
- **어디에:** **MRL train 만**, 사진을 불러올 때마다 `k = uniform(K_MIN, K_MAX)` 새로 뽑음, 라벨과 상관없이 전부, `prep_eye` 보다 먼저 (순서: `vscale` → 증강 → `prep_eye`). MRL val · test, 173, 548 에는 안 넣음
- **설정 5개 (K_MIN, K_MAX):** (1.0, 1.0) **기준** · (0.85, 1.0) · (0.75, 1.0) · (0.75, 0.9) · (0.8, 0.8)
- **학습:** 설정마다 **MRL → 173 → 548 전체를 처음부터** (기준도 같은 시드로 다시 — 지금 서버 `eye_cnn_548.pt` 는 예전 324명 데이터라 기준으로 안 씀)
  - 단계마다 방식은 2-5 · 3-4 · 7-3 과 같음: MRL (ImageNet, `LR`, 최대 `EPOCHS`, 조기 종료 `PATIENCE`) → 173 (`LR/10`, 5, 2, 감음 · 뜸 반반) → 548 (`LR/10`, 5, 2, 반반, val = 548 학습용 val 115명의 눈)
  - **이어받기 파일은 설정 · 시드 · 단계마다 따로:** `v12_<설정>_s<시드>_<단계>_last.pt` → 다른 설정 학습을 이어 가지 않음. 끝난 설정은 `eye_cnn12_<설정>_s<시드>.pt` 가 있으면 건너뜀
  - **W&B:** 설정 · 시드마다 run 하나 (`b_v12_<설정>_s<시드>`, group `vscale12`), 단계별 에포크 기록 + 참고 숫자. 사진은 안 올림
- **고르기 = 548 학습용 val 115명 (PC, 서버 판정 그대로)** — 눈 조각에는 EAR 이 없어서 얼굴 사진으로 다시: `aihub548.py eval --set trainval --models ...` → `pick`
  - 서버 `_normalize` 와 같은 취소 규칙 (감음 = CNN ≥ t, 취소 = CNN < t − 0.1 그리고 EAR ≥ 0.26). t 를 0.11 ~ 0.95 훑어 **뜬 눈 오경보 ≤ 3% 인 가장 낮은 t** → 그때 감은 눈 취소 · 반쯤 감은 눈 취소
  - **설정 고르기 · 서버 반영 · 시드 재확인:** 아래 **12-0 확정 규칙** 그대로
- **151명 (Validation) 은 마지막 한 번:** 고른 설정과 기준을 **각자 val 에서 고른 t** 로 한 번씩 (`eval --set val` → `final`). 결과를 보고 다시 고르지 않음. 팀 사진은 고르는 기준이 아님 (시연 확인용으로 따로)
- **서버 반영 (팀장):** 고른 설정의 시드 0 모델 파일로 교체 + `config.CNN_CLOSED_PROB` = val 에서 고른 t (후보가 하나도 없으면 변경 없음 — 12-0)
- **보고:** 151명 세 숫자 (뜬 눈 오경보 · 감은 눈 취소 · 반쯤 감은 눈 취소) 를 건수와 함께. MRL test 는 참고 (떨어져도 괜찮음)
- **실행 순서 (Colab T4):** 1-1 → 1-2 → 1-3 → 2-1 → 2-1b (`USE_WANDB = True`) → 2-2 → 2-3 → 2-4 → 3-1 → 3-2 → 7-1 → **12-1 → 12-2 → 12-3**
  - `eye_preprocess.py` 는 **vscale 이 들어간 새 파일**로 `/content` 에 다시 올리기 (2-1 전에)
  - 시간: 설정 하나 약 20 ~ 30분 × 5 ≈ **2 ~ 2.5시간**. 끊기면 2-1 ~ 7-1 다시 → 12-1 · 12-2 → 12-3 (끝난 단계 · 설정은 건너뜀)

### 12-0. 확정 규칙 (11/04 팀 결정 — 결과를 보기 전에 정함)

> 팀장이 정함 (11/04). **새 모델 5개의 val 결과는 이 규칙을 커밋한 뒤에만 봄**. PC `aihub548.py pick` 이 이 규칙 그대로

1. **vscale:** 눌린 사진을 가운데에 두고, 빈칸은 원래 사진의 맨 위 · 맨 아래 부분을 복사
2. **후보** = 548 학습용 val 에서 **뜬 눈 오경보 3% 이하인 t 가 있고**, 그 t 에서 **감은 눈 취소 0건**
3. 후보 중 **반쯤 감은 눈 취소율이 가장 낮은 설정**. 기준보다 **2%p 넘게 낮을 때만 보정 채택**
4. 기준이 이기면 **새 기준 모델 + 그 t 로 교체** (763명으로 새로 학습했으므로)
5. **후보가 하나도 없을 때만 서버 변경 없음** (`eye_cnn_548.pt` + 0.4 유지), 결과만 보고. **기준이 탈락하고 다른 후보가 있으면 2%p 비교 없이 후보 중 반쯤 취소율이 가장 낮은 설정** (이때도 시드 재확인은 상위 2개 + 기준). 151명 보고에서 기준이 t 를 못 찾았으면 "해당 없음" (감은 눈 취소로 탈락했으면 t 가 있으니 그대로 잼) — *11/04 수정: 처음 규칙 '기준 탈락이면 변경 없음' 은 통과한 설정까지 버리게 되어 고침 (팀장)*
6. t: 0.11 ~ 0.95, 0.01 간격 (0.10 이하는 취소가 아예 안 생김) / 분모 = 얼굴 찾은 사진, 판정 불가는 따로
7. **시드 재확인:** 상위 2개 + 기준을 시드 1로 한 번 더, 시드마다 t 다시 찾기, **두 시드 모두 2번 통과**, 비교는 반쯤 취소율 평균
8. 서버에 넣는 건 **뽑힌 설정의 시드 0 모델 + 그 t**
9. 동점이면 **기준에 가까운 설정**
10. **151명은 최종 모델 · 기준 모델 각 1회만**, 결과가 나빠도 다시 고르지 않고 그대로 보고
11. 팀 사진은 고르는 데 안 씀, 따로 보고 (시연에 문제가 될 만큼 나쁘면 교체 전에 강사님께 보고)
12. 노트북 11번 (정규화 · k-fold) 은 **보류 (이 실험 뒤)**

### 12-1. 설정 · `vscale` 확인 (눈으로 — W&B 에는 사진 안 올림)

In [ ]:
import random
importlib.reload(eye_preprocess)
from eye_preprocess import vscale, prep_eye
need = ('IMGS', 'IDX', 'PREP', 'Y', 'CROPS', 'AIDX', 'APREP', 'AY', 'C548', 'I548', 'P548', 'Y548', 'augment', 'EyeCNN', 'USE_WANDB')
assert all(k in globals() for k in need), f"먼저 실행: {[k for k in need if k not in globals()]} (2-1 ~ 2-4 · 2-1b · 3-1 · 3-2 · 7-1)"
assert 'test' in PREP, '2-3 의 PREP (MRL val · test)'

CONFIGS = [(1.0, 1.0), (0.85, 1.0), (0.75, 1.0), (0.75, 0.9), (0.8, 0.8)]   # (K_MIN, K_MAX), 첫 번째 = 기준
BASE12 = (1.0, 1.0)


def tag12(c):
    return f'k{round(c[0] * 100):03d}_{round(c[1] * 100):03d}'   # (0.85, 1.0) → k085_100 (PC aihub548.py pick 이 이 이름을 읽음)


print('설정:', [tag12(c) for c in CONFIGS])
rng = np.random.default_rng(0)
pick = [rng.choice(IDX['train'][Y[IDX['train']] == yv]) for yv in (0, 0, 0, 1, 1)]
ks = sorted({k for c in CONFIGS for k in c}, reverse=True)
fig, axes = plt.subplots(len(pick), len(ks), figsize=(1.6 * len(ks), 1.7 * len(pick)))
for r, j in enumerate(pick):
    for c, k in enumerate(ks):
        axes[r, c].imshow(prep_eye(vscale(IMGS[j], k), SIZE), cmap='gray')
        axes[r, c].set_title(f"{'감음' if Y[j] else '뜸'} k={k}", fontsize=8); axes[r, c].axis('off')
plt.suptitle('vscale — 세로만 k배 (위 · 아래는 원래 사진 줄 복사), prep_eye 뒤 모습', fontsize=10)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/vscale12_check.png', dpi=90); plt.show()

### 12-2. 데이터셋 · 단계 학습 함수 · 설정 하나 끝까지 (`run12`)

- MRL train 만 `vscale` → 증강(2-4 `augment`) → `prep_eye`. 173 · 548 train 은 증강 → `prep_eye` (지금과 같음). val · test 는 증강 없음 (`PREP` · `APREP` · `P548`)
- 단계마다 val 손실이 가장 낮은 가중치를 다음 단계 시작점으로 (2-5 · 3-4 · 7-3 과 같음)
- 저장: `eye_cnn12_<설정>_s<시드>.pt` (서버 형식 그대로 — `state_dict` · `arch` · `input_size`, 추가로 `vscale` · `seed`) → PC `ml_models/` 에 받아 시험

In [ ]:
class Mrl12(torch.utils.data.Dataset):          # MRL train — 불러올 때마다 k 새로 뽑아 vscale
    def __init__(self, idx, cfg):
        self.idx, self.cfg = idx, cfg
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        r = np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))
        img = IMGS[j]
        if self.cfg != (1.0, 1.0):
            img = vscale(img, r.uniform(*self.cfg))
        return torch.from_numpy(prep_eye(augment(img, r), SIZE))[None], torch.tensor(Y[j])


class Arr12(torch.utils.data.Dataset):          # 173 · 548 train — 지금과 같음 (vscale 없음)
    def __init__(self, src, ys, idx):
        self.src, self.ys, self.idx = src, ys, idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        r = np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))
        return torch.from_numpy(prep_eye(augment(self.src[j], r), SIZE))[None], torch.tensor(self.ys[j], dtype=torch.float32)


def half_w(y):                                  # 감음 · 뜸 반반 뽑기 무게 (3-3 · 7-2 와 같음)
    return np.where(y == 1, 0.5 / y.mean(), 0.5 / (1 - y.mean()))


@torch.no_grad()
def pred12(m, X, bs=512):
    m.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(m(torch.from_numpy(X[i:i + bs])[:, None].to(device)).float()).cpu())
    return torch.cat(out).numpy()


def bce12(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def score12(p, y, th=0.4):
    pr = p >= th
    rc, ro = float(pr[y == 1].mean()), float((~pr[y == 0]).mean())
    return dict(bal=round((rc + ro) / 2, 4), closed_recall=round(rc, 4), open_recall=round(ro, 4), n=int(len(y)))


def fit12(name, stage, init_state, lr, epochs, patience, ds, weights, vX, vy, run, seed):
    m = EyeCNN(pretrained=init_state is None).to(device)
    if init_state is not None:
        m.net.load_state_dict(init_state)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    g = torch.Generator().manual_seed(seed * 100 + len(stage))
    sampler = (torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True, generator=g) if weights is not None
               else torch.utils.data.RandomSampler(ds, generator=g))
    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
    last, best_p = f'{TRAIN_DIR}/v12_{name}_{stage}_last.pt', f'{TRAIN_DIR}/v12_{name}_{stage}_best.pt'   # 설정 · 시드 · 단계마다 따로
    start, h, best_l, bad_n = 0, [], float('inf'), 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, h, best_l, bad_n = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
        print(f'{name} {stage}: {start + 1}번째 에포크부터 이어서')
    for ep in range(start, epochs):
        if bad_n >= patience:
            print(f'{name} {stage}: 조기 종료'); break
        t0 = time.time(); m.train(); tl = n = 0
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        p = pred12(m, vX); vl = bce12(p, vy)
        row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), **{f'val_{k}': v for k, v in score12(p, vy, 0.5).items() if k != 'n'},
                   sec=round(time.time() - t0))
        h.append(row); print(name, stage, row)
        if run:
            wandb.log({f'{stage}/{k}': v for k, v in row.items()})
        if vl < best_l:
            best_l, bad_n = vl, 0; torch.save(m.net.state_dict(), best_p)
        else:
            bad_n += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=h, best=best_l, bad=bad_n), last)
    return torch.load(best_p, map_location='cpu', weights_only=True), h


def run12(cfg, seed):
    name = f'{tag12(cfg)}_s{seed}'
    out = f'{TRAIN_DIR}/eye_cnn12_{name}.pt'
    if os.path.exists(out):
        print('끝난 설정 (건너뜀):', out); return
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    run = None
    if USE_WANDB:
        run = wandb.init(project='eye-cnn', name=f'b_v12_{name}', id=f'b_v12_{name}', group='vscale12', resume='allow',
                         config=dict(model='resnet18', input=SIZE, batch=BATCH, seed=seed, k_min=cfg[0], k_max=cfg[1], lr=LR,
                                     chain='MRL → AI-Hub 173 → AI-Hub 548', vscale='MRL train 만, prep_eye 전',
                                     mrl=dict(epochs=EPOCHS, patience=PATIENCE), ft=dict(lr=LR / 10, epochs=5, patience=2)),
                         settings=wandb.Settings(console='off'))
    s1, h1 = fit12(name, 'mrl', None, LR, EPOCHS, PATIENCE, Mrl12(IDX['train'], cfg), None, PREP['val'], Y[IDX['val']], run, seed)
    s2, h2 = fit12(name, 'a173', s1, LR / 10, 5, 2, Arr12(CROPS, AY, AIDX['train']), half_w(AY[AIDX['train']]),
                   APREP['val'], AY[AIDX['val']], run, seed)
    s3, h3 = fit12(name, 'a548', s2, LR / 10, 5, 2, Arr12(C548, Y548, I548['train']), half_w(Y548[I548['train']]),
                   P548['val'], Y548[I548['val']], run, seed)
    m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(s3)
    ref = {'MRL_test': score12(pred12(m, PREP['test']), Y[IDX['test']]),           # 참고 (떨어져도 괜찮음)
           'AIHub173_test': score12(pred12(m, APREP['test']), AY[AIDX['test']]),
           'AIHub548_val_eye': score12(pred12(m, P548['val']), Y548[I548['val']])}  # 눈 조각 기준 (고르기는 PC 서버 판정)
    print(name, ref)
    torch.save(dict(state_dict=s3, arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE, output='logit 1개, sigmoid = 감음 확률',
                    prep='eye_preprocess.prep_eye', vscale=dict(k_min=cfg[0], k_max=cfg[1], where='MRL train 만, prep_eye 전'), seed=seed,
                    data='MRL → AI-Hub 173 → AI-Hub 548 (763명, Validation 151명 뺌, 반쯤 감음 뺌)', test=ref,
                    hist={'mrl': h1, 'a173': h2, 'a548': h3}, made=str(datetime.datetime.now())[:16], torch=str(torch.__version__)), out)
    torch.load(out, map_location='cpu', weights_only=True)   # 서버와 같은 방식으로 읽히나
    if run:
        wandb.summary.update({f'{k}_{a}': float(v) for k, d in ref.items() for a, v in d.items()}); wandb.finish()
    print('저장:', out)


loss_fn = torch.nn.BCEWithLogitsLoss()
print('MRL train', len(IDX['train']), '/ 173 train', len(AIDX['train']), '/ 548 train', len(I548['train']), '· val', len(I548['val']))

### 12-3. 실행 — 설정 5개 × 시드 0 (끊기면 이 셀 다시)

In [ ]:
for cfg in CONFIGS:
    run12(cfg, SEED)
print('끝 → 드라이브 eye_cnn/eye_cnn12_*.pt 를 PC sleep_detector_api/ml_models/ 로 (GitHub X) → 12-4')

### 12-4. PC — 548 학습용 val 115명으로 t · 설정 고르기

```
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval --set trainval --models eye_cnn12_k100_100_s0.pt eye_cnn12_k085_100_s0.pt eye_cnn12_k075_100_s0.pt eye_cnn12_k075_090_s0.pt eye_cnn12_k080_080_s0.pt
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py pick
```

- `eval --set trainval`: 학습용 묶음 3개 (TS_G1_1 · TS_G2_1 · TS_U_1) 중 `people_split.json` 의 val 115명 얼굴 사진 → 서버 판정 그대로 (MediaPipe · `crop_eye` · EAR · CNN). 사진은 한 번 읽고 모델마다 CSV
- `pick`: 모델마다 오경보 ≤ 3% 인 가장 낮은 t → 감은 눈 · 반쯤 취소 → 고르기 규칙 → **시드 더 돌릴 상위 2개** 와 판정 출력

### 12-5. 시드 재확인 — 상위 2개 + 기준을 시드 1로 (12-0 규칙 7, 12-4 `pick` 이 고른 2개)

In [ ]:
TOP2 = []            # ← 12-4 pick 이 출력한 'TOP2 = [...]' 를 그대로 (예: [(0.85, 1.0), (0.75, 1.0)])
SEEDS_MORE = [1]     # 규칙 7: 시드 1 로 한 번 더
assert TOP2, 'PC pick 결과를 TOP2 에 (후보가 하나도 없으면 12-5 는 안 돌림 — 서버 변경 없음)'
for cfg in TOP2 + [BASE12]:   # 기준도 같이 (공정한 비교)
    for sd in SEEDS_MORE:
        run12(cfg, sd)
print('끝 → eye_cnn12_*_s1.pt 3개도 PC 로 → eval --set trainval --models ... → pick 다시 (두 시드 모두 통과 · 반쯤 취소 평균으로 판정)')

### 12-6. 151명 최종 (PC, 한 번만) · 결과 (실행 뒤 채움)

```
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval --set val --models <고른 모델> eye_cnn12_k100_100_s0.pt
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py final --csv result_val_<고른 모델>.csv --t <pick 의 t>
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py final --csv result_val_eye_cnn12_k100_100_s0.csv --t <기준의 t>
```

- **규칙 11 안전 확인 (11/04 팀장, 고르기 X):** `aihub548.py safety --csvs team_eye_cnn_548.csv:0.4,team_<최종>.csv:<t>,team_eye_cnn12_k100_100_s0.csv:<기준 t>` — 팀 사진 · 조건 사진을 자기 t 로 (뜬 눈 오경보 · 감은 눈 취소 · 반쯤 취소 · 고개 돌림 · 숙임 · 마스크 · 선글라스) + 일반 뜬 눈 CNN 값 0.03 · 0.1 · 0.3 넘는 비율
- **최종 보고에 함께 (11/04 팀장):** ① t 가 범위 하한 근처, 일부 설정은 0.11 에 걸림 → 완전히 같은 조건 비교 아님 ② 취소선(t − 0.1)이 0.03 미만이 되는 영향 ③ 기준값은 사진 1장 평가로 정한 값 (서버는 3장 다수결 예정). `config` 는 팀장이 결정한 뒤 강사님 컨펌
- 결과를 보고 설정 · t 를 다시 고르지 않음. 팀 사진은 `team --model <고른 모델>` 로 시연 확인용으로 따로
**결과 (11/05 PC — 151명은 최종 · 기준 각 1회, 다시 고르지 않음)**

> 기준값은 사진 1장 평가로 정한 값 (서버는 3장 다수결 예정, 11/04 팀장 안내). 고른 t 가 범위 하한 근처이고 일부는 0.11 에 걸림 → 설정끼리 완전히 같은 조건 비교는 아님

① 고르기 — 548 학습용 val 115명 (`pick`, 시드 0 + 1)

| 설정 | 시드 0: t · 오경보 · 반쯤 취소 | 시드 1: t · 오경보 · 반쯤 취소 | 반쯤 평균 | 감은 눈 취소 |
|---|---|---|---|---|
| **k085_100 (0.85 ~ 1.0)** | 0.11 · 2.5% · 29.6% | 0.11 · 2.5% · 29.6% | **29.6%** | 0 / 0 |
| k080_080 (0.8 고정) | 0.13 · 2.5% · 27.7% | 0.11 · 2.2% · 32.8% | 30.2% | 0 / 0 |
| k100_100 (기준) | 0.12 · 2.8% · 31.2% | 0.11 · 2.0% · 34.2% | 32.7% | 0 / 0 |
| (시드 0 만) k075_100 · k075_090 | 0.11 · 2.8% · 30.1% / 0.11 · 2.7% · 33.4% | — | — | 0 |

- 판정: **k085_100** — 기준보다 반쯤 취소 3.1%p 낮음 (> 2%p) → vscale 보정 채택 → 서버 후보 `eye_cnn12_k085_100_s0.pt` + t 0.11 (`config` 는 팀장 결정 뒤 강사님 컨펌, 아직 안 바꿈)

② 151명 (548 Validation) — 각자 val 에서 고른 t 로 1회 (`final`)

| 모델 · t | 뜬 눈 오경보 (취소 못 함) | 감은 눈 취소 | 반쯤 감은 눈 취소 |
|---|---|---|---|
| **최종 k085_100 · 0.11** | 5.1% (377/7,380, 4.6 ~ 5.6%) | 0.0% (0/900, 0.0 ~ 0.4%) | 34.9% (283/810, 31.7 ~ 38.3%) |
| 기준 k100_100 · 0.12 | 4.5% (330/7,380, 4.0 ~ 5.0%) | 0.0% (0/900) | 33.6% (272/810, 30.4 ~ 36.9%) |
| (참고) 지금 서버 `eye_cnn_548` · 0.4 | 3.2% (236/7,380) | 0.0% (0/900) | 43.6% (353/810) |

- 판정 불가 (얼굴 못 찾음): 뜸 170/7,550 · 감음 56/956 · 반쯤 21/831 (CNN 값 없는 경우 0)
- **151명에서는 vscale 우위가 재현되지 않음** (최종 34.9% vs 기준 33.6%, 95% 범위 겹침) — 결과대로 보고, 다시 고르지 않음
- **오경보가 val 목표 3% 를 넘음** (5.1% · 4.5%) — t 0.11 · 0.12 면 취소선(t − 0.1)이 감음 확률 0.01 · 0.02 미만 → 뜬 눈 CNN 값이 조금만 올라도 취소 못 함
- 곡선 (보고용, 151명으로 고르지 않음): 같은 오경보에서 세 모델 거의 겹침 — 오경보 3% 지점 반쯤 취소 `eye_cnn_548` 50.2% · 기준 48.9% · 최종 49.5%, 오경보 ≈ 5% 지점 35.9% (t 0.13) · 33.6% · 36.9% → **반쯤 취소가 준 것은 대부분 t 를 낮춘 효과** (그림 `D:ihub548\curve_result_val_vs_..._k085_100_s0.png`)

③ 팀 사진 · 조건 사진 (규칙 11, 고르기 X — `safety`)

| 지표 | `eye_cnn_548` · 0.4 | 최종 · 0.11 | 기준 · 0.12 |
|---|---|---|---|
| 일반 뜬 눈 오경보 | 0.6% (1/160) | 5.6% (9/160) | 1.2% (2/160) |
| 감은 눈 취소 | 0/159 | 0/159 | 0/159 |
| 반쯤 취소 | 22.9% (25/109) | 6.4% (7/109) | 12.8% (14/109) |
| 고개 돌림 오경보 | 0/20 | 3/20 | 0/20 |
| 고개 숙임 오경보 | 10/10 | 10/10 | 10/10 |
| 마스크 오경보 · 감은 눈 취소 | 4/19 · 0 | 4/19 · 0 | 4/19 · 0 |
| 선글라스 오경보 · 감은 눈 취소 | 6/10 · 0 | 10/10 · 0 | 10/10 · 0 |
| 일반 뜬 눈 CNN > 0.01 · 0.03 · 0.1 · 0.3 | 11 · 4 · 2 · 1장 | 8 · 4 · 1 · 0장 | 3 · 1 · 0 · 0장 |

- 선글라스 뜬 눈은 폰이 의심 안 함 (0/10, A 확인) → 서버까지 안 옴. 고개 숙임은 모델 무관 (3장 다수결로도 안 풀림)

## 13. 아직 안 해 본 방법 — 반쯤 감은 눈 학습 · 손실 함수 · 해상도 (11/05)

- **왜:** 12번(vscale)은 115명에서 이겼지만 151명에서 재현 안 됨 — 같은 오경보에서 곡선이 겹침 (반쯤 취소가 준 것은 대부분 t 를 낮춘 효과). **반쯤 감은 눈을 모델이 직접 '감음 쪽' 으로 보게** 만드는 방법은 아직 안 함
- **기준 = `eye_cnn12_k100_100_s0.pt`** (12번 기준, MRL → 173 → 548). 아래 설정은 **기준의 173 단계 가중치(`v12_k100_100_s0_a173_best.pt`)에서 시작해 548 단계만 다르게** (`res96` 만 처음부터) → 다른 점이 하나씩만
- **설정 5개 (548 단계):**

| 이름 | 바꾸는 것 | 노리는 것 |
|---|---|---|
| `half1` | 548 학습용 train 의 **반쯤 감은 눈을 감음(1)** 으로 넣음 | 반쯤 취소 직접 줄이기 (대신 뜬 눈 오경보 늘 수 있음) |
| `half05` | 반쯤 감은 눈을 **0.5** 로 (부드러운 라벨) | 반쯤 = '확실히 뜸' 이 아니게 (가운데 값) |
| `focal` | 손실 = **focal loss** (γ = 2) — 쉬운 뜬 눈보다 헷갈리는 눈에 무게 | 경계 근처 눈 |
| `cw65` | 뽑기 비율 **감음 65 : 뜸 35** (지금 50 : 50) | 감음 쪽으로 기울이기 |
| `res96` | 입력 **96×96** (지금 64) — MRL → 173 → 548 처음부터, 548 눈은 원래 96 으로 저장돼 있음 | 가늘게 뜬 눈의 눈꺼풀 틈 |

- 반쯤 감은 눈은 **548 학습용 train 사람 것만** 학습에 씀 (val 115명 · Validation 151명 반쯤 눈은 안 씀 → 고르기 · 시험 공정). 12번까지는 반쯤을 학습에서 뺐음 (11/03 결정) — 이번에만 바꿔 보는 실험
- 조기 종료 val 은 12번과 같음 (548 val 115명의 뜸 · 감음 눈, 반쯤 없음) → 설정끼리 같은 잣대
- 개인 기준(운행 전 뜬 눈 5장)은 **학습 없이 PC 에서 115명 CSV 로 먼저 확인함 (11/05)** — 같은 오경보에서 반쯤 취소 차이 0.5 ~ 2%p (흔들림 안쪽) → 13번에 안 넣음 (`D:\scratch` 메모)

### 13-0. 확정 규칙 (결과 보기 전에 정함 — 12-0 과 같은 방식, **11/05 팀장 수정 반영**)

> **11/05 수정 (팀장):** 13번 학습은 끝났지만 **115명 고르기(`pick`) 결과는 보기 전에** 아래 1 · 2 · 5 를 고쳐 커밋함. 이유: 12번에서 "오경보 3% 이하인 가장 낮은 t" 규칙 때문에 t 가 범위 바닥 0.11 로 내려갔고 (취소선 t − 0.1 = 감음 확률 0.01), **151명에서 오경보 5.1% 로 재현되지 않음** · 팀 사진 오경보도 늘었음. 13번은 뜬 눈을 더 '감음 쪽' 으로 보게 하는 실험이라 그대로 두면 비교가 또 흐려짐

1. **후보** = 548 학습용 val 115명 (`eval --set trainval`) 에서 뜬 눈 오경보 ≤ 3% 인 t (**0.20 ~ 0.95**, 0.01 간격, 가장 낮은 t — 취소선이 감음 확률 0.1 이상) 가 있고 그 t 에서 감은 눈 취소 0건. 기준(`k100_100`)도 같은 범위로 다시 찾음 (`pick --exp 13` 이 0.20 부터)
2. 후보 중 반쯤 취소율 가장 낮은 설정, **기준(`k100_100`)보다 2%p 넘게 낮을 때만** 13번 설정 채택. **채택이 없으면 "후보 없음" → 서버는 `eye_cnn_548` + 0.4 그대로** (12번 기준 모델로 바꾸지 않음)
3. 시드 재확인: 상위 2개 + 기준을 시드 1 로 (기준 시드 1 = `eye_cnn12_k100_100_s1.pt` 이미 있음), 두 시드 모두 통과 · 반쯤 취소 평균
4. 동점이면 기준에 가까운 설정 (`focal` · `cw65` → `half05` → `half1` → `res96` 순)
5. **최종 시험은 새 데이터로 한 번, 채택 규칙은 `notebooks/new_test_rules.md` 5번** (지금 서버 `eye_cnn_548` + 0.4 vs 최종 후보 1개, 시험 전에 커밋). 조건을 못 넘으면 서버 그대로
   - 요약 (11/05 팀장 두 번째 답): **TS_G1_2 (G1 256명)** 반쯤 취소 2%p 넘게 낮음 + 같은 사람끼리 차이의 95% 범위가 0 을 넘지 않음 + 감은 눈 취소 안 늘어남 + 오경보 증가 1%p 이하 / **71421 Training** 감은 눈 취소 안 늘어남 + 오경보 증가 1%p 이하 / 173 은 참고. 하나라도 못 미치면 `eye_cnn_548` + 0.4 유지, 나빠도 다시 고르지 않고 그대로 보고 (`aihub548.py compare`)
6. 팀 사진은 고르는 데 안 씀 — 최종 시험 뒤 안전 확인 (규칙 11)

- **반쯤 감은 눈 문구 (11/05 팀장):** "반쯤 취소" 의 목표는 **"폰이 의심해 서버로 온 반쯤 감은 사진은 취소하지 않는 것"**. '경고해야 하는 눈' · '놓침' 으로 일반화하지 않음 (앱에서 반쯤은 1.5초 미세수면 대상이 아니고, 폰이 놓친 사진은 서버로 오지 않음). 감은 눈 취소와 합치지 않고 따로 보고

- **실행 순서 (Colab T4):** 1-1 → 1-2 → 1-3 → 2-1 → 2-1b → 2-2 → 2-3 → 2-4 → 3-1 → 3-2 → 7-1 → 12-1 → 12-2 → **13-1 → 13-2 → 13-3**
  - 시간: 548 단계만인 4개는 하나에 약 5 ~ 10분, `res96` 은 약 30 ~ 40분 → **약 1시간 ~ 1시간 반**. 끊기면 13-3 다시 (끝난 설정은 건너뜀)

### 13-1. 반쯤 감은 눈 (train 사람만) · 손실 · 데이터셋

In [ ]:
need = ('parts', 'ys', 'C548', 'Y548', 'I548', 'P548', 'm548', 'fit12', 'Arr12', 'Mrl12', 'pred12', 'score12', 'half_w', 'EyeCNN')
assert all(k in globals() for k in need), f"먼저 실행: {[k for k in need if k not in globals()]} (7-1 · 12-1 · 12-2)"
BASE_A173 = f'{TRAIN_DIR}/v12_k100_100_s0_a173_best.pt'   # 12번 기준의 173 단계 끝 가중치 (시드 0)
assert os.path.exists(BASE_A173), f'{BASE_A173} 없음 — 12-3 기준 (k100_100, 시드 0) 학습이 드라이브에 남아 있어야 함'

# 반쯤 감은 눈 — 548 학습용 train 사람 것만 (val 115명 · Validation 151명은 안 씀)
H548 = np.concatenate([z['X'][(y < 0) & (z['split'] == 'train')] for z, y in zip(parts, ys)])
h_driver = np.concatenate([z['driver'][(y < 0) & (z['split'] == 'train')] for z, y in zip(parts, ys)])
assert not set(h_driver) & set(m548.driver[I548['val']]), '반쯤 눈에 val 사람이 섞임'
print(f'반쯤 감은 눈 (train 사람 {len(set(h_driver))}명): {len(H548)}개 · 548 train 뜸 {(Y548[I548["train"]] == 0).sum()} · 감음 {(Y548[I548["train"]] == 1).sum()}')

VARIANTS13 = ['half1', 'half05', 'focal', 'cw65', 'res96']


class Mix13(torch.utils.data.Dataset):          # 548 train (+ 반쯤 감은 눈) — 큰 배열을 복사하지 않고 두 곳에서 읽음
    def __init__(self, half_y=None, size=None):
        self.n0, self.half_y, self.size = len(I548['train']), half_y, size
        self.y = np.concatenate([Y548[I548['train']], np.full(len(H548), half_y, np.float32)]) if half_y is not None else Y548[I548['train']]
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        img = C548[I548['train'][i]] if i < self.n0 else H548[i - self.n0]
        r = np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))
        return torch.from_numpy(prep_eye(augment(img, r), self.size or SIZE))[None], torch.tensor(self.y[i], dtype=torch.float32)


def w13(y, closed_share=0.5):                   # 뽑기 무게: '감음 쪽'(y ≥ 0.5) closed_share, 뜸 나머지
    c = y >= 0.5
    return np.where(c, closed_share / c.mean(), (1 - closed_share) / (~c).mean())


class Focal(torch.nn.Module):                   # focal loss (γ = 2) — logit 입력, BCE 와 같은 모양
    def __init__(self, gamma=2.0):
        super().__init__(); self.g = gamma
    def forward(self, logit, y):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logit, y, reduction='none')
        p = torch.sigmoid(logit); pt = p * y + (1 - p) * (1 - y)
        return ((1 - pt) ** self.g * bce).mean()


print('설정:', VARIANTS13, '/ 기준 = eye_cnn12_k100_100_s0.pt (이미 있음)')

### 13-2. 설정 하나 끝까지 (`run13`) — 548 단계만 다르게 (`res96` 은 처음부터)

In [ ]:
def run13(var, seed):
    global loss_fn, SIZE
    name = f'{var}_s{seed}'
    out = f'{TRAIN_DIR}/eye_cnn13_{name}.pt'
    if os.path.exists(out):
        print('끝난 설정 (건너뜀):', out); return
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    size = 96 if var == 'res96' else SIZE
    run = None
    if USE_WANDB:
        run = wandb.init(project='eye-cnn', name=f'b_v13_{name}', id=f'b_v13_{name}', group='exp13', resume='allow',
                         config=dict(model='resnet18', input=size, batch=BATCH, seed=seed, variant=var, lr=LR,
                                     start='12번 기준 173 단계 가중치' if var != 'res96' else '처음부터 (ImageNet)'),
                         settings=wandb.Settings(console='off'))
    old_loss, old_size = loss_fn, SIZE
    try:
        if var == 'res96':
            SIZE = 96   # Mrl12 · Arr12 · prep_eye 가 전역 SIZE 를 씀
            pv = lambda src, idx: np.stack([prep_eye(src[j], 96) for j in idx])
            v_mrl, v_173, v_548 = pv(IMGS, IDX['val']), pv(CROPS, AIDX['val']), pv(C548, I548['val'])
            s1, _ = fit12(f'v13_{name}', 'mrl', None, LR, EPOCHS, PATIENCE, Mrl12(IDX['train'], (1.0, 1.0)), None, v_mrl, Y[IDX['val']], run, seed)
            s2, _ = fit12(f'v13_{name}', 'a173', s1, LR / 10, 5, 2, Arr12(CROPS, AY, AIDX['train']), half_w(AY[AIDX['train']]),
                          v_173, AY[AIDX['val']], run, seed)
            ds = Mix13(size=96)
            s3, h3 = fit12(f'v13_{name}', 'a548', s2, LR / 10, 5, 2, ds, w13(ds.y), v_548, Y548[I548['val']], run, seed)
            del v_mrl, v_173
        else:
            init = torch.load(BASE_A173, map_location='cpu', weights_only=True)
            ds = Mix13(half_y={'half1': 1.0, 'half05': 0.5}.get(var))
            share = 0.65 if var == 'cw65' else 0.5
            if var == 'focal':
                loss_fn = Focal(2.0)
            v_548 = P548['val']
            s3, h3 = fit12(f'v13_{name}', 'a548', init, LR / 10, 5, 2, ds, w13(ds.y, share), v_548, Y548[I548['val']], run, seed)
        m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(s3)
        ref = {'AIHub548_val_eye': score12(pred12(m, v_548), Y548[I548['val']])}   # 참고 (고르기는 PC 서버 판정)
        if var != 'res96':
            ref['MRL_test'] = score12(pred12(m, PREP['test']), Y[IDX['test']])
            ref['AIHub173_test'] = score12(pred12(m, APREP['test']), AY[AIDX['test']])
        print(name, ref)
        torch.save(dict(state_dict=s3, arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=size, output='logit 1개, sigmoid = 감음 확률',
                        prep='eye_preprocess.prep_eye', variant=var, seed=seed,
                        data='MRL → AI-Hub 173 → AI-Hub 548 (763명, Validation 151명 뺌)' + (', 548 train 반쯤 감은 눈 포함' if var.startswith('half') else ''),
                        test=ref, hist={'a548': h3}, made=str(datetime.datetime.now())[:16], torch=str(torch.__version__)), out)
        torch.load(out, map_location='cpu', weights_only=True)
        if run:
            wandb.summary.update({f'{k}_{a}': float(v) for k, d in ref.items() for a, v in d.items()}); wandb.finish()
        print('저장:', out)
    finally:
        loss_fn, SIZE = old_loss, old_size

### 13-3. 실행 — 설정 5개 × 시드 0 (끊기면 이 셀 다시)

In [ ]:
for var in VARIANTS13:
    run13(var, SEED)
print('끝 → 드라이브 eye_cnn/eye_cnn13_*.pt 를 PC sleep_detector_api/ml_models/ 로 (GitHub X) → 13-4')

### 13-4. PC — 548 학습용 val 115명으로 고르기 (12번 기준과 같이)

```
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py eval --set trainval --models eye_cnn13_half1_s0.pt eye_cnn13_half05_s0.pt eye_cnn13_focal_s0.pt eye_cnn13_cw65_s0.pt eye_cnn13_res96_s0.pt
sleep_detector_api\venv\Scripts\python.exe notebooks\aihub548.py pick --exp 13
```

- `pick --exp 13` 은 t 를 **0.20 ~ 0.95** 에서 찾음 (13-0 규칙 1, 11/05 팀장 수정)
- 기준 `result_trainval_eye_cnn12_k100_100_s0.csv` 는 12번에서 이미 있음 → 다시 안 돌림. `pick --exp 13` 이 기준 + 13번 CSV 를 같이 읽음
- `res96` 은 입력 96 → `eval` 이 모델 파일의 `input_size` 를 읽어 맞춤 (서버에 넣으려면 `config.CNN_INPUT_SIZE = 96` 도 같이)

### 13-5. 시드 재확인 — 상위 2개 + 기준을 시드 1로 (13-0 규칙 3, 기준 시드 1 은 12번 것)

In [ ]:
TOP2_13 = []         # ← 13-4 pick 이 출력한 상위 2개 이름 (예: ['half05', 'focal'])
assert TOP2_13, 'PC pick 결과를 TOP2_13 에 (후보가 없으면 13-5 는 안 돌림)'
for var in TOP2_13:
    run13(var, 1)
print('끝 → eye_cnn13_*_s1.pt 도 PC 로 → eval --set trainval --models ... → pick --exp 13 다시 (기준 시드 1 = eye_cnn12_k100_100_s1)')

### 13-6. 최종 — 새 데이터로 한 번 (11/05 사용자 결정, 결과 보고 다시 고르지 않음)

- 151명은 12번에서 이미 씀 → **새로 받은 데이터** 로 고른 설정 · 기준 · 지금 서버(`eye_cnn_548` + 0.4) 를 각자 고른 t 로 한 번씩
- 보고 형식 고정: 전체 / 판정됨 / 판정 불가 / 취소 · 오경보 / 비율 / 95% 범위. **기준값은 사진 1장 평가로 정한 값 (서버는 3장 다수결)**